# 1182행 RG3가 실제 몇 Cycle인지, 불량 Cycle이 몇 개인지, RH/LH 구조가 어떻게 되어 있는지 확정

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD
# ============================================================

PATH = r"../../../data/raw\moldset_labeled_rg3_다시되돌림.csv"

df = pd.read_csv(PATH)

print("=" * 100)
print("RAW RG3")
print("=" * 100)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. TimeStamp
# ============================================================

df["TimeStamp"] = pd.to_datetime(df["TimeStamp"])

df = (
    df
    .sort_values(["TimeStamp", "original_index"])
    .reset_index(drop=True)
)


# ============================================================
# 3. TARGET
# ============================================================

print("\n" + "=" * 100)
print("TARGET")
print("=" * 100)

print(df["PassOrFail"].value_counts().sort_index())

print("\nDefect Rate:")
print(df["PassOrFail"].mean())


# ============================================================
# 4. SIDE 확인
# ============================================================

print("\n" + "=" * 100)
print("SIDE × TARGET")
print("=" * 100)

print(
    pd.crosstab(
        df["RH_LH"],
        df["PassOrFail"],
        margins=True
    )
)


# ============================================================
# 5. 센서 컬럼 정의
# ============================================================

META_COLS = {
    "PassOrFail",
    "TimeStamp",
    "RH_LH",
    "PART_NAME",
    "PART_FACT_SERIAL",
    "PART_FACT_PLAN_DATE",
    "Reason",
    "original_index",
    "Unnamed: 0",
    "_id",
    "EQUIP_CD",
    "EQUIP_NAME",
    "PART_NO"
}

sensor_cols = [
    c for c in df.select_dtypes(include=np.number).columns
    if c not in META_COLS
]

print("\n" + "=" * 100)
print("SENSOR COLUMNS")
print("=" * 100)

print("Sensor count:", len(sensor_cols))
print(sensor_cols)


# ============================================================
# 6. TimeStamp = Cycle 가정 확인
# ============================================================

cycle_size = (
    df
    .groupby("TimeStamp")
    .size()
)

print("\n" + "=" * 100)
print("CYCLE SIZE")
print("=" * 100)

print("Unique TimeStamp:", df["TimeStamp"].nunique())

print("\nRows per TimeStamp:")
print(cycle_size.value_counts().sort_index())


# ============================================================
# 7. 동일 TimeStamp 내 Side 구조
# ============================================================

side_per_cycle = (
    df
    .groupby("TimeStamp")["RH_LH"]
    .agg(
        lambda x: tuple(sorted(x.astype(str).unique()))
    )
)

print("\n" + "=" * 100)
print("SIDE COMBINATION PER CYCLE")
print("=" * 100)

print(side_per_cycle.value_counts())


# ============================================================
# 8. 동일 TimeStamp에서 센서가 정말 같은지 확인
# ============================================================

sensor_nunique = (
    df
    .groupby("TimeStamp")[sensor_cols]
    .nunique(dropna=False)
)

different_sensor_cycle = (
    sensor_nunique.max(axis=1) > 1
)

print("\n" + "=" * 100)
print("CYCLE SENSOR CONSISTENCY")
print("=" * 100)

print(
    "동일 TimeStamp에서 센서 조합이 2개 이상인 Cycle:",
    different_sensor_cycle.sum()
)


# ============================================================
# 9. 동일 Cycle 내 Target 조합
# ============================================================

target_combo = (
    df
    .groupby("TimeStamp")["PassOrFail"]
    .agg(
        lambda x: tuple(sorted(x.unique()))
    )
)

print("\n" + "=" * 100)
print("TARGET COMBINATION PER CYCLE")
print("=" * 100)

print(target_combo.value_counts())


# 정상 + 불량이 동시에 존재하는 Cycle
conflict_cycles = target_combo[
    target_combo.apply(
        lambda x: (0 in x) and (1 in x)
    )
]

print(
    "\n정상/불량이 동시에 존재하는 Cycle:",
    len(conflict_cycles)
)


# ============================================================
# 10. Cycle-level Dataset 생성
# ============================================================

cycle_sensor = (
    df
    .groupby("TimeStamp")[sensor_cols]
    .first()
)

cycle_target = (
    df
    .groupby("TimeStamp")["PassOrFail"]
    .max()
    .rename("CycleDefect")
)

cycle_rows = (
    df
    .groupby("TimeStamp")
    .size()
    .rename("Rows_In_Cycle")
)

cycle_defect_side = (
    df[df["PassOrFail"] == 1]
    .groupby("TimeStamp")["RH_LH"]
    .agg(
        lambda x: ",".join(
            sorted(x.astype(str).unique())
        )
    )
    .rename("Defect_Side")
)

cycle_df = pd.concat(
    [
        cycle_sensor,
        cycle_target,
        cycle_rows,
        cycle_defect_side
    ],
    axis=1
).reset_index()

cycle_df["Defect_Side"] = (
    cycle_df["Defect_Side"]
    .fillna("None")
)


print("\n" + "=" * 100)
print("CYCLE LEVEL DATA")
print("=" * 100)

print("Shape:", cycle_df.shape)

print("\nTarget:")
print(
    cycle_df["CycleDefect"]
    .value_counts()
    .sort_index()
)

print("\nDefect rate:")
print(
    cycle_df["CycleDefect"].mean()
)

print("\nDefect Side:")
print(
    cycle_df["Defect_Side"]
    .value_counts()
)


# ============================================================
# 11. 시간 간격 / Session
# ============================================================

cycle_df = (
    cycle_df
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)

cycle_df["gap_min"] = (
    cycle_df["TimeStamp"]
    .diff()
    .dt.total_seconds()
    / 60
)

# 5분 초과 공백이면 새 생산 Session
cycle_df["session_id"] = (
    (cycle_df["gap_min"] > 5)
    .cumsum()
)

session_summary = (
    cycle_df
    .groupby("session_id")
    .agg(
        Start=("TimeStamp", "min"),
        End=("TimeStamp", "max"),
        Cycles=("CycleDefect", "size"),
        Defect_Cycles=("CycleDefect", "sum"),
        Defect_Rate=("CycleDefect", "mean")
    )
)

print("\n" + "=" * 100)
print("SESSION SUMMARY")
print("=" * 100)

print(session_summary.to_string())


# ============================================================
# 12. 불량 Cycle 시간
# ============================================================

print("\n" + "=" * 100)
print("DEFECT CYCLES")
print("=" * 100)

print(
    cycle_df.loc[
        cycle_df["CycleDefect"] == 1,
        [
            "TimeStamp",
            "session_id",
            "Defect_Side"
        ]
    ].to_string(index=False)
)

In [ ]:
# ============================================================
# RG3 Cycle-level Temporal Modeling
#
# 목표
# 1) 현재 Cycle만
# 2) 현재 + 직전 Cycle 변화량
# 3) 현재 + 변화량 + 최근 3 Cycle 정상상태 대비 편차
#
# 검증
# 과거 Session -> 미래 Session
# ============================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)


# ============================================================
# 1. 데이터 복사
# ============================================================

data = (
    cycle_df
    .sort_values("TimeStamp")
    .reset_index(drop=True)
    .copy()
)


# ============================================================
# 2. 원본 센서 컬럼
# ============================================================

exclude_cols = {
    "TimeStamp",
    "CycleDefect",
    "Rows_In_Cycle",
    "Defect_Side",
    "gap_min",
    "session_id"
}

raw_features = [
    c for c in data.select_dtypes(include=np.number).columns
    if c not in exclude_cols
]

print("=" * 100)
print("RAW FEATURES")
print("=" * 100)

print("Before constant removal:", len(raw_features))


# ============================================================
# 3. 상수 컬럼 제거
# ============================================================

constant_cols = [
    c for c in raw_features
    if data[c].nunique(dropna=False) <= 1
]

print("\nConstant columns:")
print(constant_cols)

raw_features = [
    c for c in raw_features
    if c not in constant_cols
]

print("\nAfter constant removal:", len(raw_features))


# ============================================================
# 4. Temporal Feature 생성
#
# 중요:
# 모든 rolling 계산은 session 안에서만 수행
# session을 넘어 과거값을 섞지 않음
# ============================================================

for col in raw_features:

    # --------------------------------------------------------
    # 직전 Cycle
    # --------------------------------------------------------

    lag1 = (
        data
        .groupby("session_id")[col]
        .shift(1)
    )

    data[f"{col}_lag1"] = lag1


    # --------------------------------------------------------
    # 현재 - 직전 Cycle
    # --------------------------------------------------------

    data[f"{col}_diff1"] = (
        data[col] - lag1
    )


    # --------------------------------------------------------
    # 현재 Cycle 이전의 최근 3 Cycle 평균
    #
    # 현재값 포함 X
    # 반드시 shift(1) 후 rolling
    # --------------------------------------------------------

    prev_mean3 = (
        data
        .groupby("session_id")[col]
        .transform(
            lambda s:
            s.shift(1)
            .rolling(
                window=3,
                min_periods=1
            )
            .mean()
        )
    )

    data[f"{col}_prev_mean3"] = prev_mean3


    # --------------------------------------------------------
    # 최근 3 Cycle 표준편차
    # --------------------------------------------------------

    prev_std3 = (
        data
        .groupby("session_id")[col]
        .transform(
            lambda s:
            s.shift(1)
            .rolling(
                window=3,
                min_periods=2
            )
            .std(ddof=0)
        )
    )

    data[f"{col}_prev_std3"] = prev_std3


    # --------------------------------------------------------
    # 현재값이 최근 정상수준에서 얼마나 벗어났는지
    # --------------------------------------------------------

    data[f"{col}_dev3"] = (
        data[col] - prev_mean3
    )


# ============================================================
# 5. Feature Set 정의
# ============================================================

diff_features = [
    f"{c}_diff1"
    for c in raw_features
]

dev_features = [
    f"{c}_dev3"
    for c in raw_features
]

std_features = [
    f"{c}_prev_std3"
    for c in raw_features
]


feature_sets = {

    # 현재 Cycle 센서만
    "V0_current":
        raw_features,

    # 현재값 + 직전 Cycle 변화량
    "V1_diff":
        (
            raw_features
            + diff_features
        ),

    # 현재값
    # + 직전 변화량
    # + 최근 3 Cycle 평균 대비 편차
    # + 최근 3 Cycle 변동성
    "V2_temporal":
        (
            raw_features
            + diff_features
            + dev_features
            + std_features
        )
}


print("\n" + "=" * 100)
print("FEATURE SETS")
print("=" * 100)

for name, cols in feature_sets.items():
    print(
        f"{name:15s}: {len(cols)} features"
    )


# ============================================================
# 6. Walk-forward Session Split
# ============================================================

splits = []

for valid_session in range(2, 7):

    train_sessions = list(
        range(valid_session)
    )

    splits.append(
        (
            train_sessions,
            [valid_session]
        )
    )


print("\n" + "=" * 100)
print("TIME SPLITS")
print("=" * 100)

for i, (tr, va) in enumerate(splits, 1):
    print(
        f"Fold {i}: "
        f"Train {tr} -> Valid {va}"
    )


# ============================================================
# 7. Model 정의 함수
#
# 매 Fold마다 새 모델 생성
# ============================================================

def get_models():

    return {

        "Logistic": Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    C=0.5,
                    max_iter=3000,
                    random_state=42
                )
            )
        ]),

        "RandomForest": Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=6,
                    min_samples_leaf=2,
                    max_features="sqrt",
                    class_weight="balanced_subsample",
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]),

        "ExtraTrees": Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "model",
                ExtraTreesClassifier(
                    n_estimators=500,
                    max_depth=6,
                    min_samples_leaf=2,
                    max_features="sqrt",
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1
                )
            )
        ])
    }


# ============================================================
# 8. Walk-forward 평가
# ============================================================

results = []

oof_predictions = []


for feature_name, features in feature_sets.items():

    print(
        "\n\n",
        "=" * 100
    )

    print(
        "FEATURE SET:",
        feature_name
    )

    print(
        "=" * 100
    )


    for fold, (
        train_sessions,
        valid_sessions
    ) in enumerate(
        splits,
        start=1
    ):

        train_mask = (
            data["session_id"]
            .isin(train_sessions)
        )

        valid_mask = (
            data["session_id"]
            .isin(valid_sessions)
        )


        X_train = (
            data.loc[
                train_mask,
                features
            ]
        )

        y_train = (
            data.loc[
                train_mask,
                "CycleDefect"
            ]
            .astype(int)
        )


        X_valid = (
            data.loc[
                valid_mask,
                features
            ]
        )

        y_valid = (
            data.loc[
                valid_mask,
                "CycleDefect"
            ]
            .astype(int)
        )


        print(
            f"\nFold {fold}"
            f" | Train Session {train_sessions}"
            f" -> Valid {valid_sessions}"
        )

        print(
            "Train:",
            len(y_train),
            "| Defects:",
            int(y_train.sum())
        )

        print(
            "Valid:",
            len(y_valid),
            "| Defects:",
            int(y_valid.sum())
        )


        baseline_pr = (
            y_valid.mean()
        )


        models = get_models()


        for model_name, model in models.items():

            model.fit(
                X_train,
                y_train
            )

            pred = (
                model.predict_proba(
                    X_valid
                )[:, 1]
            )


            pr_auc = (
                average_precision_score(
                    y_valid,
                    pred
                )
            )


            roc_auc = (
                roc_auc_score(
                    y_valid,
                    pred
                )
            )


            results.append({

                "Feature_Set":
                    feature_name,

                "Model":
                    model_name,

                "Fold":
                    fold,

                "Train_Sessions":
                    str(train_sessions),

                "Valid_Session":
                    valid_sessions[0],

                "Train_N":
                    len(y_train),

                "Train_Defects":
                    int(y_train.sum()),

                "Valid_N":
                    len(y_valid),

                "Valid_Defects":
                    int(y_valid.sum()),

                "Baseline_PR":
                    baseline_pr,

                "PR_AUC":
                    pr_auc,

                "PR_Uplift":
                    (
                        pr_auc
                        - baseline_pr
                    ),

                "PR_Ratio":
                    (
                        pr_auc
                        / baseline_pr
                        if baseline_pr > 0
                        else np.nan
                    ),

                "ROC_AUC":
                    roc_auc
            })


            # OOF 저장
            temp_pred = pd.DataFrame({

                "TimeStamp":
                    data.loc[
                        valid_mask,
                        "TimeStamp"
                    ].values,

                "session_id":
                    data.loc[
                        valid_mask,
                        "session_id"
                    ].values,

                "y_true":
                    y_valid.values,

                "y_prob":
                    pred,

                "Feature_Set":
                    feature_name,

                "Model":
                    model_name,

                "Fold":
                    fold
            })

            oof_predictions.append(
                temp_pred
            )


# ============================================================
# 9. 결과 DataFrame
# ============================================================

result_df = pd.DataFrame(
    results
)

oof_df = pd.concat(
    oof_predictions,
    ignore_index=True
)


# ============================================================
# 10. Fold별 결과
# ============================================================

print(
    "\n\n",
    "=" * 100
)

print(
    "FOLD RESULTS"
)

print(
    "=" * 100
)

print(
    result_df[
        [
            "Feature_Set",
            "Model",
            "Fold",
            "Valid_Session",
            "Valid_Defects",
            "Baseline_PR",
            "PR_AUC",
            "PR_Ratio",
            "ROC_AUC"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 11. 모델 요약
# ============================================================

summary = (
    result_df
    .groupby(
        [
            "Feature_Set",
            "Model"
        ]
    )
    .agg(

        PR_Mean=(
            "PR_AUC",
            "mean"
        ),

        PR_Median=(
            "PR_AUC",
            "median"
        ),

        PR_Min=(
            "PR_AUC",
            "min"
        ),

        PR_Max=(
            "PR_AUC",
            "max"
        ),

        PR_Std=(
            "PR_AUC",
            "std"
        ),

        PR_Ratio_Mean=(
            "PR_Ratio",
            "mean"
        ),

        ROC_Mean=(
            "ROC_AUC",
            "mean"
        ),

        ROC_Min=(
            "ROC_AUC",
            "min"
        )
    )
    .reset_index()
    .sort_values(
        [
            "PR_Mean",
            "PR_Min"
        ],
        ascending=False
    )
)


print(
    "\n\n",
    "=" * 100
)

print(
    "MODEL SUMMARY"
)

print(
    "=" * 100
)

print(
    summary.to_string(
        index=False
    )
)


# ============================================================
# 12. 전체 OOF PR-AUC
#
# Fold별 평균과 별개로
# 시간순 OOF 예측 전체 성능도 확인
# ============================================================

oof_summary = []

for (
    feature_name,
    model_name
), group in oof_df.groupby(
    [
        "Feature_Set",
        "Model"
    ]
):

    pr_auc = (
        average_precision_score(
            group["y_true"],
            group["y_prob"]
        )
    )

    baseline = (
        group["y_true"]
        .mean()
    )

    oof_summary.append({

        "Feature_Set":
            feature_name,

        "Model":
            model_name,

        "OOF_N":
            len(group),

        "OOF_Defects":
            int(
                group["y_true"]
                .sum()
            ),

        "Baseline_PR":
            baseline,

        "OOF_PR_AUC":
            pr_auc,

        "OOF_PR_Ratio":
            pr_auc / baseline
    })


oof_summary = (
    pd.DataFrame(
        oof_summary
    )
    .sort_values(
        "OOF_PR_AUC",
        ascending=False
    )
)


print(
    "\n\n",
    "=" * 100
)

print(
    "OOF SUMMARY"
)

print(
    "=" * 100
)

print(
    oof_summary.to_string(
        index=False
    )
)

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# PATH
# ============================================================

BASE_DIR = (
    r"../../.."
    r"\1. 사출성형기 AI 데이터셋"
    r"\1. 사출성형기 AI 데이터셋"
)

RG3_PATH = BASE_DIR + r"\moldset_labeled_rg3_다시되돌림.csv"
ORIGINAL_PATH = BASE_DIR + r"\moldset_labeled.csv"


# ============================================================
# 1. LOAD
# ============================================================

rg3 = pd.read_csv(RG3_PATH)
original = pd.read_csv(ORIGINAL_PATH)


print("=" * 100)
print("SHAPE")
print("=" * 100)

print("RG3     :", rg3.shape)
print("Original:", original.shape)


# ============================================================
# 2. RG3 원본 행 추출
# ============================================================

print("\n" + "=" * 100)
print("PART_NAME")
print("=" * 100)

print(
    original["PART_NAME"]
    .value_counts()
    .to_string()
)


rg3_original = (
    original[
        original["PART_NAME"]
        .astype(str)
        .str.contains(
            "RG3",
            case=False,
            na=False
        )
    ]
    .copy()
)


print("\nRG3 Original Shape:", rg3_original.shape)

print("\nRG3 PART_NAME:")
print(
    rg3_original["PART_NAME"]
    .value_counts()
)


# ============================================================
# 3. 현재 RG3와 원본 컬럼 비교
# ============================================================

current_cols = set(rg3.columns)
original_cols = set(rg3_original.columns)

common_cols = sorted(
    current_cols & original_cols
)

extra_cols = sorted(
    original_cols - current_cols
)


print("\n" + "=" * 100)
print("COLUMN COMPARISON")
print("=" * 100)

print("\n현재 RG3 컬럼:", len(current_cols))
print("원본 RG3 컬럼:", len(original_cols))

print("\n공통 컬럼:", len(common_cols))
print(common_cols)

print("\n원본에만 존재하는 컬럼:")
print(extra_cols)


# ============================================================
# 4. 추가 컬럼 상세 진단
# ============================================================

rows = []

for col in extra_cols:

    s = rg3_original[col]

    numeric = pd.to_numeric(
        s,
        errors="coerce"
    )

    numeric_rate = numeric.notna().mean()

    if numeric_rate > 0.95:

        dtype_type = "numeric"

        n_unique = numeric.nunique(
            dropna=True
        )

        missing = numeric.isna().sum()

        mean = numeric.mean()
        std = numeric.std()
        min_val = numeric.min()
        max_val = numeric.max()

    else:

        dtype_type = "categorical"

        n_unique = s.nunique(
            dropna=True
        )

        missing = s.isna().sum()

        mean = np.nan
        std = np.nan
        min_val = np.nan
        max_val = np.nan


    rows.append({

        "Feature": col,

        "Type": dtype_type,

        "N_Unique": n_unique,

        "Missing": missing,

        "Missing_Rate":
            missing / len(s),

        "Mean": mean,

        "Std": std,

        "Min": min_val,

        "Max": max_val
    })


extra_summary = pd.DataFrame(rows)


print("\n" + "=" * 100)
print("EXTRA COLUMN SUMMARY")
print("=" * 100)

print(
    extra_summary
    .sort_values(
        [
            "Type",
            "N_Unique"
        ],
        ascending=[
            True,
            False
        ]
    )
    .to_string(index=False)
)


# ============================================================
# 5. 숫자형 추가 공정변수 후보
# ============================================================

candidate_extra = (
    extra_summary[
        (extra_summary["Type"] == "numeric")
        &
        (extra_summary["N_Unique"] > 1)
        &
        (extra_summary["Missing_Rate"] < 0.5)
    ]
    .copy()
)


print("\n" + "=" * 100)
print("NUMERIC EXTRA CANDIDATES")
print("=" * 100)

print(
    candidate_extra
    .sort_values(
        "N_Unique",
        ascending=False
    )
    .to_string(index=False)
)


# ============================================================
# 6. 행 수 / 시간 범위 확인
# ============================================================

if "TimeStamp" in rg3_original.columns:

    rg3_original["TimeStamp"] = pd.to_datetime(
        rg3_original["TimeStamp"]
    )

    print("\n" + "=" * 100)
    print("TIME RANGE")
    print("=" * 100)

    print(
        rg3_original["TimeStamp"].min(),
        "~",
        rg3_original["TimeStamp"].max()
    )

    print(
        "Unique TimeStamp:",
        rg3_original["TimeStamp"].nunique()
    )


# ============================================================
# 7. Target
# ============================================================

print("\n" + "=" * 100)
print("ORIGINAL TARGET")
print("=" * 100)

print(
    rg3_original["PassOrFail"]
    .value_counts(
        dropna=False
    )
)

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# RG3 원본
# ============================================================

rg3_o = rg3_original.copy()

rg3_o["TimeStamp"] = pd.to_datetime(
    rg3_o["TimeStamp"]
)

rg3_o = (
    rg3_o
    .sort_values(
        ["TimeStamp", "PART_NAME"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 1. Switch Over Position 분포
# ============================================================

print("=" * 100)
print("SWITCH OVER POSITION")
print("=" * 100)

print(
    rg3_o["Switch_Over_Position"]
    .value_counts(dropna=False)
    .sort_index()
)


print("\nNon-zero rows:")

print(
    rg3_o.loc[
        rg3_o["Switch_Over_Position"] != 0,
        [
            "TimeStamp",
            "PART_NAME",
            "PassOrFail",
            "PART_FACT_SERIAL",
            "PART_FACT_PLAN_DATE",
            "Switch_Over_Position"
        ]
    ].to_string(index=False)
)


# ============================================================
# 2. Cycle 단위 Switch Over
# ============================================================

switch_cycle = (
    rg3_o
    .groupby("TimeStamp")
    .agg(
        Switch_Over_Position=(
            "Switch_Over_Position",
            "first"
        ),
        CycleDefect=(
            "PassOrFail",
            "max"
        ),
        PART_FACT_SERIAL=(
            "PART_FACT_SERIAL",
            "first"
        ),
        PART_FACT_PLAN_DATE=(
            "PART_FACT_PLAN_DATE",
            "first"
        )
    )
    .reset_index()
)


print("\n" + "=" * 100)
print("SWITCH × TARGET")
print("=" * 100)

print(
    pd.crosstab(
        switch_cycle["Switch_Over_Position"],
        switch_cycle["CycleDefect"],
        margins=True
    )
)

In [ ]:
# ============================================================
# Cycle-level metadata
# ============================================================

meta_cycle = (
    rg3_o
    .groupby("TimeStamp")
    .agg(
        CycleDefect=(
            "PassOrFail",
            "max"
        ),

        PART_FACT_SERIAL=(
            "PART_FACT_SERIAL",
            "first"
        ),

        PART_FACT_PLAN_DATE=(
            "PART_FACT_PLAN_DATE",
            "first"
        ),

        Switch_Over_Position=(
            "Switch_Over_Position",
            "first"
        )
    )
    .reset_index()
    .sort_values("TimeStamp")
)


# 기존 cycle_df에서 session 가져오기
meta_cycle = meta_cycle.merge(
    cycle_df[
        [
            "TimeStamp",
            "session_id"
        ]
    ],
    on="TimeStamp",
    how="left"
)


# ============================================================
# SERIAL × SESSION
# ============================================================

print("\n" + "=" * 100)
print("SERIAL × SESSION")
print("=" * 100)

print(
    pd.crosstab(
        meta_cycle["session_id"],
        meta_cycle["PART_FACT_SERIAL"],
        margins=True
    )
)


# ============================================================
# SERIAL × TARGET
# ============================================================

print("\n" + "=" * 100)
print("SERIAL × TARGET")
print("=" * 100)

serial_target = (
    meta_cycle
    .groupby("PART_FACT_SERIAL")
    .agg(
        Cycles=("CycleDefect", "size"),
        Defects=("CycleDefect", "sum"),
        Defect_Rate=("CycleDefect", "mean"),
        First_Time=("TimeStamp", "min"),
        Last_Time=("TimeStamp", "max")
    )
)

print(
    serial_target.to_string()
)


# ============================================================
# PLAN DATE × TARGET
# ============================================================

print("\n" + "=" * 100)
print("PLAN DATE × TARGET")
print("=" * 100)

date_target = (
    meta_cycle
    .groupby("PART_FACT_PLAN_DATE")
    .agg(
        Cycles=("CycleDefect", "size"),
        Defects=("CycleDefect", "sum"),
        Defect_Rate=("CycleDefect", "mean"),
        First_Time=("TimeStamp", "min"),
        Last_Time=("TimeStamp", "max")
    )
)

print(
    date_target.to_string()
)


# ============================================================
# SERIAL 변화 지점
# ============================================================

meta_cycle["serial_change"] = (
    meta_cycle["PART_FACT_SERIAL"]
    !=
    meta_cycle["PART_FACT_SERIAL"].shift(1)
)

print("\n" + "=" * 100)
print("SERIAL CHANGE POINTS")
print("=" * 100)

print(
    meta_cycle.loc[
        meta_cycle["serial_change"],
        [
            "TimeStamp",
            "session_id",
            "PART_FACT_SERIAL",
            "PART_FACT_PLAN_DATE"
        ]
    ].to_string(index=False)
)

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD
# ============================================================

BASE_DIR = (
    r"../../.."
    r"\1. 사출성형기 AI 데이터셋"
    r"\1. 사출성형기 AI 데이터셋"
)

UNLABELED_PATH = BASE_DIR + r"\unlabeled_data.csv"

unlabeled = pd.read_csv(UNLABELED_PATH)

print("=" * 100)
print("UNLABELED RAW")
print("=" * 100)

print("Shape:", unlabeled.shape)


# ============================================================
# 2. RG3 추출
# ============================================================

rg3_u = (
    unlabeled[
        unlabeled["PART_NAME"]
        .astype(str)
        .str.contains(
            "RG3",
            case=False,
            na=False
        )
    ]
    .copy()
)


print("\nRG3 unlabeled rows:", len(rg3_u))

print("\nPART_NAME:")
print(
    rg3_u["PART_NAME"]
    .value_counts()
)


# ============================================================
# 3. TimeStamp
# ============================================================

rg3_u["TimeStamp"] = pd.to_datetime(
    rg3_u["TimeStamp"]
)

rg3_u = (
    rg3_u
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


print("\nTime Range:")
print(
    rg3_u["TimeStamp"].min(),
    "~",
    rg3_u["TimeStamp"].max()
)

print(
    "Unique TimeStamp:",
    rg3_u["TimeStamp"].nunique()
)


# ============================================================
# 4. Side 추출
# ============================================================

rg3_u["RH_LH"] = np.where(
    rg3_u["PART_NAME"]
    .astype(str)
    .str.contains(
        "RH",
        case=False,
        na=False
    ),
    "RH",
    "LH"
)


print("\n" + "=" * 100)
print("SIDE")
print("=" * 100)

print(
    rg3_u["RH_LH"]
    .value_counts()
)


# ============================================================
# 5. Cycle Size
# ============================================================

cycle_size = (
    rg3_u
    .groupby("TimeStamp")
    .size()
)


print("\n" + "=" * 100)
print("ROWS PER TIMESTAMP")
print("=" * 100)

print(
    cycle_size
    .value_counts()
    .sort_index()
)


# ============================================================
# 6. Side combination
# ============================================================

side_combo = (
    rg3_u
    .groupby("TimeStamp")["RH_LH"]
    .agg(
        lambda x:
        tuple(
            sorted(
                x.unique()
            )
        )
    )
)


print("\n" + "=" * 100)
print("SIDE COMBINATION")
print("=" * 100)

print(
    side_combo
    .value_counts()
)


# ============================================================
# 7. 사용할 24개 Sensor
# labeled와 정확히 동일
# ============================================================

sensor_cols = [
    'Injection_Time',
    'Filling_Time',
    'Plasticizing_Time',
    'Cycle_Time',
    'Clamp_Close_Time',
    'Cushion_Position',
    'Plasticizing_Position',
    'Clamp_Open_Position',
    'Max_Injection_Speed',
    'Max_Screw_RPM',
    'Average_Screw_RPM',
    'Max_Injection_Pressure',
    'Max_Switch_Over_Pressure',
    'Max_Back_Pressure',
    'Average_Back_Pressure',
    'Barrel_Temperature_1',
    'Barrel_Temperature_2',
    'Barrel_Temperature_3',
    'Barrel_Temperature_4',
    'Barrel_Temperature_5',
    'Barrel_Temperature_6',
    'Hopper_Temperature',
    'Mold_Temperature_3',
    'Mold_Temperature_4'
]


# ============================================================
# 8. 같은 Cycle의 LH/RH 센서 일치 여부
# ============================================================

sensor_nunique = (
    rg3_u
    .groupby("TimeStamp")[sensor_cols]
    .nunique(dropna=False)
)

different_cycle = (
    sensor_nunique.max(axis=1) > 1
)


print("\n" + "=" * 100)
print("SENSOR CONSISTENCY")
print("=" * 100)

print(
    "동일 TimeStamp에서 "
    "LH/RH 센서값이 다른 Cycle:",
    different_cycle.sum()
)


# ============================================================
# 9. Cycle-level 변환
# ============================================================

rg3_u_cycle = (
    rg3_u
    .groupby("TimeStamp")[sensor_cols]
    .first()
    .reset_index()
)


print("\n" + "=" * 100)
print("UNLABELED CYCLE DATA")
print("=" * 100)

print(
    "Cycle shape:",
    rg3_u_cycle.shape
)


# ============================================================
# 10. labeled Cycle과 분포 비교
# ============================================================

labeled_compare = (
    cycle_df[
        sensor_cols
    ]
    .agg(
        ["mean", "std", "min", "max"]
    )
    .T
)

labeled_compare.columns = [
    "Labeled_Mean",
    "Labeled_Std",
    "Labeled_Min",
    "Labeled_Max"
]


unlabeled_compare = (
    rg3_u_cycle[
        sensor_cols
    ]
    .agg(
        ["mean", "std", "min", "max"]
    )
    .T
)

unlabeled_compare.columns = [
    "Unlabeled_Mean",
    "Unlabeled_Std",
    "Unlabeled_Min",
    "Unlabeled_Max"
]


compare = (
    labeled_compare
    .join(
        unlabeled_compare
    )
)


# 평균 차이를 labeled std 기준으로 계산
compare["Mean_Diff_Z"] = (
    (
        compare["Unlabeled_Mean"]
        -
        compare["Labeled_Mean"]
    )
    /
    compare["Labeled_Std"]
    .replace(0, np.nan)
)


# std 비율
compare["Std_Ratio"] = (
    compare["Unlabeled_Std"]
    /
    compare["Labeled_Std"]
    .replace(0, np.nan)
)


print("\n" + "=" * 100)
print("LABELED vs UNLABELED DISTRIBUTION")
print("=" * 100)

print(
    compare[
        [
            "Labeled_Mean",
            "Unlabeled_Mean",
            "Mean_Diff_Z",
            "Labeled_Std",
            "Unlabeled_Std",
            "Std_Ratio"
        ]
    ]
    .sort_values(
        "Mean_Diff_Z",
        key=lambda s: s.abs(),
        ascending=False
    )
    .to_string()
)


# ============================================================
# 11. 결측
# ============================================================

print("\n" + "=" * 100)
print("MISSING")
print("=" * 100)

missing = (
    rg3_u_cycle[sensor_cols]
    .isna()
    .sum()
)

print(
    missing[
        missing > 0
    ]
)

print(
    "\nTotal missing:",
    int(missing.sum())
)

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# PATH
# ============================================================

BASE_DIR = (
    r"../../.."
    r"\1. 사출성형기 AI 데이터셋"
    r"\1. 사출성형기 AI 데이터셋"
)

LABELED_PATH = BASE_DIR + r"\moldset_labeled.csv"
UNLABELED_PATH = BASE_DIR + r"\unlabeled_data.csv"


# ============================================================
# 1. LOAD
# ============================================================

labeled_raw = pd.read_csv(LABELED_PATH)
unlabeled_raw = pd.read_csv(UNLABELED_PATH)


# ============================================================
# 2. 정확한 RG3 RH 제품만 선택
# ============================================================

TARGET_PRODUCT = "RG3 MOLD'G W/SHLD, RH"

lab = (
    labeled_raw[
        labeled_raw["PART_NAME"].eq(TARGET_PRODUCT)
    ]
    .copy()
)

unlab = (
    unlabeled_raw[
        unlabeled_raw["PART_NAME"].eq(TARGET_PRODUCT)
    ]
    .copy()
)


lab["TimeStamp"] = pd.to_datetime(
    lab["TimeStamp"]
)

unlab["TimeStamp"] = pd.to_datetime(
    unlab["TimeStamp"]
)


lab = (
    lab
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)

unlab = (
    unlab
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


print("=" * 100)
print("EXACT RG3 RH")
print("=" * 100)

print(
    "Labeled  :",
    lab.shape
)

print(
    "Unlabeled:",
    unlab.shape
)


print("\nLabeled Target:")
print(
    lab["PassOrFail"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 3. 사용할 24개 센서
# ============================================================

sensor_cols = [
    'Injection_Time',
    'Filling_Time',
    'Plasticizing_Time',
    'Cycle_Time',
    'Clamp_Close_Time',
    'Cushion_Position',
    'Plasticizing_Position',
    'Clamp_Open_Position',
    'Max_Injection_Speed',
    'Max_Screw_RPM',
    'Average_Screw_RPM',
    'Max_Injection_Pressure',
    'Max_Switch_Over_Pressure',
    'Max_Back_Pressure',
    'Average_Back_Pressure',
    'Barrel_Temperature_1',
    'Barrel_Temperature_2',
    'Barrel_Temperature_3',
    'Barrel_Temperature_4',
    'Barrel_Temperature_5',
    'Barrel_Temperature_6',
    'Hopper_Temperature',
    'Mold_Temperature_3',
    'Mold_Temperature_4'
]


# ============================================================
# 4. 시간 범위
# ============================================================

print("\n" + "=" * 100)
print("TIME RANGE")
print("=" * 100)

print(
    "Labeled:",
    lab["TimeStamp"].min(),
    "~",
    lab["TimeStamp"].max()
)

print(
    "Unlabeled:",
    unlab["TimeStamp"].min(),
    "~",
    unlab["TimeStamp"].max()
)


# ============================================================
# 5. Timestamp 중복 여부
# ============================================================

lab_times = set(
    lab["TimeStamp"]
)

unlab_times = set(
    unlab["TimeStamp"]
)

time_overlap = (
    lab_times
    &
    unlab_times
)


print("\n" + "=" * 100)
print("TIMESTAMP OVERLAP")
print("=" * 100)

print(
    "Labeled unique timestamps:",
    lab["TimeStamp"].nunique()
)

print(
    "Unlabeled unique timestamps:",
    unlab["TimeStamp"].nunique()
)

print(
    "Overlap timestamps:",
    len(time_overlap)
)


if len(time_overlap) > 0:

    overlap_sample = sorted(
        time_overlap
    )[:20]

    print("\nOverlap sample:")

    for x in overlap_sample:
        print(x)


# ============================================================
# 6. 센서 완전중복 검사
#
# Timestamp가 달라도 24개 센서값이
# 완전히 같은 행이 존재하는지
# ============================================================

lab_sensor_key = (
    lab[sensor_cols]
    .astype(str)
    .agg("|".join, axis=1)
)

unlab_sensor_key = (
    unlab[sensor_cols]
    .astype(str)
    .agg("|".join, axis=1)
)

sensor_overlap = (
    set(lab_sensor_key)
    &
    set(unlab_sensor_key)
)


print("\n" + "=" * 100)
print("EXACT SENSOR ROW OVERLAP")
print("=" * 100)

print(
    "Exact duplicated sensor patterns:",
    len(sensor_overlap)
)

print(
    "Labeled rows matching unlabeled pattern:",
    int(
        lab_sensor_key
        .isin(sensor_overlap)
        .sum()
    )
)

print(
    "Unlabeled rows matching labeled pattern:",
    int(
        unlab_sensor_key
        .isin(sensor_overlap)
        .sum()
    )
)


# ============================================================
# 7. Labeled raw 값 확인
#
# 이제 평균 0 / std 1이 나오면 안 됨
# ============================================================

print("\n" + "=" * 100)
print("LABELED RAW SENSOR CHECK")
print("=" * 100)

raw_check = (
    lab[sensor_cols]
    .agg(
        [
            "mean",
            "std",
            "min",
            "max"
        ]
    )
    .T
)

print(
    raw_check.to_string()
)


# ============================================================
# 8. Raw 분포 비교
# ============================================================

lab_stats = (
    lab[sensor_cols]
    .agg(
        ["mean", "std"]
    )
    .T
)

lab_stats.columns = [
    "Labeled_Mean",
    "Labeled_Std"
]


unlab_stats = (
    unlab[sensor_cols]
    .agg(
        ["mean", "std"]
    )
    .T
)

unlab_stats.columns = [
    "Unlabeled_Mean",
    "Unlabeled_Std"
]


compare = (
    lab_stats
    .join(unlab_stats)
)


compare["Mean_Diff_Z"] = (
    (
        compare["Unlabeled_Mean"]
        -
        compare["Labeled_Mean"]
    )
    /
    compare["Labeled_Std"]
    .replace(0, np.nan)
)


compare["Std_Ratio"] = (
    compare["Unlabeled_Std"]
    /
    compare["Labeled_Std"]
    .replace(0, np.nan)
)


print("\n" + "=" * 100)
print("RAW DISTRIBUTION COMPARISON")
print("=" * 100)

print(
    compare[
        [
            "Labeled_Mean",
            "Unlabeled_Mean",
            "Mean_Diff_Z",
            "Labeled_Std",
            "Unlabeled_Std",
            "Std_Ratio"
        ]
    ]
    .sort_values(
        "Mean_Diff_Z",
        key=lambda s: s.abs(),
        ascending=False
    )
    .to_string()
)


# ============================================================
# 9. Unlabeled 생산량 날짜별
# ============================================================

unlab["date"] = (
    unlab["TimeStamp"]
    .dt.date
)


date_counts = (
    unlab
    .groupby("date")
    .size()
)


print("\n" + "=" * 100)
print("UNLABELED DATE COUNTS")
print("=" * 100)

print(
    date_counts.to_string()
)


# ============================================================
# 10. 월별 생산량
# ============================================================

unlab["month"] = (
    unlab["TimeStamp"]
    .dt.to_period("M")
    .astype(str)
)


print("\n" + "=" * 100)
print("UNLABELED MONTH COUNTS")
print("=" * 100)

print(
    unlab["month"]
    .value_counts()
    .sort_index()
    .to_string()
)

                 과거 unlabeled RG3 RH
                        │
                  정상공정 Reference
                        │
              RobustScaler fit
                        │
        ┌───────────────┴──────────────┐   
 Isolation Forest              PCA Reconstruction   
        │                               │
     IF Score                       PCA Error   
        └───────────────┬──────────────┘      
                        │      
                  anomaly features   
                        │      
24 raw sensors + temporal + anomaly score   
                        │   
                 최종 classifier

In [ ]:
# ============================================================
# RG3 Labeled / Unlabeled
# Operating Regime Diagnosis
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. Metadata 구조 비교
# ============================================================

meta_cols = [
    "EQUIP_CD",
    "EQUIP_NAME",
    "PART_NO",
    "PART_FACT_SERIAL",
    "PART_FACT_PLAN_DATE"
]

meta_cols = [
    c for c in meta_cols
    if c in lab.columns
    and c in unlab.columns
]


print("=" * 100)
print("METADATA PROFILE")
print("=" * 100)


for col in meta_cols:

    print("\n" + "-" * 100)
    print(col)
    print("-" * 100)

    print("\n[Labeled]")
    print(
        lab[col]
        .value_counts(
            dropna=False
        )
        .head(30)
        .to_string()
    )

    print(
        "\nUnique:",
        lab[col].nunique(
            dropna=False
        )
    )


    print("\n[Unlabeled]")
    print(
        unlab[col]
        .value_counts(
            dropna=False
        )
        .head(30)
        .to_string()
    )

    print(
        "\nUnique:",
        unlab[col].nunique(
            dropna=False
        )
    )


# ============================================================
# 2. EQUIP + PART_NO 조합 확인
# ============================================================

group_cols = [
    c for c in [
        "EQUIP_CD",
        "EQUIP_NAME",
        "PART_NO"
    ]
    if c in lab.columns
    and c in unlab.columns
]


print("\n\n" + "=" * 100)
print("LABELED OPERATING GROUPS")
print("=" * 100)


lab_group = (
    lab
    .groupby(
        group_cols,
        dropna=False
    )
    .agg(
        Rows=(
            "TimeStamp",
            "size"
        ),
        Unique_Time=(
            "TimeStamp",
            "nunique"
        ),
        Start=(
            "TimeStamp",
            "min"
        ),
        End=(
            "TimeStamp",
            "max"
        )
    )
    .reset_index()
)


print(
    lab_group
    .to_string(index=False)
)


print("\n\n" + "=" * 100)
print("UNLABELED OPERATING GROUPS")
print("=" * 100)


unlab_group = (
    unlab
    .groupby(
        group_cols,
        dropna=False
    )
    .agg(
        Rows=(
            "TimeStamp",
            "size"
        ),
        Unique_Time=(
            "TimeStamp",
            "nunique"
        ),
        Start=(
            "TimeStamp",
            "min"
        ),
        End=(
            "TimeStamp",
            "max"
        )
    )
    .reset_index()
    .sort_values(
        "Rows",
        ascending=False
    )
)


print(
    unlab_group
    .to_string(index=False)
)


# ============================================================
# 3. 동일 Timestamp의 실제 Metadata 비교
# ============================================================

selected_meta = [
    c for c in [
        "_id",
        "EQUIP_CD",
        "EQUIP_NAME",
        "PART_NO",
        "PART_FACT_SERIAL",
        "PART_FACT_PLAN_DATE"
    ]
    if c in lab.columns
    and c in unlab.columns
]


selected_sensor = [
    "Injection_Time",
    "Cycle_Time",
    "Cushion_Position",
    "Max_Injection_Speed",
    "Max_Injection_Pressure",
    "Barrel_Temperature_3",
    "Mold_Temperature_3"
]


merge_cols = (
    ["TimeStamp"]
    + selected_meta
    + selected_sensor
)


overlap = (
    lab[merge_cols]
    .merge(
        unlab[merge_cols],
        on="TimeStamp",
        how="inner",
        suffixes=(
            "_LAB",
            "_UNLAB"
        )
    )
)


print("\n\n" + "=" * 100)
print("OVERLAP DETAIL")
print("=" * 100)

print(
    "Merged overlap rows:",
    len(overlap)
)


# ============================================================
# 4. Metadata 동일 여부
# ============================================================

print("\n" + "=" * 100)
print("OVERLAP METADATA MATCH RATE")
print("=" * 100)


for col in selected_meta:

    a = (
        overlap[
            f"{col}_LAB"
        ]
        .astype(str)
    )

    b = (
        overlap[
            f"{col}_UNLAB"
        ]
        .astype(str)
    )

    match_rate = (
        a.eq(b)
        .mean()
    )

    print(
        f"{col:25s}: "
        f"{match_rate:.4f}"
    )


# ============================================================
# 5. 동일 Timestamp의 Sensor 차이
# ============================================================

print("\n" + "=" * 100)
print("OVERLAP SENSOR DIFFERENCE")
print("=" * 100)


sensor_diff_rows = []


for col in selected_sensor:

    lab_v = pd.to_numeric(
        overlap[
            f"{col}_LAB"
        ],
        errors="coerce"
    )

    unlab_v = pd.to_numeric(
        overlap[
            f"{col}_UNLAB"
        ],
        errors="coerce"
    )

    abs_diff = (
        lab_v
        -
        unlab_v
    ).abs()


    sensor_diff_rows.append({

        "Feature":
            col,

        "LAB_Mean":
            lab_v.mean(),

        "UNLAB_Mean":
            unlab_v.mean(),

        "Mean_Abs_Diff":
            abs_diff.mean(),

        "Median_Abs_Diff":
            abs_diff.median(),

        "Exact_Match_Rate":
            lab_v.eq(
                unlab_v
            ).mean()
    })


sensor_diff_df = pd.DataFrame(
    sensor_diff_rows
)


print(
    sensor_diff_df
    .to_string(index=False)
)


# ============================================================
# 6. 겹치는 Timestamp 샘플 직접 확인
# ============================================================

show_cols = ["TimeStamp"]

for col in selected_meta:

    show_cols.extend(
        [
            f"{col}_LAB",
            f"{col}_UNLAB"
        ]
    )


show_cols += [
    "Injection_Time_LAB",
    "Injection_Time_UNLAB",

    "Cushion_Position_LAB",
    "Cushion_Position_UNLAB",

    "Max_Injection_Pressure_LAB",
    "Max_Injection_Pressure_UNLAB"
]


print("\n" + "=" * 100)
print("OVERLAP SAMPLE")
print("=" * 100)

print(
    overlap[
        show_cols
    ]
    .head(20)
    .to_string(index=False)
)


# ============================================================
# 7. Labeled에서 고정되어 있는 공정 Metadata 찾기
# ============================================================

stable_cols = []


for col in [
    "EQUIP_CD",
    "EQUIP_NAME",
    "PART_NO"
]:

    if (
        col in lab.columns
        and
        col in unlab.columns
        and
        lab[col].nunique(
            dropna=False
        ) == 1
    ):

        stable_cols.append(
            col
        )


print("\n" + "=" * 100)
print("STABLE LABELED PROCESS KEYS")
print("=" * 100)

print(stable_cols)


# ============================================================
# 8. Labeled와 동일 Metadata 조건으로 Unlabeled 필터링
# ============================================================

mask = pd.Series(
    True,
    index=unlab.index
)


for col in stable_cols:

    labeled_value = (
        lab[col]
        .iloc[0]
    )

    mask &= (
        unlab[col]
        ==
        labeled_value
    )


unlab_same_process = (
    unlab[
        mask
    ]
    .copy()
    .sort_values(
        "TimeStamp"
    )
)


print("\n" + "=" * 100)
print("SAME PROCESS UNLABELED")
print("=" * 100)

print(
    "Rows:",
    len(
        unlab_same_process
    )
)

print(
    "Unique TimeStamp:",
    unlab_same_process[
        "TimeStamp"
    ].nunique()
)


if len(unlab_same_process) > 0:

    print(
        "Time:",
        unlab_same_process[
            "TimeStamp"
        ].min(),
        "~",
        unlab_same_process[
            "TimeStamp"
        ].max()
    )


# ============================================================
# 9. 미래정보 없는 Reference 후보
#
# Labeled 시작일보다 과거 데이터만
# ============================================================

labeled_start = (
    lab["TimeStamp"]
    .min()
)


unlab_reference = (
    unlab_same_process[
        unlab_same_process[
            "TimeStamp"
        ]
        <
        labeled_start
    ]
    .copy()
)


print("\n" + "=" * 100)
print("PAST REFERENCE CANDIDATES")
print("=" * 100)

print(
    "Rows:",
    len(
        unlab_reference
    )
)


if len(unlab_reference) > 0:

    print(
        "Time:",
        unlab_reference[
            "TimeStamp"
        ].min(),
        "~",
        unlab_reference[
            "TimeStamp"
        ].max()
    )


# ============================================================
# 10. 동일 공정 subset에서 분포 다시 비교
# ============================================================

if len(unlab_same_process) > 0:

    rows = []


    for col in sensor_cols:

        a = pd.to_numeric(
            lab[col],
            errors="coerce"
        )

        b = pd.to_numeric(
            unlab_same_process[col],
            errors="coerce"
        )


        a_std = a.std()
        b_std = b.std()


        pooled_std = np.sqrt(
            (
                a_std ** 2
                +
                b_std ** 2
            )
            / 2
        )


        if (
            pooled_std
            > 0
        ):

            smd = (
                b.mean()
                -
                a.mean()
            ) / pooled_std

        else:

            smd = np.nan


        in_lab_range = (
            (
                b
                >=
                a.min()
            )
            &
            (
                b
                <=
                a.max()
            )
        ).mean()


        rows.append({

            "Feature":
                col,

            "Labeled_Mean":
                a.mean(),

            "Matched_Unlab_Mean":
                b.mean(),

            "Pooled_SMD":
                smd,

            "Labeled_Std":
                a_std,

            "Matched_Unlab_Std":
                b_std,

            "Unlab_In_Labeled_Range":
                in_lab_range
        })


    matched_compare = (
        pd.DataFrame(
            rows
        )
    )


    print(
        "\n"
        + "=" * 100
    )

    print(
        "MATCHED PROCESS DISTRIBUTION"
    )

    print(
        "=" * 100
    )


    print(
        matched_compare
        .sort_values(
            "Pooled_SMD",
            key=lambda s: s.abs(),
            ascending=False
        )
        .to_string(
            index=False
        )
    )

In [ ]:
# ============================================================
# RG3 Local Normal Reference Diagnosis
# - Labeled와 겹치는 Timestamp 완전 제거
# - 미래 Unlabeled 제거
# - Train 정상 Cycle과 유사한 Unlabeled만 탐색
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors


# ============================================================
# 1. Labeled 정리
# ============================================================

lab2 = (
    lab
    .copy()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


# Session 재생성
lab2["gap_min"] = (
    lab2["TimeStamp"]
    .diff()
    .dt.total_seconds()
    / 60
)

lab2["session_id"] = (
    (lab2["gap_min"] > 5)
    .cumsum()
)


print("=" * 100)
print("LABELED SESSION")
print("=" * 100)

print(
    lab2
    .groupby("session_id")
    .agg(
        Start=("TimeStamp", "min"),
        End=("TimeStamp", "max"),
        N=("PassOrFail", "size"),
        Defects=("PassOrFail", "sum")
    )
    .to_string()
)


# ============================================================
# 2. Unlabeled Leakage 제거
#
# 중요:
# Labeled에 존재하는 Timestamp는
# Train/Valid 여부와 관계없이 전부 제거
# ============================================================

all_labeled_times = set(
    lab2["TimeStamp"]
)


unlab_clean = (
    unlab[
        ~unlab["TimeStamp"]
        .isin(all_labeled_times)
    ]
    .copy()
)


print("\n" + "=" * 100)
print("LEAKAGE REMOVAL")
print("=" * 100)

print(
    "Original unlabeled:",
    len(unlab)
)

print(
    "Removed timestamp-overlap rows:",
    len(unlab) - len(unlab_clean)
)

print(
    "Remaining:",
    len(unlab_clean)
)


# ============================================================
# 3. Unlabeled 내부 동일 Timestamp 중복 처리
#
# RH 하나인데 동일 Timestamp가 중복되어 있는 경우
# 센서 평균으로 하나의 Cycle로 통합
# ============================================================

agg_dict = {
    col: "mean"
    for col in sensor_cols
}


# 분석용 메타도 일부 보존
for col in [
    "PART_FACT_SERIAL",
    "PART_FACT_PLAN_DATE"
]:
    if col in unlab_clean.columns:
        agg_dict[col] = "first"


unlab_clean = (
    unlab_clean
    .groupby(
        "TimeStamp",
        as_index=False
    )
    .agg(
        agg_dict
    )
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


print(
    "After timestamp dedup:",
    len(unlab_clean)
)


# ============================================================
# 4. Fold 정의
# ============================================================

valid_sessions = [
    2, 3, 4, 5, 6
]


reference_results = []


# ============================================================
# 5. Fold별 Local Reference 탐색
# ============================================================

for valid_session in valid_sessions:

    print(
        "\n\n"
        + "=" * 100
    )

    print(
        f"VALID SESSION {valid_session}"
    )

    print(
        "=" * 100
    )


    # --------------------------------------------------------
    # Train / Valid
    # --------------------------------------------------------

    train = (
        lab2[
            lab2["session_id"]
            <
            valid_session
        ]
        .copy()
    )

    valid = (
        lab2[
            lab2["session_id"]
            ==
            valid_session
        ]
        .copy()
    )


    # Train 정상만 reference 기준으로 사용
    train_normal = (
        train[
            train["PassOrFail"]
            ==
            0
        ]
        .copy()
    )


    valid_start = (
        valid["TimeStamp"]
        .min()
    )


    # --------------------------------------------------------
    # Validation 시작 이전의 Unlabeled만 사용
    # --------------------------------------------------------

    candidates = (
        unlab_clean[
            unlab_clean["TimeStamp"]
            <
            valid_start
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Train 정상에서 변화가 존재하는 센서만
    # --------------------------------------------------------

    usable_features = []

    for col in sensor_cols:

        s = pd.to_numeric(
            train_normal[col],
            errors="coerce"
        )

        if (
            s.nunique(dropna=True) > 1
            and
            s.std() > 1e-12
        ):
            usable_features.append(
                col
            )


    print(
        "Train:",
        len(train),
        "| defects:",
        int(train["PassOrFail"].sum())
    )

    print(
        "Train normals:",
        len(train_normal)
    )

    print(
        "Valid:",
        len(valid),
        "| defects:",
        int(valid["PassOrFail"].sum())
    )

    print(
        "Past unlabeled candidates:",
        len(candidates)
    )

    print(
        "Usable sensors:",
        len(usable_features)
    )


    # --------------------------------------------------------
    # Scaling
    #
    # fit은 오직 Train 정상
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train_normal = (
        scaler.fit_transform(
            train_normal[
                usable_features
            ]
        )
    )

    X_candidate = (
        scaler.transform(
            candidates[
                usable_features
            ]
        )
    )


    # ========================================================
    # 6. Train 정상 내부의 KNN 거리 분포
    #
    # 각 정상 Cycle이 다른 정상 Cycle들과
    # 얼마나 떨어져 있는지 계산
    # ========================================================

    k = min(
        5,
        len(train_normal) - 1
    )


    nn_train = NearestNeighbors(
        n_neighbors=k + 1,
        metric="euclidean"
    )

    nn_train.fit(
        X_train_normal
    )


    train_distances, _ = (
        nn_train.kneighbors(
            X_train_normal
        )
    )


    # 첫 번째는 자기 자신 거리 0
    train_k_distance = (
        train_distances[:, k]
    )


    q90 = np.quantile(
        train_k_distance,
        0.90
    )

    q95 = np.quantile(
        train_k_distance,
        0.95
    )

    q99 = np.quantile(
        train_k_distance,
        0.99
    )


    print("\nTrain normal KNN distance")

    print(
        f"Q90 = {q90:.4f}"
    )

    print(
        f"Q95 = {q95:.4f}"
    )

    print(
        f"Q99 = {q99:.4f}"
    )


    # ========================================================
    # 7. Unlabeled -> Train 정상까지 거리
    # ========================================================

    nn_ref = NearestNeighbors(
        n_neighbors=k,
        metric="euclidean"
    )

    nn_ref.fit(
        X_train_normal
    )


    candidate_distances, _ = (
        nn_ref.kneighbors(
            X_candidate
        )
    )


    candidate_k_distance = (
        candidate_distances[
            :, -1
        ]
    )


    candidates["knn_distance"] = (
        candidate_k_distance
    )


    # ========================================================
    # 8. 여러 threshold에서 Reference 수 확인
    # ========================================================

    thresholds = {

        "Q90":
            q90,

        "Q95":
            q95,

        "Q99":
            q99,

        "Q95_x1.5":
            q95 * 1.5,

        "Q95_x2":
            q95 * 2,

        "Q95_x3":
            q95 * 3
    }


    print(
        "\nLOCAL REFERENCE COUNTS"
    )


    row_result = {

        "Valid_Session":
            valid_session,

        "Train_N":
            len(train),

        "Train_Defects":
            int(
                train["PassOrFail"]
                .sum()
            ),

        "Train_Normal":
            len(train_normal),

        "Valid_N":
            len(valid),

        "Valid_Defects":
            int(
                valid["PassOrFail"]
                .sum()
            ),

        "Past_Unlabeled":
            len(candidates),

        "N_Features":
            len(usable_features),

        "KNN_Q95":
            q95
    }


    for name, threshold in thresholds.items():

        selected = (
            candidates[
                candidates["knn_distance"]
                <=
                threshold
            ]
        )


        count = (
            len(selected)
        )


        ratio = (
            count
            /
            len(candidates)
            if len(candidates) > 0
            else np.nan
        )


        print(
            f"{name:12s}"
            f" threshold={threshold:8.4f}"
            f" | N={count:6d}"
            f" | {ratio*100:6.2f}%"
        )


        row_result[
            f"N_{name}"
        ] = count


    # ========================================================
    # 9. Q95 선택 데이터 시간 범위
    # ========================================================

    selected_q95 = (
        candidates[
            candidates[
                "knn_distance"
            ]
            <=
            q95
        ]
    )


    if len(selected_q95) > 0:

        print(
            "\nQ95 Reference Time:"
        )

        print(
            selected_q95[
                "TimeStamp"
            ].min(),
            "~",
            selected_q95[
                "TimeStamp"
            ].max()
        )


        # Serial도 확인
        if (
            "PART_FACT_SERIAL"
            in selected_q95.columns
        ):

            print(
                "\nQ95 Serial:"
            )

            print(
                selected_q95[
                    "PART_FACT_SERIAL"
                ]
                .value_counts()
                .head(20)
                .to_string()
            )


    reference_results.append(
        row_result
    )


# ============================================================
# 10. 전체 요약
# ============================================================

reference_summary = (
    pd.DataFrame(
        reference_results
    )
)


print(
    "\n\n"
    + "=" * 100
)

print(
    "LOCAL REFERENCE SUMMARY"
)

print(
    "=" * 100
)


print(
    reference_summary
    .to_string(
        index=False
    )
)

In [ ]:
# ============================================================
# RG3 Semi-supervised Fusion Model
#
# Local Unlabeled Reference
#   -> Isolation Forest
#   -> PCA Reconstruction Error
#   -> KNN Distance
#
# 이후
# Sensor / Temporal / Temporal+Anomaly 비교
#
# Validation:
# Past Sessions -> Future Session
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import (
    IsolationForest,
    ExtraTreesClassifier,
    RandomForestClassifier
)

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)


# ============================================================
# 1. Labeled 데이터 준비
# ============================================================

data = (
    lab
    .copy()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Session
# ------------------------------------------------------------

data["gap_min"] = (
    data["TimeStamp"]
    .diff()
    .dt.total_seconds()
    / 60
)

data["session_id"] = (
    (data["gap_min"] > 5)
    .cumsum()
)


# ============================================================
# 2. 완전 상수 센서 제거
# ============================================================

raw_features = [
    c for c in sensor_cols
    if data[c].nunique(dropna=False) > 1
]


print("=" * 100)
print("FEATURES")
print("=" * 100)

print("Original:", len(sensor_cols))
print("Usable  :", len(raw_features))

print("\nRemoved:")
print(
    [
        c for c in sensor_cols
        if c not in raw_features
    ]
)


# ============================================================
# 3. Temporal Feature
#
# 반드시 Session 내부에서만 계산
# ============================================================

for col in raw_features:

    # 직전 cycle
    lag1 = (
        data
        .groupby("session_id")[col]
        .shift(1)
    )

    # 직전 대비 변화
    data[f"{col}_diff1"] = (
        data[col]
        -
        lag1
    )

    # 과거 3 cycle 평균
    prev_mean3 = (
        data
        .groupby("session_id")[col]
        .transform(
            lambda s:
            s.shift(1)
            .rolling(
                3,
                min_periods=1
            )
            .mean()
        )
    )

    # 과거 3 cycle 표준편차
    prev_std3 = (
        data
        .groupby("session_id")[col]
        .transform(
            lambda s:
            s.shift(1)
            .rolling(
                3,
                min_periods=2
            )
            .std(ddof=0)
        )
    )

    # 최근 정상수준 대비 편차
    data[f"{col}_dev3"] = (
        data[col]
        -
        prev_mean3
    )

    data[f"{col}_std3"] = (
        prev_std3
    )


diff_features = [
    f"{c}_diff1"
    for c in raw_features
]

dev_features = [
    f"{c}_dev3"
    for c in raw_features
]

std_features = [
    f"{c}_std3"
    for c in raw_features
]


temporal_features = (
    raw_features
    +
    diff_features
    +
    dev_features
    +
    std_features
)


# ============================================================
# 4. Unlabeled 누수 제거
# ============================================================

all_labeled_times = set(
    data["TimeStamp"]
)


unlab_clean = (
    unlab[
        ~unlab["TimeStamp"]
        .isin(all_labeled_times)
    ]
    .copy()
)


# 같은 Timestamp 중복 -> 평균
unlab_clean = (
    unlab_clean
    .groupby(
        "TimeStamp",
        as_index=False
    )[raw_features]
    .mean()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


print("\n" + "=" * 100)
print("UNLABELED")
print("=" * 100)

print(
    "Clean unlabeled:",
    len(unlab_clean)
)


# ============================================================
# 5. 모델
# ============================================================

def get_models():

    return {

        "Logistic":

            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    )
                ),
                (
                    "scaler",
                    StandardScaler()
                ),
                (
                    "model",
                    LogisticRegression(
                        C=0.5,
                        class_weight="balanced",
                        max_iter=3000,
                        random_state=42
                    )
                )
            ]),


        "RandomForest":

            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    )
                ),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=500,
                        max_depth=6,
                        min_samples_leaf=2,
                        max_features="sqrt",
                        class_weight="balanced_subsample",
                        random_state=42,
                        n_jobs=-1
                    )
                )
            ]),


        "ExtraTrees":

            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    )
                ),
                (
                    "model",
                    ExtraTreesClassifier(
                        n_estimators=500,
                        max_depth=6,
                        min_samples_leaf=2,
                        max_features="sqrt",
                        class_weight="balanced",
                        random_state=42,
                        n_jobs=-1
                    )
                )
            ])
    }


# ============================================================
# 6. 결과 저장
# ============================================================

results = []

oof_list = []

reference_log = []


# ============================================================
# 7. Walk-forward
# ============================================================

for valid_session in range(2, 7):

    print(
        "\n\n"
        + "=" * 100
    )

    print(
        f"VALID SESSION {valid_session}"
    )

    print(
        "=" * 100
    )


    # ========================================================
    # Train / Validation
    # ========================================================

    train_mask = (
        data["session_id"]
        <
        valid_session
    )

    valid_mask = (
        data["session_id"]
        ==
        valid_session
    )


    train = (
        data[
            train_mask
        ]
        .copy()
    )

    valid = (
        data[
            valid_mask
        ]
        .copy()
    )


    train_normal = (
        train[
            train["PassOrFail"]
            ==
            0
        ]
        .copy()
    )


    valid_start = (
        valid["TimeStamp"]
        .min()
    )


    # ========================================================
    # 8. Validation 이전 Unlabeled만
    # ========================================================

    candidates = (
        unlab_clean[
            unlab_clean["TimeStamp"]
            <
            valid_start
        ]
        .copy()
    )


    # ========================================================
    # 9. Train normal scaling
    # ========================================================

    scaler_ref = (
        StandardScaler()
    )


    X_normal = (
        scaler_ref
        .fit_transform(
            train_normal[
                raw_features
            ]
        )
    )


    X_candidate = (
        scaler_ref
        .transform(
            candidates[
                raw_features
            ]
        )
    )


    # ========================================================
    # 10. Local Reference Selection
    #
    # Train normal KNN Q95
    # ========================================================

    k_select = min(
        5,
        len(train_normal) - 1
    )


    nn_train = (
        NearestNeighbors(
            n_neighbors=
                k_select + 1
        )
    )

    nn_train.fit(
        X_normal
    )


    train_dist, _ = (
        nn_train
        .kneighbors(
            X_normal
        )
    )


    q95 = (
        np.quantile(
            train_dist[:, k_select],
            0.95
        )
    )


    nn_select = (
        NearestNeighbors(
            n_neighbors=k_select
        )
    )

    nn_select.fit(
        X_normal
    )


    candidate_dist, _ = (
        nn_select
        .kneighbors(
            X_candidate
        )
    )


    candidate_kdist = (
        candidate_dist[:, -1]
    )


    reference_mask = (
        candidate_kdist
        <=
        q95
    )


    reference = (
        candidates.loc[
            reference_mask
        ]
        .copy()
    )


    print(
        "Train:",
        len(train),
        "| Defects:",
        int(
            train[
                "PassOrFail"
            ].sum()
        )
    )

    print(
        "Valid:",
        len(valid),
        "| Defects:",
        int(
            valid[
                "PassOrFail"
            ].sum()
        )
    )

    print(
        "Past candidates:",
        len(candidates)
    )

    print(
        "Local reference:",
        len(reference)
    )

    print(
        "Q95:",
        round(q95, 4)
    )


    reference_log.append({

        "Valid_Session":
            valid_session,

        "Train_N":
            len(train),

        "Train_Defects":
            int(
                train[
                    "PassOrFail"
                ].sum()
            ),

        "Valid_N":
            len(valid),

        "Valid_Defects":
            int(
                valid[
                    "PassOrFail"
                ].sum()
            ),

        "Candidates":
            len(candidates),

        "Reference_N":
            len(reference),

        "Q95":
            q95
    })


    # ========================================================
    # 11. Anomaly 모델용 scaling
    #
    # fit은 Local Unlabeled Reference에만 수행
    #
    # Train/Valid 모두 unsupervised model 입장에서
    # 외부 데이터가 됨
    # ========================================================

    anomaly_scaler = (
        StandardScaler()
    )


    X_ref = (
        anomaly_scaler
        .fit_transform(
            reference[
                raw_features
            ]
        )
    )


    X_train_raw = (
        anomaly_scaler
        .transform(
            train[
                raw_features
            ]
        )
    )


    X_valid_raw = (
        anomaly_scaler
        .transform(
            valid[
                raw_features
            ]
        )
    )


    # ========================================================
    # 12. Isolation Forest
    # ========================================================

    iso = (
        IsolationForest(
            n_estimators=500,
            contamination="auto",
            max_samples="auto",
            random_state=42,
            n_jobs=-1
        )
    )


    iso.fit(
        X_ref
    )


    # sklearn score_samples는 높을수록 정상
    # 따라서 - 부호 → 높을수록 이상
    train_if = (
        -iso.score_samples(
            X_train_raw
        )
    )

    valid_if = (
        -iso.score_samples(
            X_valid_raw
        )
    )


    # ========================================================
    # 13. PCA Reconstruction
    #
    # 분산 95% 설명
    # ========================================================

    pca = (
        PCA(
            n_components=0.95,
            random_state=42
        )
    )


    X_ref_pca = (
        pca.fit_transform(
            X_ref
        )
    )


    # Train
    train_pca = (
        pca.transform(
            X_train_raw
        )
    )

    train_recon = (
        pca.inverse_transform(
            train_pca
        )
    )

    train_pca_error = (
        np.mean(
            (
                X_train_raw
                -
                train_recon
            ) ** 2,
            axis=1
        )
    )


    # Valid
    valid_pca = (
        pca.transform(
            X_valid_raw
        )
    )

    valid_recon = (
        pca.inverse_transform(
            valid_pca
        )
    )

    valid_pca_error = (
        np.mean(
            (
                X_valid_raw
                -
                valid_recon
            ) ** 2,
            axis=1
        )
    )


    # ========================================================
    # 14. KNN Anomaly Score
    #
    # Local Reference까지의 평균거리
    # ========================================================

    k_anom = min(
        10,
        len(reference)
    )


    nn_anomaly = (
        NearestNeighbors(
            n_neighbors=k_anom,
            metric="euclidean"
        )
    )


    nn_anomaly.fit(
        X_ref
    )


    train_knn_dist, _ = (
        nn_anomaly
        .kneighbors(
            X_train_raw
        )
    )


    valid_knn_dist, _ = (
        nn_anomaly
        .kneighbors(
            X_valid_raw
        )
    )


    train_knn_score = (
        train_knn_dist
        .mean(axis=1)
    )

    valid_knn_score = (
        valid_knn_dist
        .mean(axis=1)
    )


    # ========================================================
    # 15. Anomaly Feature 부착
    # ========================================================

    train = (
        train.copy()
    )

    valid = (
        valid.copy()
    )


    train[
        "anomaly_if"
    ] = train_if

    valid[
        "anomaly_if"
    ] = valid_if


    train[
        "anomaly_pca"
    ] = train_pca_error

    valid[
        "anomaly_pca"
    ] = valid_pca_error


    train[
        "anomaly_knn"
    ] = train_knn_score

    valid[
        "anomaly_knn"
    ] = valid_knn_score


    anomaly_features = [
        "anomaly_if",
        "anomaly_pca",
        "anomaly_knn"
    ]


    # ========================================================
    # 16. Feature Set
    # ========================================================

    feature_sets = {

        "V0_sensor":
            raw_features,

        "V1_temporal":
            temporal_features,

        "V2_fusion":
            (
                temporal_features
                +
                anomaly_features
            ),

        # 이상점수 자체에 실제 정보가 있는지 확인
        "V3_anomaly_only":
            anomaly_features
    }


    # ========================================================
    # 17. 모델 평가
    # ========================================================

    y_train = (
        train[
            "PassOrFail"
        ]
        .astype(int)
    )

    y_valid = (
        valid[
            "PassOrFail"
        ]
        .astype(int)
    )


    baseline_pr = (
        y_valid.mean()
    )


    for (
        feature_name,
        features
    ) in feature_sets.items():


        models = (
            get_models()
        )


        for (
            model_name,
            model
        ) in models.items():


            X_train = (
                train[
                    features
                ]
            )

            X_valid = (
                valid[
                    features
                ]
            )


            model.fit(
                X_train,
                y_train
            )


            pred = (
                model
                .predict_proba(
                    X_valid
                )[:, 1]
            )


            pr_auc = (
                average_precision_score(
                    y_valid,
                    pred
                )
            )


            roc_auc = (
                roc_auc_score(
                    y_valid,
                    pred
                )
            )


            results.append({

                "Valid_Session":
                    valid_session,

                "Feature_Set":
                    feature_name,

                "Model":
                    model_name,

                "Train_N":
                    len(train),

                "Train_Defects":
                    int(
                        y_train.sum()
                    ),

                "Valid_N":
                    len(valid),

                "Valid_Defects":
                    int(
                        y_valid.sum()
                    ),

                "Reference_N":
                    len(reference),

                "Baseline_PR":
                    baseline_pr,

                "PR_AUC":
                    pr_auc,

                "PR_Ratio":
                    (
                        pr_auc
                        /
                        baseline_pr
                    ),

                "ROC_AUC":
                    roc_auc
            })


            temp = pd.DataFrame({

                "TimeStamp":
                    valid[
                        "TimeStamp"
                    ].values,

                "session_id":
                    valid[
                        "session_id"
                    ].values,

                "y_true":
                    y_valid.values,

                "y_prob":
                    pred,

                "Feature_Set":
                    feature_name,

                "Model":
                    model_name
            })


            oof_list.append(
                temp
            )


# ============================================================
# 18. 결과
# ============================================================

result_df = (
    pd.DataFrame(
        results
    )
)


oof_df = (
    pd.concat(
        oof_list,
        ignore_index=True
    )
)


# ============================================================
# 19. Fold 결과
# ============================================================

print(
    "\n\n"
    + "=" * 100
)

print(
    "FOLD RESULTS"
)

print(
    "=" * 100
)


print(
    result_df[
        [
            "Valid_Session",
            "Feature_Set",
            "Model",
            "Valid_Defects",
            "Reference_N",
            "Baseline_PR",
            "PR_AUC",
            "PR_Ratio",
            "ROC_AUC"
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 20. Fold Summary
# ============================================================

summary = (
    result_df
    .groupby(
        [
            "Feature_Set",
            "Model"
        ]
    )
    .agg(

        PR_Mean=(
            "PR_AUC",
            "mean"
        ),

        PR_Median=(
            "PR_AUC",
            "median"
        ),

        PR_Min=(
            "PR_AUC",
            "min"
        ),

        PR_Max=(
            "PR_AUC",
            "max"
        ),

        PR_Std=(
            "PR_AUC",
            "std"
        ),

        Ratio_Mean=(
            "PR_Ratio",
            "mean"
        ),

        ROC_Mean=(
            "ROC_AUC",
            "mean"
        ),

        ROC_Min=(
            "ROC_AUC",
            "min"
        )
    )
    .reset_index()
    .sort_values(
        [
            "PR_Median",
            "PR_Mean"
        ],
        ascending=False
    )
)


print(
    "\n\n"
    + "=" * 100
)

print(
    "MODEL SUMMARY"
)

print(
    "=" * 100
)


print(
    summary.to_string(
        index=False
    )
)


# ============================================================
# 21. 전체 OOF
# ============================================================

oof_results = []


for (
    feature_name,
    model_name
), group in oof_df.groupby(
    [
        "Feature_Set",
        "Model"
    ]
):


    baseline = (
        group[
            "y_true"
        ]
        .mean()
    )


    pr_auc = (
        average_precision_score(
            group[
                "y_true"
            ],
            group[
                "y_prob"
            ]
        )
    )


    roc_auc = (
        roc_auc_score(
            group[
                "y_true"
            ],
            group[
                "y_prob"
            ]
        )
    )


    oof_results.append({

        "Feature_Set":
            feature_name,

        "Model":
            model_name,

        "N":
            len(group),

        "Defects":
            int(
                group[
                    "y_true"
                ]
                .sum()
            ),

        "Baseline_PR":
            baseline,

        "OOF_PR_AUC":
            pr_auc,

        "PR_Ratio":
            (
                pr_auc
                /
                baseline
            ),

        "OOF_ROC_AUC":
            roc_auc
    })


oof_summary = (
    pd.DataFrame(
        oof_results
    )
    .sort_values(
        "OOF_PR_AUC",
        ascending=False
    )
)


print(
    "\n\n"
    + "=" * 100
)

print(
    "OOF SUMMARY"
)

print(
    "=" * 100
)


print(
    oof_summary.to_string(
        index=False
    )
)


# ============================================================
# 22. Reference 로그
# ============================================================

print(
    "\n\n"
    + "=" * 100
)

print(
    "REFERENCE SUMMARY"
)

print(
    "=" * 100
)


print(
    pd.DataFrame(
        reference_log
    )
    .to_string(
        index=False
    )
)

In [ ]:
# ============================================================
# Anomaly Feature Ablation
# ============================================================

anomaly_feature_sets = {

    "IF_only":
        ["anomaly_if"],

    "PCA_only":
        ["anomaly_pca"],

    "KNN_only":
        ["anomaly_knn"],

    "IF_PCA":
        [
            "anomaly_if",
            "anomaly_pca"
        ],

    "IF_KNN":
        [
            "anomaly_if",
            "anomaly_knn"
        ],

    "PCA_KNN":
        [
            "anomaly_pca",
            "anomaly_knn"
        ],

    "ALL_3":
        [
            "anomaly_if",
            "anomaly_pca",
            "anomaly_knn"
        ]
}

In [ ]:
model = ExtraTreesClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [ ]:
# ============================================================
# RG3 FINAL ANOMALY ABLATION + THRESHOLD ANALYSIS
#
# 필요 변수:
#   lab
#   unlab
#   sensor_cols
#
# 출력:
# 1) Fold별 Anomaly Ablation
# 2) OOF Ablation Summary
# 3) 최종 Feature Set
# 4) Threshold Table
# 5) Recall 80% / 90% 운영점
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import (
    IsolationForest,
    ExtraTreesClassifier
)
from sklearn.decomposition import PCA

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# ============================================================
# 1. Labeled 준비
# ============================================================

data = (
    lab
    .copy()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)

data["TimeStamp"] = pd.to_datetime(
    data["TimeStamp"]
)


# ------------------------------------------------------------
# Session 생성
# ------------------------------------------------------------

data["gap_min"] = (
    data["TimeStamp"]
    .diff()
    .dt.total_seconds()
    / 60
)

data["session_id"] = (
    (data["gap_min"] > 5)
    .cumsum()
)


# ============================================================
# 2. 사용 가능한 센서
# ============================================================

raw_features = [
    c for c in sensor_cols
    if data[c].nunique(dropna=False) > 1
]


print("=" * 100)
print("FEATURES")
print("=" * 100)

print("Original:", len(sensor_cols))
print("Usable  :", len(raw_features))

print(
    "Removed :",
    [
        c
        for c in sensor_cols
        if c not in raw_features
    ]
)


# ============================================================
# 3. Unlabeled 정리
# ============================================================

unlab2 = (
    unlab
    .copy()
)

unlab2["TimeStamp"] = pd.to_datetime(
    unlab2["TimeStamp"]
)


# ------------------------------------------------------------
# Labeled와 Timestamp 겹치는 데이터 제거
# ------------------------------------------------------------

labeled_times = set(
    data["TimeStamp"]
)

unlab_clean = (
    unlab2[
        ~unlab2["TimeStamp"]
        .isin(labeled_times)
    ]
    .copy()
)


# ------------------------------------------------------------
# 동일 Timestamp 중복 → 평균
# ------------------------------------------------------------

unlab_clean = (
    unlab_clean
    .groupby(
        "TimeStamp",
        as_index=False
    )[raw_features]
    .mean()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


print("\n" + "=" * 100)
print("UNLABELED")
print("=" * 100)

print(
    "Clean unlabeled:",
    len(unlab_clean)
)


# ============================================================
# 4. Ablation 조합
# ============================================================

feature_sets = {

    "IF_only":
        ["anomaly_if"],

    "PCA_only":
        ["anomaly_pca"],

    "KNN_only":
        ["anomaly_knn"],

    "IF_PCA":
        [
            "anomaly_if",
            "anomaly_pca"
        ],

    "IF_KNN":
        [
            "anomaly_if",
            "anomaly_knn"
        ],

    "PCA_KNN":
        [
            "anomaly_pca",
            "anomaly_knn"
        ],

    "ALL_3":
        [
            "anomaly_if",
            "anomaly_pca",
            "anomaly_knn"
        ]
}


# ============================================================
# 5. 저장
# ============================================================

fold_results = []

oof_predictions = []


# ============================================================
# 6. Walk-forward
# ============================================================

for valid_session in range(2, 7):

    print(
        "\n\n"
        + "=" * 100
    )

    print(
        f"VALID SESSION {valid_session}"
    )

    print(
        "=" * 100
    )


    # --------------------------------------------------------
    # Train / Valid
    # --------------------------------------------------------

    train = (
        data[
            data["session_id"]
            <
            valid_session
        ]
        .copy()
    )

    valid = (
        data[
            data["session_id"]
            ==
            valid_session
        ]
        .copy()
    )


    train_normal = (
        train[
            train["PassOrFail"]
            ==
            0
        ]
        .copy()
    )


    valid_start = (
        valid["TimeStamp"]
        .min()
    )


    # ========================================================
    # 7. 미래정보 없는 과거 Unlabeled
    # ========================================================

    candidates = (
        unlab_clean[
            unlab_clean["TimeStamp"]
            <
            valid_start
        ]
        .copy()
    )


    # ========================================================
    # 8. Local Reference 선택
    #
    # scaler fit = Train Normal만
    # ========================================================

    selector_scaler = (
        StandardScaler()
    )


    X_normal = (
        selector_scaler
        .fit_transform(
            train_normal[
                raw_features
            ]
        )
    )

    X_candidates = (
        selector_scaler
        .transform(
            candidates[
                raw_features
            ]
        )
    )


    # --------------------------------------------------------
    # Train 정상 내부 KNN 거리
    # --------------------------------------------------------

    k_select = min(
        5,
        len(train_normal) - 1
    )


    nn_train = (
        NearestNeighbors(
            n_neighbors=
                k_select + 1,
            metric="euclidean"
        )
    )


    nn_train.fit(
        X_normal
    )


    train_dist, _ = (
        nn_train
        .kneighbors(
            X_normal
        )
    )


    q95 = (
        np.quantile(
            train_dist[:, k_select],
            0.95
        )
    )


    # --------------------------------------------------------
    # Unlabeled → Train normal 거리
    # --------------------------------------------------------

    nn_select = (
        NearestNeighbors(
            n_neighbors=k_select,
            metric="euclidean"
        )
    )


    nn_select.fit(
        X_normal
    )


    candidate_dist, _ = (
        nn_select
        .kneighbors(
            X_candidates
        )
    )


    candidate_kdist = (
        candidate_dist[:, -1]
    )


    reference_mask = (
        candidate_kdist
        <=
        q95
    )


    reference = (
        candidates
        .loc[
            reference_mask
        ]
        .copy()
    )


    print(
        "Train:",
        len(train),
        "| Defects:",
        int(
            train["PassOrFail"]
            .sum()
        )
    )

    print(
        "Valid:",
        len(valid),
        "| Defects:",
        int(
            valid["PassOrFail"]
            .sum()
        )
    )

    print(
        "Candidates:",
        len(candidates)
    )

    print(
        "Reference:",
        len(reference)
    )

    print(
        "Q95:",
        round(q95, 4)
    )


    # ========================================================
    # 9. Anomaly scaling
    #
    # fit = Local Reference만
    # ========================================================

    anomaly_scaler = (
        StandardScaler()
    )


    X_ref = (
        anomaly_scaler
        .fit_transform(
            reference[
                raw_features
            ]
        )
    )


    X_train = (
        anomaly_scaler
        .transform(
            train[
                raw_features
            ]
        )
    )


    X_valid = (
        anomaly_scaler
        .transform(
            valid[
                raw_features
            ]
        )
    )


    # ========================================================
    # 10. Isolation Forest
    # ========================================================

    iso = (
        IsolationForest(
            n_estimators=500,
            contamination="auto",
            random_state=42,
            n_jobs=-1
        )
    )


    iso.fit(
        X_ref
    )


    train_if = (
        -iso.score_samples(
            X_train
        )
    )

    valid_if = (
        -iso.score_samples(
            X_valid
        )
    )


    # ========================================================
    # 11. PCA Reconstruction Error
    # ========================================================

    pca = (
        PCA(
            n_components=0.95,
            random_state=42
        )
    )


    pca.fit(
        X_ref
    )


    # Train
    z_train = (
        pca.transform(
            X_train
        )
    )

    recon_train = (
        pca.inverse_transform(
            z_train
        )
    )

    train_pca = (
        np.mean(
            (
                X_train
                -
                recon_train
            ) ** 2,
            axis=1
        )
    )


    # Valid
    z_valid = (
        pca.transform(
            X_valid
        )
    )

    recon_valid = (
        pca.inverse_transform(
            z_valid
        )
    )

    valid_pca = (
        np.mean(
            (
                X_valid
                -
                recon_valid
            ) ** 2,
            axis=1
        )
    )


    # ========================================================
    # 12. KNN Anomaly Score
    # ========================================================

    k_anomaly = min(
        10,
        len(reference)
    )


    nn_anomaly = (
        NearestNeighbors(
            n_neighbors=k_anomaly,
            metric="euclidean"
        )
    )


    nn_anomaly.fit(
        X_ref
    )


    train_knn_dist, _ = (
        nn_anomaly
        .kneighbors(
            X_train
        )
    )

    valid_knn_dist, _ = (
        nn_anomaly
        .kneighbors(
            X_valid
        )
    )


    train_knn = (
        train_knn_dist
        .mean(axis=1)
    )

    valid_knn = (
        valid_knn_dist
        .mean(axis=1)
    )


    # ========================================================
    # 13. Anomaly Feature DataFrame
    # ========================================================

    train_anom = pd.DataFrame({

        "anomaly_if":
            train_if,

        "anomaly_pca":
            train_pca,

        "anomaly_knn":
            train_knn
    })


    valid_anom = pd.DataFrame({

        "anomaly_if":
            valid_if,

        "anomaly_pca":
            valid_pca,

        "anomaly_knn":
            valid_knn
    })


    y_train = (
        train["PassOrFail"]
        .astype(int)
        .reset_index(drop=True)
    )


    y_valid = (
        valid["PassOrFail"]
        .astype(int)
        .reset_index(drop=True)
    )


    baseline = (
        y_valid.mean()
    )


    # ========================================================
    # 14. Ablation
    # ========================================================

    for (
        feature_name,
        features
    ) in feature_sets.items():


        model = (
            ExtraTreesClassifier(
                n_estimators=500,
                max_depth=6,
                min_samples_leaf=2,
                max_features="sqrt",
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )


        model.fit(
            train_anom[
                features
            ],
            y_train
        )


        pred = (
            model.predict_proba(
                valid_anom[
                    features
                ]
            )[:, 1]
        )


        pr_auc = (
            average_precision_score(
                y_valid,
                pred
            )
        )


        roc_auc = (
            roc_auc_score(
                y_valid,
                pred
            )
        )


        fold_results.append({

            "Valid_Session":
                valid_session,

            "Feature_Set":
                feature_name,

            "Valid_N":
                len(valid),

            "Defects":
                int(
                    y_valid.sum()
                ),

            "Reference_N":
                len(reference),

            "Baseline_PR":
                baseline,

            "PR_AUC":
                pr_auc,

            "PR_Ratio":
                pr_auc / baseline,

            "ROC_AUC":
                roc_auc
        })


        temp = pd.DataFrame({

            "TimeStamp":
                valid[
                    "TimeStamp"
                ]
                .values,

            "session_id":
                valid_session,

            "y_true":
                y_valid.values,

            "y_prob":
                pred,

            "Feature_Set":
                feature_name
        })


        oof_predictions.append(
            temp
        )


# ============================================================
# 15. 결과 합치기
# ============================================================

fold_df = (
    pd.DataFrame(
        fold_results
    )
)


oof_df = (
    pd.concat(
        oof_predictions,
        ignore_index=True
    )
)


# ============================================================
# 16. FOLD RESULTS
# ============================================================

print(
    "\n\n"
    + "=" * 100
)

print(
    "ANOMALY ABLATION - FOLD RESULTS"
)

print(
    "=" * 100
)


print(
    fold_df[
        [
            "Valid_Session",
            "Feature_Set",
            "Defects",
            "Baseline_PR",
            "PR_AUC",
            "PR_Ratio",
            "ROC_AUC"
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 17. Fold Summary
# ============================================================

fold_summary = (
    fold_df
    .groupby(
        "Feature_Set"
    )
    .agg(

        PR_Mean=(
            "PR_AUC",
            "mean"
        ),

        PR_Median=(
            "PR_AUC",
            "median"
        ),

        PR_Min=(
            "PR_AUC",
            "min"
        ),

        PR_Max=(
            "PR_AUC",
            "max"
        ),

        PR_Std=(
            "PR_AUC",
            "std"
        ),

        Ratio_Mean=(
            "PR_Ratio",
            "mean"
        ),

        ROC_Mean=(
            "ROC_AUC",
            "mean"
        )
    )
    .reset_index()
)


# ============================================================
# 18. OOF Summary
# ============================================================

oof_summary_rows = []


for feature_name, group in (
    oof_df
    .groupby(
        "Feature_Set"
    )
):

    baseline = (
        group[
            "y_true"
        ]
        .mean()
    )


    pr_auc = (
        average_precision_score(
            group[
                "y_true"
            ],
            group[
                "y_prob"
            ]
        )
    )


    roc_auc = (
        roc_auc_score(
            group[
                "y_true"
            ],
            group[
                "y_prob"
            ]
        )
    )


    # 몇 개 Fold에서 baseline 초과?
    fold_part = (
        fold_df[
            fold_df[
                "Feature_Set"
            ]
            ==
            feature_name
        ]
    )


    wins = (
        fold_part[
            "PR_AUC"
        ]
        >
        fold_part[
            "Baseline_PR"
        ]
    ).sum()


    oof_summary_rows.append({

        "Feature_Set":
            feature_name,

        "N":
            len(group),

        "Defects":
            int(
                group[
                    "y_true"
                ]
                .sum()
            ),

        "Baseline_PR":
            baseline,

        "OOF_PR_AUC":
            pr_auc,

        "PR_Ratio":
            pr_auc / baseline,

        "OOF_ROC_AUC":
            roc_auc,

        "Fold_Wins":
            int(wins)
    })


oof_summary = (
    pd.DataFrame(
        oof_summary_rows
    )
)


final_summary = (
    fold_summary
    .merge(
        oof_summary,
        on="Feature_Set",
        how="left"
    )
    .sort_values(
        [
            "OOF_PR_AUC",
            "PR_Median"
        ],
        ascending=False
    )
)


print(
    "\n\n"
    + "=" * 100
)

print(
    "ANOMALY ABLATION - FINAL SUMMARY"
)

print(
    "=" * 100
)


print(
    final_summary[
        [
            "Feature_Set",
            "OOF_PR_AUC",
            "PR_Ratio",
            "PR_Median",
            "PR_Mean",
            "PR_Min",
            "PR_Max",
            "Fold_Wins",
            "OOF_ROC_AUC"
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 19. 최종 Feature Set 선택
#
# OOF PR-AUC 최대
# ============================================================

best_feature_set = (
    final_summary
    .iloc[0][
        "Feature_Set"
    ]
)


print(
    "\n"
    + "=" * 100
)

print(
    "SELECTED FEATURE SET"
)

print(
    "=" * 100
)

print(
    best_feature_set
)


# ============================================================
# 20. 최종 OOF Prediction
# ============================================================

best_oof = (
    oof_df[
        oof_df[
            "Feature_Set"
        ]
        ==
        best_feature_set
    ]
    .copy()
    .sort_values(
        "TimeStamp"
    )
    .reset_index(drop=True)
)


print(
    "\nOOF N:",
    len(best_oof)
)

print(
    "OOF Defects:",
    int(
        best_oof[
            "y_true"
        ]
        .sum()
    )
)

print(
    "OOF PR-AUC:",
    average_precision_score(
        best_oof[
            "y_true"
        ],
        best_oof[
            "y_prob"
        ]
    )
)


# ============================================================
# 21. Threshold 전체 탐색
#
# 고정 0.5가 아니라
# 실제 OOF 확률분포 기준
# ============================================================

thresholds = (
    np.unique(
        best_oof[
            "y_prob"
        ]
        .values
    )
)


threshold_rows = []


for threshold in thresholds:

    pred_class = (
        best_oof[
            "y_prob"
        ]
        >=
        threshold
    ).astype(int)


    tn, fp, fn, tp = (
        confusion_matrix(
            best_oof[
                "y_true"
            ],
            pred_class,
            labels=[
                0,
                1
            ]
        )
        .ravel()
    )


    precision = (
        precision_score(
            best_oof[
                "y_true"
            ],
            pred_class,
            zero_division=0
        )
    )


    recall = (
        recall_score(
            best_oof[
                "y_true"
            ],
            pred_class,
            zero_division=0
        )
    )


    f1 = (
        f1_score(
            best_oof[
                "y_true"
            ],
            pred_class,
            zero_division=0
        )
    )


    inspection_rate = (
        pred_class.mean()
    )


    threshold_rows.append({

        "Threshold":
            threshold,

        "Precision":
            precision,

        "Recall":
            recall,

        "F1":
            f1,

        "TP":
            tp,

        "FP":
            fp,

        "FN":
            fn,

        "TN":
            tn,

        "Inspection_Rate":
            inspection_rate,

        "Inspection_N":
            int(
                pred_class.sum()
            )
    })


threshold_df = (
    pd.DataFrame(
        threshold_rows
    )
)


# ============================================================
# 22. 대표 Threshold
# ============================================================

# ------------------------------------------------------------
# Recall 목표별:
# 해당 Recall 이상을 만족하면서
# 검사율이 최소인 threshold
# ------------------------------------------------------------

policy_rows = []


for target_recall in [
    1.00,
    0.95,
    0.90,
    0.80,
    0.70,
    0.60
]:

    candidates_policy = (
        threshold_df[
            threshold_df[
                "Recall"
            ]
            >=
            target_recall
        ]
        .copy()
    )


    if len(candidates_policy) == 0:
        continue


    best_policy = (
        candidates_policy
        .sort_values(
            [
                "Inspection_Rate",
                "Precision",
                "Threshold"
            ],
            ascending=[
                True,
                False,
                False
            ]
        )
        .iloc[0]
    )


    row = (
        best_policy
        .to_dict()
    )

    row[
        "Target_Recall"
    ] = target_recall

    policy_rows.append(
        row
    )


policy_df = (
    pd.DataFrame(
        policy_rows
    )
)


# ============================================================
# 23. F1 최대점
# ============================================================

best_f1 = (
    threshold_df
    .sort_values(
        [
            "F1",
            "Recall"
        ],
        ascending=False
    )
    .iloc[0]
)


print(
    "\n\n"
    + "=" * 100
)

print(
    "RECALL TARGET POLICIES"
)

print(
    "=" * 100
)


if len(policy_df) > 0:

    print(
        policy_df[
            [
                "Target_Recall",
                "Threshold",
                "Precision",
                "Recall",
                "F1",
                "TP",
                "FP",
                "FN",
                "TN",
                "Inspection_N",
                "Inspection_Rate"
            ]
        ]
        .to_string(
            index=False
        )
    )


print(
    "\n\n"
    + "=" * 100
)

print(
    "BEST F1 OPERATING POINT"
)

print(
    "=" * 100
)


print(
    best_f1[
        [
            "Threshold",
            "Precision",
            "Recall",
            "F1",
            "TP",
            "FP",
            "FN",
            "TN",
            "Inspection_N",
            "Inspection_Rate"
        ]
    ]
    .to_string()
)


# ============================================================
# 24. 보기 쉽게 검사율 %
# ============================================================

print(
    "\n\n"
    + "=" * 100
)

print(
    "POLICY SUMMARY (%)"
)

print(
    "=" * 100
)


if len(policy_df) > 0:

    pretty_policy = (
        policy_df[
            [
                "Target_Recall",
                "Threshold",
                "Precision",
                "Recall",
                "F1",
                "TP",
                "FP",
                "FN",
                "Inspection_N",
                "Inspection_Rate"
            ]
        ]
        .copy()
    )


    pretty_policy[
        "Precision"
    ] *= 100

    pretty_policy[
        "Recall"
    ] *= 100

    pretty_policy[
        "Inspection_Rate"
    ] *= 100


    pretty_policy = (
        pretty_policy.rename(
            columns={
                "Precision":
                    "Precision_%",
                "Recall":
                    "Recall_%",
                "Inspection_Rate":
                    "Inspection_%"
            }
        )
    )


    print(
        pretty_policy
        .to_string(
            index=False
        )
    )

In [ ]:
# ============================================================
# RG3 FINAL MODEL TRAINING
#
# Final architecture
# - Local Normal Reference
# - PCA Reconstruction Error
# - KNN Distance
# - ExtraTrees Classifier
#
# Output
# - rg3_final_model.joblib
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import average_precision_score


# ============================================================
# 1. PATH
# ============================================================

BASE_DIR = (
    r"../../.."
    r"\1. 사출성형기 AI 데이터셋"
    r"\1. 사출성형기 AI 데이터셋"
)

OUTPUT_DIR = (
    r"../../.."
    r"\rg3_final"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


LABELED_PATH = (
    BASE_DIR
    + r"\moldset_labeled.csv"
)

UNLABELED_PATH = (
    BASE_DIR
    + r"\unlabeled_data.csv"
)


# ============================================================
# 2. LOAD
# ============================================================

labeled = pd.read_csv(
    LABELED_PATH
)

unlabeled = pd.read_csv(
    UNLABELED_PATH
)


TARGET_PRODUCT = (
    "RG3 MOLD'G W/SHLD, RH"
)


lab = (
    labeled[
        labeled["PART_NAME"]
        ==
        TARGET_PRODUCT
    ]
    .copy()
)

unlab = (
    unlabeled[
        unlabeled["PART_NAME"]
        ==
        TARGET_PRODUCT
    ]
    .copy()
)


lab["TimeStamp"] = pd.to_datetime(
    lab["TimeStamp"]
)

unlab["TimeStamp"] = pd.to_datetime(
    unlab["TimeStamp"]
)


lab = (
    lab
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)

unlab = (
    unlab
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


print("=" * 100)
print("DATA")
print("=" * 100)

print(
    "Labeled:",
    len(lab)
)

print(
    "Defects:",
    int(
        lab["PassOrFail"]
        .sum()
    )
)

print(
    "Unlabeled:",
    len(unlab)
)


# ============================================================
# 3. SENSOR
# ============================================================

sensor_cols = [
    'Injection_Time',
    'Filling_Time',
    'Plasticizing_Time',
    'Cycle_Time',
    'Clamp_Close_Time',
    'Cushion_Position',
    'Plasticizing_Position',
    'Clamp_Open_Position',
    'Max_Injection_Speed',
    'Max_Screw_RPM',
    'Average_Screw_RPM',
    'Max_Injection_Pressure',
    'Max_Switch_Over_Pressure',
    'Max_Back_Pressure',
    'Average_Back_Pressure',
    'Barrel_Temperature_1',
    'Barrel_Temperature_2',
    'Barrel_Temperature_3',
    'Barrel_Temperature_4',
    'Barrel_Temperature_5',
    'Barrel_Temperature_6',
    'Hopper_Temperature',
    'Mold_Temperature_3',
    'Mold_Temperature_4'
]


# 완전 상수 제거
raw_features = [
    col
    for col in sensor_cols
    if lab[col].nunique(
        dropna=False
    ) > 1
]


print("\nUsable sensors:")
print(
    len(raw_features)
)

print(
    raw_features
)


# ============================================================
# 4. Unlabeled Leakage 제거
# ============================================================

labeled_times = set(
    lab["TimeStamp"]
)


unlab_clean = (
    unlab[
        ~unlab["TimeStamp"]
        .isin(
            labeled_times
        )
    ]
    .copy()
)


print(
    "\nRemoved overlap:",
    len(unlab)
    -
    len(unlab_clean)
)


# ============================================================
# 5. 동일 Timestamp 중복 제거
# ============================================================

unlab_clean = (
    unlab_clean
    .groupby(
        "TimeStamp",
        as_index=False
    )[raw_features]
    .mean()
    .sort_values(
        "TimeStamp"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 6. 최종 학습 시점보다 과거 Unlabeled만
# ============================================================

deployment_cutoff = (
    lab["TimeStamp"]
    .max()
)


unlab_past = (
    unlab_clean[
        unlab_clean[
            "TimeStamp"
        ]
        <
        deployment_cutoff
    ]
    .copy()
)


print(
    "Past unlabeled:",
    len(unlab_past)
)


# ============================================================
# 7. Labeled Normal
# ============================================================

normal = (
    lab[
        lab["PassOrFail"]
        ==
        0
    ]
    .copy()
)


print(
    "Labeled normal:",
    len(normal)
)


# ============================================================
# 8. Local Reference Selector
#
# scaler는 labeled 정상에 fit
# ============================================================

selector_scaler = (
    StandardScaler()
)


X_normal = (
    selector_scaler
    .fit_transform(
        normal[
            raw_features
        ]
    )
)


X_unlab = (
    selector_scaler
    .transform(
        unlab_past[
            raw_features
        ]
    )
)


# ============================================================
# 9. Labeled normal 내부 KNN 거리
# ============================================================

K_SELECT = 5


nn_train = (
    NearestNeighbors(
        n_neighbors=
            K_SELECT + 1,
        metric="euclidean"
    )
)


nn_train.fit(
    X_normal
)


normal_distances, _ = (
    nn_train
    .kneighbors(
        X_normal
    )
)


# 자기 자신 제외한 5번째 이웃
normal_k_distance = (
    normal_distances[
        :,
        K_SELECT
    ]
)


q95 = (
    np.quantile(
        normal_k_distance,
        0.95
    )
)


print(
    "\nFinal Q95:",
    q95
)


# ============================================================
# 10. Unlabeled -> Normal 거리
# ============================================================

selector_knn = (
    NearestNeighbors(
        n_neighbors=K_SELECT,
        metric="euclidean"
    )
)


selector_knn.fit(
    X_normal
)


unlab_distances, _ = (
    selector_knn
    .kneighbors(
        X_unlab
    )
)


unlab_k_distance = (
    unlab_distances[
        :,
        -1
    ]
)


reference_mask = (
    unlab_k_distance
    <=
    q95
)


reference = (
    unlab_past
    .loc[
        reference_mask
    ]
    .copy()
)


print(
    "Final Local Reference:",
    len(reference)
)

print(
    "Reference Ratio:",
    len(reference)
    /
    len(unlab_past)
)


print(
    "Reference Time:",
    reference[
        "TimeStamp"
    ].min(),
    "~",
    reference[
        "TimeStamp"
    ].max()
)


# ============================================================
# 11. Anomaly Scaler
#
# Local reference에 fit
# ============================================================

anomaly_scaler = (
    StandardScaler()
)


X_reference = (
    anomaly_scaler
    .fit_transform(
        reference[
            raw_features
        ]
    )
)


X_labeled = (
    anomaly_scaler
    .transform(
        lab[
            raw_features
        ]
    )
)


# ============================================================
# 12. PCA
# ============================================================

pca = (
    PCA(
        n_components=0.95,
        random_state=42
    )
)


pca.fit(
    X_reference
)


print(
    "\nPCA Components:",
    pca.n_components_
)

print(
    "Explained variance:",
    pca.explained_variance_ratio_
    .sum()
)


# ============================================================
# 13. PCA Reconstruction Error
# ============================================================

X_pca = (
    pca.transform(
        X_labeled
    )
)


X_reconstructed = (
    pca.inverse_transform(
        X_pca
    )
)


pca_error = (
    np.mean(
        (
            X_labeled
            -
            X_reconstructed
        ) ** 2,
        axis=1
    )
)


# ============================================================
# 14. KNN Anomaly
# ============================================================

K_ANOMALY = 10


knn_anomaly = (
    NearestNeighbors(
        n_neighbors=
            K_ANOMALY,
        metric="euclidean"
    )
)


knn_anomaly.fit(
    X_reference
)


knn_distances, _ = (
    knn_anomaly
    .kneighbors(
        X_labeled
    )
)


knn_score = (
    knn_distances
    .mean(
        axis=1
    )
)


# ============================================================
# 15. Final Training Features
# ============================================================

X_final = pd.DataFrame({

    "anomaly_pca":
        pca_error,

    "anomaly_knn":
        knn_score
})


y_final = (
    lab[
        "PassOrFail"
    ]
    .astype(int)
)


print(
    "\n"
    + "=" * 100
)

print(
    "FINAL TRAINING FEATURES"
)

print(
    "=" * 100
)


print(
    X_final.describe()
)


# ============================================================
# 16. ExtraTrees Final
# ============================================================

final_model = (
    ExtraTreesClassifier(
        n_estimators=500,
        max_depth=6,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
)


final_model.fit(
    X_final,
    y_final
)


# ============================================================
# 17. Training score
#
# 참고용일 뿐 성능보고용 X
# ============================================================

train_prob = (
    final_model
    .predict_proba(
        X_final
    )[:, 1]
)


train_pr = (
    average_precision_score(
        y_final,
        train_prob
    )
)


print(
    "\nTraining PR-AUC "
    "(참고용 / 성능보고 금지):",
    train_pr
)


# ============================================================
# 18. Feature Importance
# ============================================================

importance = pd.DataFrame({

    "Feature": [
        "anomaly_pca",
        "anomaly_knn"
    ],

    "Importance":
        final_model
        .feature_importances_
})


importance = (
    importance
    .sort_values(
        "Importance",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "FINAL FEATURE IMPORTANCE"
)

print(
    "=" * 100
)


print(
    importance
    .to_string(
        index=False
    )
)


# ============================================================
# 19. 모델 Bundle 저장
# ============================================================

model_bundle = {

    # Metadata
    "model_name":
        "RG3_PCA_KNN_ExtraTrees",

    "target_product":
        TARGET_PRODUCT,

    "raw_features":
        raw_features,

    # Reference selection
    "selector_scaler":
        selector_scaler,

    "reference_q95":
        q95,

    "reference_n":
        len(reference),

    # Anomaly model
    "anomaly_scaler":
        anomaly_scaler,

    "pca":
        pca,

    "knn_anomaly":
        knn_anomaly,

    # Classifier
    "classifier":
        final_model,

    # Feature
    "final_features": [
        "anomaly_pca",
        "anomaly_knn"
    ],

    # Evaluation result
    "oof_pr_auc":
        0.16198735564473354,

    "oof_baseline_pr":
        0.04321,

    "oof_pr_ratio":
        3.748850,

    # 운영 후보
    # 주의: OOF 기반 참고 threshold
    "oof_policy_threshold":
        0.410312,

    "oof_policy_recall":
        0.619048,

    "oof_policy_precision":
        0.097015,

    "oof_policy_inspection_rate":
        0.275720
}


MODEL_PATH = (
    OUTPUT_DIR
    + r"\rg3_final_model.joblib"
)


joblib.dump(
    model_bundle,
    MODEL_PATH
)


print(
    "\n"
    + "=" * 100
)

print(
    "MODEL SAVED"
)

print(
    "=" * 100
)

print(
    MODEL_PATH
)


# ============================================================
# 20. Feature Importance 저장
# ============================================================

importance.to_csv(
    OUTPUT_DIR
    + r"\rg3_final_feature_importance.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 21. Reference 저장
# ============================================================

reference.to_csv(
    OUTPUT_DIR
    + r"\rg3_local_normal_reference.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 22. Labeled anomaly score 저장
# ============================================================

score_df = pd.DataFrame({

    "TimeStamp":
        lab[
            "TimeStamp"
        ],

    "PassOrFail":
        y_final,

    "anomaly_pca":
        pca_error,

    "anomaly_knn":
        knn_score,

    "Train_Risk_Probability":
        train_prob
})


score_df.to_csv(
    OUTPUT_DIR
    + r"\rg3_labeled_anomaly_scores.csv",
    index=False,
    encoding="utf-8-sig"
)


print(
    "\nFiles saved:"
)

print(
    "- rg3_final_model.joblib"
)

print(
    "- rg3_final_feature_importance.csv"
)

print(
    "- rg3_local_normal_reference.csv"
)

print(
    "- rg3_labeled_anomaly_scores.csv"
)

In [ ]:
import joblib
import numpy as np
import pandas as pd


MODEL_PATH = (
    r"../../.."
    r"\rg3_final"
    r"\rg3_final_model.joblib"
)


bundle = joblib.load(
    MODEL_PATH
)


def predict_rg3_defect(
    new_data
):

    """
    new_data:
        pandas DataFrame
        RG3 RH 공정센서 데이터

    return:
        Risk_Probability
        PCA_Error
        KNN_Distance
        Inspection_Flag
    """

    features = (
        bundle[
            "raw_features"
        ]
    )


    # --------------------------------------------
    # 컬럼 검사
    # --------------------------------------------

    missing_cols = [
        c
        for c in features
        if c not in new_data.columns
    ]


    if missing_cols:

        raise ValueError(
            f"Missing columns: "
            f"{missing_cols}"
        )


    X = (
        new_data[
            features
        ]
        .copy()
    )


    # --------------------------------------------
    # Reference 기준 scaling
    # --------------------------------------------

    X_scaled = (
        bundle[
            "anomaly_scaler"
        ]
        .transform(
            X
        )
    )


    # --------------------------------------------
    # PCA Reconstruction Error
    # --------------------------------------------

    pca = (
        bundle[
            "pca"
        ]
    )


    z = (
        pca.transform(
            X_scaled
        )
    )


    recon = (
        pca.inverse_transform(
            z
        )
    )


    pca_error = (
        np.mean(
            (
                X_scaled
                -
                recon
            ) ** 2,
            axis=1
        )
    )


    # --------------------------------------------
    # KNN Distance
    # --------------------------------------------

    knn = (
        bundle[
            "knn_anomaly"
        ]
    )


    distances, _ = (
        knn.kneighbors(
            X_scaled
        )
    )


    knn_score = (
        distances
        .mean(axis=1)
    )


    # --------------------------------------------
    # Classification
    # --------------------------------------------

    X_model = pd.DataFrame({

        "anomaly_pca":
            pca_error,

        "anomaly_knn":
            knn_score
    })


    risk_prob = (
        bundle[
            "classifier"
        ]
        .predict_proba(
            X_model
        )[:, 1]
    )


    # --------------------------------------------
    # OOF 운영 후보 threshold
    # --------------------------------------------

    threshold = (
        bundle[
            "oof_policy_threshold"
        ]
    )


    result = pd.DataFrame({

        "PCA_Error":
            pca_error,

        "KNN_Distance":
            knn_score,

        "Risk_Probability":
            risk_prob,

        "Inspection_Flag":
            (
                risk_prob
                >=
                threshold
            ).astype(int)
    })


    return result

In [ ]:
sample = lab.tail(10)

result = predict_rg3_defect(
    sample
)

print(result)

이 코드가 뽑는 것
- OOF 기준 TP / FP / FN / TN
- 그룹별 PCA Error / KNN Distance / Risk 차이
- Session별 오탐·미탐
- 대표 TP / FP / FN 사례
- 각 Cycle에서 PCA 이상에 가장 크게 기여한 센서 TOP 3
- 정상 Reference와 가장 많이 달랐던 센서 TOP 3
- CSV 저장

In [ ]:
# ============================================================
# RG3 FINAL ERROR ANALYSIS
#
# Final model:
#   PCA Reconstruction Error
#   + KNN Distance
#   -> ExtraTrees
#
# OOF Walk-forward 기준 오류 분석
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# ============================================================
# 1. 설정
# ============================================================

OUTPUT_DIR = (
    r"../../.."
    r"\rg3_final"
    r"\error_analysis"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# OOF에서 선택한 운영 예시 threshold
THRESHOLD = 0.410312


# ============================================================
# 2. Labeled 준비
# ============================================================

data = (
    lab
    .copy()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)

data["TimeStamp"] = pd.to_datetime(
    data["TimeStamp"]
)


# ------------------------------------------------------------
# Session 생성
# ------------------------------------------------------------

data["gap_min"] = (
    data["TimeStamp"]
    .diff()
    .dt.total_seconds()
    / 60
)

data["session_id"] = (
    (data["gap_min"] > 5)
    .cumsum()
)


# ============================================================
# 3. 사용 Sensor
# ============================================================

raw_features = [
    c
    for c in sensor_cols
    if data[c].nunique(
        dropna=False
    ) > 1
]


print("=" * 100)
print("FEATURES")
print("=" * 100)

print(
    "Usable:",
    len(raw_features)
)

print(
    "Removed:",
    [
        c
        for c in sensor_cols
        if c not in raw_features
    ]
)


# ============================================================
# 4. Unlabeled Leakage 제거
# ============================================================

unlab2 = (
    unlab
    .copy()
)

unlab2["TimeStamp"] = pd.to_datetime(
    unlab2["TimeStamp"]
)


all_labeled_times = set(
    data["TimeStamp"]
)


unlab_clean = (
    unlab2[
        ~unlab2["TimeStamp"]
        .isin(
            all_labeled_times
        )
    ]
    .copy()
)


# 동일 Timestamp 중복 → 평균
unlab_clean = (
    unlab_clean
    .groupby(
        "TimeStamp",
        as_index=False
    )[raw_features]
    .mean()
    .sort_values(
        "TimeStamp"
    )
    .reset_index(drop=True)
)


print(
    "Clean unlabeled:",
    len(unlab_clean)
)


# ============================================================
# 5. OOF 저장
# ============================================================

oof_rows = []


# ============================================================
# 6. Walk-forward
# ============================================================

for valid_session in range(2, 7):

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"SESSION {valid_session}"
    )

    print(
        "=" * 100
    )


    # --------------------------------------------------------
    # Train / Validation
    # --------------------------------------------------------

    train = (
        data[
            data["session_id"]
            <
            valid_session
        ]
        .copy()
    )

    valid = (
        data[
            data["session_id"]
            ==
            valid_session
        ]
        .copy()
    )


    train_normal = (
        train[
            train["PassOrFail"] == 0
        ]
        .copy()
    )


    valid_start = (
        valid["TimeStamp"]
        .min()
    )


    # ========================================================
    # 7. Validation 이전 Unlabeled
    # ========================================================

    candidates = (
        unlab_clean[
            unlab_clean["TimeStamp"]
            <
            valid_start
        ]
        .copy()
    )


    # ========================================================
    # 8. Local Reference 선택
    # ========================================================

    selector_scaler = (
        StandardScaler()
    )


    X_normal = (
        selector_scaler
        .fit_transform(
            train_normal[
                raw_features
            ]
        )
    )


    X_candidates = (
        selector_scaler
        .transform(
            candidates[
                raw_features
            ]
        )
    )


    K_SELECT = min(
        5,
        len(train_normal) - 1
    )


    # --------------------------------------------------------
    # Normal 내부 거리
    # --------------------------------------------------------

    nn_normal = (
        NearestNeighbors(
            n_neighbors=
                K_SELECT + 1,
            metric="euclidean"
        )
    )


    nn_normal.fit(
        X_normal
    )


    normal_dist, _ = (
        nn_normal
        .kneighbors(
            X_normal
        )
    )


    q95 = (
        np.quantile(
            normal_dist[
                :,
                K_SELECT
            ],
            0.95
        )
    )


    # --------------------------------------------------------
    # Candidate → Normal
    # --------------------------------------------------------

    nn_select = (
        NearestNeighbors(
            n_neighbors=K_SELECT,
            metric="euclidean"
        )
    )


    nn_select.fit(
        X_normal
    )


    candidate_dist, _ = (
        nn_select
        .kneighbors(
            X_candidates
        )
    )


    candidate_kdist = (
        candidate_dist[:, -1]
    )


    reference_mask = (
        candidate_kdist
        <=
        q95
    )


    reference = (
        candidates
        .loc[
            reference_mask
        ]
        .copy()
    )


    print(
        "Train:",
        len(train),
        "| Defects:",
        int(
            train[
                "PassOrFail"
            ]
            .sum()
        )
    )

    print(
        "Valid:",
        len(valid),
        "| Defects:",
        int(
            valid[
                "PassOrFail"
            ]
            .sum()
        )
    )

    print(
        "Reference:",
        len(reference)
    )


    # ========================================================
    # 9. Anomaly Scaler
    # ========================================================

    anomaly_scaler = (
        StandardScaler()
    )


    X_ref = (
        anomaly_scaler
        .fit_transform(
            reference[
                raw_features
            ]
        )
    )


    X_train = (
        anomaly_scaler
        .transform(
            train[
                raw_features
            ]
        )
    )


    X_valid = (
        anomaly_scaler
        .transform(
            valid[
                raw_features
            ]
        )
    )


    # ========================================================
    # 10. PCA
    # ========================================================

    pca = (
        PCA(
            n_components=0.95,
            random_state=42
        )
    )


    pca.fit(
        X_ref
    )


    # --------------------------------------------------------
    # Train PCA Error
    # --------------------------------------------------------

    train_z = (
        pca.transform(
            X_train
        )
    )

    train_recon = (
        pca.inverse_transform(
            train_z
        )
    )


    train_pca_feature_error = (
        (
            X_train
            -
            train_recon
        ) ** 2
    )


    train_pca_error = (
        train_pca_feature_error
        .mean(axis=1)
    )


    # --------------------------------------------------------
    # Valid PCA Error
    # --------------------------------------------------------

    valid_z = (
        pca.transform(
            X_valid
        )
    )


    valid_recon = (
        pca.inverse_transform(
            valid_z
        )
    )


    valid_pca_feature_error = (
        (
            X_valid
            -
            valid_recon
        ) ** 2
    )


    valid_pca_error = (
        valid_pca_feature_error
        .mean(axis=1)
    )


    # ========================================================
    # 11. KNN Distance
    # ========================================================

    K_ANOMALY = min(
        10,
        len(reference)
    )


    knn = (
        NearestNeighbors(
            n_neighbors=
                K_ANOMALY,
            metric="euclidean"
        )
    )


    knn.fit(
        X_ref
    )


    train_knn_dist, train_knn_idx = (
        knn.kneighbors(
            X_train
        )
    )


    valid_knn_dist, valid_knn_idx = (
        knn.kneighbors(
            X_valid
        )
    )


    train_knn_score = (
        train_knn_dist
        .mean(axis=1)
    )


    valid_knn_score = (
        valid_knn_dist
        .mean(axis=1)
    )


    # ========================================================
    # 12. ExtraTrees
    # ========================================================

    train_anomaly = pd.DataFrame({

        "anomaly_pca":
            train_pca_error,

        "anomaly_knn":
            train_knn_score
    })


    valid_anomaly = pd.DataFrame({

        "anomaly_pca":
            valid_pca_error,

        "anomaly_knn":
            valid_knn_score
    })


    y_train = (
        train["PassOrFail"]
        .astype(int)
        .reset_index(drop=True)
    )


    model = (
        ExtraTreesClassifier(
            n_estimators=500,
            max_depth=6,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )


    model.fit(
        train_anomaly,
        y_train
    )


    valid_prob = (
        model
        .predict_proba(
            valid_anomaly
        )[:, 1]
    )


    # ========================================================
    # 13. 센서별 PCA 기여도
    # ========================================================

    pca_rank = (
        np.argsort(
            valid_pca_feature_error,
            axis=1
        )[
            :,
            ::-1
        ]
    )


    # ========================================================
    # 14. KNN 기준 가장 가까운 Reference와 차이
    #
    # 설명용 보조값
    # ========================================================

    nearest_reference_idx = (
        valid_knn_idx[:, 0]
    )


    nearest_reference = (
        X_ref[
            nearest_reference_idx
        ]
    )


    knn_feature_difference = (
        np.abs(
            X_valid
            -
            nearest_reference
        )
    )


    knn_rank = (
        np.argsort(
            knn_feature_difference,
            axis=1
        )[
            :,
            ::-1
        ]
    )


    # ========================================================
    # 15. Valid 결과 저장
    # ========================================================

    valid = (
        valid
        .reset_index(drop=True)
    )


    for i in range(
        len(valid)
    ):

        pca_top_idx = (
            pca_rank[i, :3]
        )

        knn_top_idx = (
            knn_rank[i, :3]
        )


        row = {

            "TimeStamp":
                valid.loc[
                    i,
                    "TimeStamp"
                ],

            "session_id":
                valid_session,

            "y_true":
                int(
                    valid.loc[
                        i,
                        "PassOrFail"
                    ]
                ),

            "Risk_Probability":
                valid_prob[i],

            "anomaly_pca":
                valid_pca_error[i],

            "anomaly_knn":
                valid_knn_score[i],

            "Reference_N":
                len(reference),

            "Reference_Q95":
                q95
        }


        # ----------------------------------------------------
        # PCA Top 3
        # ----------------------------------------------------

        for rank_no, idx in enumerate(
            pca_top_idx,
            start=1
        ):

            row[
                f"PCA_Top{rank_no}_Feature"
            ] = raw_features[idx]

            row[
                f"PCA_Top{rank_no}_Error"
            ] = (
                valid_pca_feature_error[
                    i,
                    idx
                ]
            )


        # ----------------------------------------------------
        # KNN Top 3
        # ----------------------------------------------------

        for rank_no, idx in enumerate(
            knn_top_idx,
            start=1
        ):

            row[
                f"KNN_Top{rank_no}_Feature"
            ] = raw_features[idx]

            row[
                f"KNN_Top{rank_no}_AbsDiff"
            ] = (
                knn_feature_difference[
                    i,
                    idx
                ]
            )


        # ----------------------------------------------------
        # 실제 Sensor 값도 보존
        # ----------------------------------------------------

        for col in raw_features:

            row[col] = (
                valid.loc[
                    i,
                    col
                ]
            )


        oof_rows.append(
            row
        )


# ============================================================
# 16. OOF DataFrame
# ============================================================

oof = (
    pd.DataFrame(
        oof_rows
    )
    .sort_values(
        "TimeStamp"
    )
    .reset_index(drop=True)
)


# ============================================================
# 17. Threshold 적용
# ============================================================

oof["y_pred"] = (
    oof[
        "Risk_Probability"
    ]
    >=
    THRESHOLD
).astype(int)


# ------------------------------------------------------------
# Error Type
# ------------------------------------------------------------

conditions = [

    (
        (oof["y_true"] == 1)
        &
        (oof["y_pred"] == 1)
    ),

    (
        (oof["y_true"] == 0)
        &
        (oof["y_pred"] == 1)
    ),

    (
        (oof["y_true"] == 1)
        &
        (oof["y_pred"] == 0)
    ),

    (
        (oof["y_true"] == 0)
        &
        (oof["y_pred"] == 0)
    )
]


choices = [
    "TP",
    "FP",
    "FN",
    "TN"
]


oof["Error_Type"] = np.select(
    conditions,
    choices,
    default="UNKNOWN"
)


# ============================================================
# 18. 전체 성능
# ============================================================

tn, fp, fn, tp = (
    confusion_matrix(
        oof["y_true"],
        oof["y_pred"],
        labels=[0, 1]
    )
    .ravel()
)


precision = (
    precision_score(
        oof["y_true"],
        oof["y_pred"],
        zero_division=0
    )
)


recall = (
    recall_score(
        oof["y_true"],
        oof["y_pred"],
        zero_division=0
    )
)


f1 = (
    f1_score(
        oof["y_true"],
        oof["y_pred"],
        zero_division=0
    )
)


pr_auc = (
    average_precision_score(
        oof["y_true"],
        oof["Risk_Probability"]
    )
)


print(
    "\n\n"
    + "=" * 100
)

print(
    "FINAL OOF POLICY PERFORMANCE"
)

print(
    "=" * 100
)


print(
    f"Threshold       : {THRESHOLD:.6f}"
)

print(
    f"PR-AUC         : {pr_auc:.6f}"
)

print(
    f"Precision      : {precision:.4f}"
)

print(
    f"Recall         : {recall:.4f}"
)

print(
    f"F1             : {f1:.4f}"
)

print(
    f"TP / FP        : {tp} / {fp}"
)

print(
    f"FN / TN        : {fn} / {tn}"
)

print(
    "Inspection N   :",
    int(
        oof["y_pred"].sum()
    )
)

print(
    "Inspection Rate:",
    f"{oof['y_pred'].mean()*100:.2f}%"
)


# ============================================================
# 19. ERROR TYPE COUNT
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "ERROR TYPE COUNT"
)

print(
    "=" * 100
)


print(
    oof[
        "Error_Type"
    ]
    .value_counts()
)


# ============================================================
# 20. TP / FP / FN / TN별 Anomaly 특성
# ============================================================

group_summary = (
    oof
    .groupby(
        "Error_Type"
    )
    .agg(

        N=(
            "y_true",
            "size"
        ),

        Risk_Mean=(
            "Risk_Probability",
            "mean"
        ),

        Risk_Median=(
            "Risk_Probability",
            "median"
        ),

        PCA_Mean=(
            "anomaly_pca",
            "mean"
        ),

        PCA_Median=(
            "anomaly_pca",
            "median"
        ),

        KNN_Mean=(
            "anomaly_knn",
            "mean"
        ),

        KNN_Median=(
            "anomaly_knn",
            "median"
        )
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "TP / FP / FN / TN ANOMALY SUMMARY"
)

print(
    "=" * 100
)


print(
    group_summary
    .to_string()
)


# ============================================================
# 21. Session별 오류
# ============================================================

session_error = (
    pd.crosstab(
        oof["session_id"],
        oof["Error_Type"]
    )
)


for col in [
    "TP",
    "FP",
    "FN",
    "TN"
]:

    if col not in session_error.columns:

        session_error[col] = 0


session_error = (
    session_error[
        [
            "TP",
            "FP",
            "FN",
            "TN"
        ]
    ]
)


session_error[
    "Inspection_N"
] = (
    session_error[
        "TP"
    ]
    +
    session_error[
        "FP"
    ]
)


session_error[
    "Actual_Defects"
] = (
    session_error[
        "TP"
    ]
    +
    session_error[
        "FN"
    ]
)


session_error[
    "Recall"
] = (
    session_error[
        "TP"
    ]
    /
    session_error[
        "Actual_Defects"
    ]
    .replace(
        0,
        np.nan
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "SESSION ERROR SUMMARY"
)

print(
    "=" * 100
)


print(
    session_error
    .to_string()
)


# ============================================================
# 22. 대표 TP
# ============================================================

tp_cases = (
    oof[
        oof[
            "Error_Type"
        ]
        ==
        "TP"
    ]
    .sort_values(
        "Risk_Probability",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "TOP TRUE POSITIVE CASES"
)

print(
    "=" * 100
)


print(
    tp_cases[
        [
            "TimeStamp",
            "session_id",
            "Risk_Probability",
            "anomaly_pca",
            "anomaly_knn",
            "PCA_Top1_Feature",
            "PCA_Top2_Feature",
            "PCA_Top3_Feature",
            "KNN_Top1_Feature",
            "KNN_Top2_Feature",
            "KNN_Top3_Feature"
        ]
    ]
    .head(10)
    .to_string(
        index=False
    )
)


# ============================================================
# 23. 대표 FP
# ============================================================

fp_cases = (
    oof[
        oof[
            "Error_Type"
        ]
        ==
        "FP"
    ]
    .sort_values(
        "Risk_Probability",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "TOP FALSE POSITIVE CASES"
)

print(
    "=" * 100
)


print(
    fp_cases[
        [
            "TimeStamp",
            "session_id",
            "Risk_Probability",
            "anomaly_pca",
            "anomaly_knn",
            "PCA_Top1_Feature",
            "PCA_Top2_Feature",
            "PCA_Top3_Feature",
            "KNN_Top1_Feature",
            "KNN_Top2_Feature",
            "KNN_Top3_Feature"
        ]
    ]
    .head(10)
    .to_string(
        index=False
    )
)


# ============================================================
# 24. 모든 FN
# ============================================================

fn_cases = (
    oof[
        oof[
            "Error_Type"
        ]
        ==
        "FN"
    ]
    .sort_values(
        "Risk_Probability",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "FALSE NEGATIVE CASES"
)

print(
    "=" * 100
)


print(
    fn_cases[
        [
            "TimeStamp",
            "session_id",
            "Risk_Probability",
            "anomaly_pca",
            "anomaly_knn",
            "PCA_Top1_Feature",
            "PCA_Top2_Feature",
            "PCA_Top3_Feature",
            "KNN_Top1_Feature",
            "KNN_Top2_Feature",
            "KNN_Top3_Feature"
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 25. 센서 출현 빈도
#
# 어떤 센서가 이상 원인 TOP1으로
# 자주 등장하는지 확인
# ============================================================

def top_feature_frequency(
    df,
    prefix
):

    cols = [
        f"{prefix}_Top1_Feature",
        f"{prefix}_Top2_Feature",
        f"{prefix}_Top3_Feature"
    ]

    temp = (
        df[
            cols
        ]
        .stack()
        .value_counts()
    )

    return temp


print(
    "\n"
    + "=" * 100
)

print(
    "TP PCA CONTRIBUTION FREQUENCY"
)

print(
    "=" * 100
)

print(
    top_feature_frequency(
        tp_cases,
        "PCA"
    )
    .head(15)
)


print(
    "\n"
    + "=" * 100
)

print(
    "FP PCA CONTRIBUTION FREQUENCY"
)

print(
    "=" * 100
)

print(
    top_feature_frequency(
        fp_cases,
        "PCA"
    )
    .head(15)
)


print(
    "\n"
    + "=" * 100
)

print(
    "FN PCA CONTRIBUTION FREQUENCY"
)

print(
    "=" * 100
)

print(
    top_feature_frequency(
        fn_cases,
        "PCA"
    )
    .head(15)
)


print(
    "\n"
    + "=" * 100
)

print(
    "TP KNN DEVIATION FREQUENCY"
)

print(
    "=" * 100
)

print(
    top_feature_frequency(
        tp_cases,
        "KNN"
    )
    .head(15)
)


print(
    "\n"
    + "=" * 100
)

print(
    "FP KNN DEVIATION FREQUENCY"
)

print(
    "=" * 100
)

print(
    top_feature_frequency(
        fp_cases,
        "KNN"
    )
    .head(15)
)


print(
    "\n"
    + "=" * 100
)

print(
    "FN KNN DEVIATION FREQUENCY"
)

print(
    "=" * 100
)

print(
    top_feature_frequency(
        fn_cases,
        "KNN"
    )
    .head(15)
)


# ============================================================
# 26. 저장
# ============================================================

oof.to_csv(
    OUTPUT_DIR
    + r"\rg3_oof_error_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)


group_summary.to_csv(
    OUTPUT_DIR
    + r"\rg3_error_type_summary.csv",
    encoding="utf-8-sig"
)


session_error.to_csv(
    OUTPUT_DIR
    + r"\rg3_session_error_summary.csv",
    encoding="utf-8-sig"
)


tp_cases.to_csv(
    OUTPUT_DIR
    + r"\rg3_true_positive_cases.csv",
    index=False,
    encoding="utf-8-sig"
)


fp_cases.to_csv(
    OUTPUT_DIR
    + r"\rg3_false_positive_cases.csv",
    index=False,
    encoding="utf-8-sig"
)


fn_cases.to_csv(
    OUTPUT_DIR
    + r"\rg3_false_negative_cases.csv",
    index=False,
    encoding="utf-8-sig"
)


print(
    "\n"
    + "=" * 100
)

print(
    "FILES SAVED"
)

print(
    "=" * 100
)

print(
    OUTPUT_DIR
)

print(
    """
- rg3_oof_error_analysis.csv
- rg3_error_type_summary.csv
- rg3_session_error_summary.csv
- rg3_true_positive_cases.csv
- rg3_false_positive_cases.csv
- rg3_false_negative_cases.csv
"""
)

| 유형 | Risk 평균 | PCA 평균 | KNN 평균 |
|---|---:|---:|---:|
| TP | 0.481 | 0.0487 | 4.114 |
| FP | 0.473 | 0.0493 | 4.036 |
| FN | 0.280 | 0.0786 | 5.300 |
| TN | 0.307 | 0.1996 | 5.540 |

In [ ]:
# ============================================================
# RG3 FINAL ROBUSTNESS TEST
#
# 1) Q90 / Q95 / Q99 sensitivity
# 2) All / Recent 3D / Recent 1D reference
# 3) Top-K inspection policy
#
# 필요 변수:
#   lab
#   unlab
#   sensor_cols
#
# Canonical model:
#   Q95 + All Past Reference
#   PCA + KNN
#   ExtraTrees
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)


# ============================================================
# 1. 데이터 준비
# ============================================================

data = (
    lab
    .copy()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)

data["TimeStamp"] = pd.to_datetime(
    data["TimeStamp"]
)


# ------------------------------------------------------------
# Session
# ------------------------------------------------------------

data["gap_min"] = (
    data["TimeStamp"]
    .diff()
    .dt.total_seconds()
    / 60
)

data["session_id"] = (
    (data["gap_min"] > 5)
    .cumsum()
)


# ============================================================
# 2. 센서
# ============================================================

raw_features = [
    c
    for c in sensor_cols
    if data[c].nunique(dropna=False) > 1
]


print("=" * 100)
print("FEATURES")
print("=" * 100)

print("Usable:", len(raw_features))
print(
    "Removed:",
    [
        c
        for c in sensor_cols
        if c not in raw_features
    ]
)


# ============================================================
# 3. Unlabeled 정리
# ============================================================

unlab2 = (
    unlab
    .copy()
)

unlab2["TimeStamp"] = pd.to_datetime(
    unlab2["TimeStamp"]
)


# Labeled와 겹치는 Timestamp 제거
labeled_times = set(
    data["TimeStamp"]
)

unlab_clean = (
    unlab2[
        ~unlab2["TimeStamp"]
        .isin(labeled_times)
    ]
    .copy()
)


# 동일 Timestamp → 평균
unlab_clean = (
    unlab_clean
    .groupby(
        "TimeStamp",
        as_index=False
    )[raw_features]
    .mean()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


print(
    "Clean unlabeled:",
    len(unlab_clean)
)


# ============================================================
# 4. ExtraTrees
# ============================================================

def make_model():

    return ExtraTreesClassifier(
        n_estimators=500,
        max_depth=6,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )


# ============================================================
# 5. 단일 설정 평가 함수
# ============================================================

def evaluate_reference_strategy(
    quantile=0.95,
    history_days=None,
    label=""
):

    """
    quantile:
        0.90 / 0.95 / 0.99

    history_days:
        None = 모든 과거
        1    = 최근 1일
        3    = 최근 3일
    """

    fold_results = []
    oof_rows = []

    for valid_session in range(2, 7):

        # ====================================================
        # Train / Valid
        # ====================================================

        train = (
            data[
                data["session_id"]
                <
                valid_session
            ]
            .copy()
        )

        valid = (
            data[
                data["session_id"]
                ==
                valid_session
            ]
            .copy()
        )

        train_normal = (
            train[
                train["PassOrFail"] == 0
            ]
            .copy()
        )

        valid_start = (
            valid["TimeStamp"]
            .min()
        )


        # ====================================================
        # 과거 Unlabeled 후보
        # ====================================================

        candidates = (
            unlab_clean[
                unlab_clean["TimeStamp"]
                <
                valid_start
            ]
            .copy()
        )


        # ----------------------------------------------------
        # 최근 N일 제한
        # ----------------------------------------------------

        if history_days is not None:

            history_start = (
                valid_start
                -
                pd.Timedelta(
                    days=history_days
                )
            )

            candidates = (
                candidates[
                    candidates["TimeStamp"]
                    >=
                    history_start
                ]
                .copy()
            )


        # ====================================================
        # Train Normal 기준 Scaling
        # ====================================================

        selector_scaler = (
            StandardScaler()
        )

        X_normal = (
            selector_scaler
            .fit_transform(
                train_normal[
                    raw_features
                ]
            )
        )

        X_candidate = (
            selector_scaler
            .transform(
                candidates[
                    raw_features
                ]
            )
        )


        # ====================================================
        # Train normal 내부 KNN 거리
        # ====================================================

        k_select = min(
            5,
            len(train_normal) - 1
        )

        nn_train = (
            NearestNeighbors(
                n_neighbors=
                    k_select + 1,
                metric="euclidean"
            )
        )

        nn_train.fit(
            X_normal
        )

        train_dist, _ = (
            nn_train
            .kneighbors(
                X_normal
            )
        )

        threshold = (
            np.quantile(
                train_dist[
                    :,
                    k_select
                ],
                quantile
            )
        )


        # ====================================================
        # Candidate → Normal
        # ====================================================

        nn_select = (
            NearestNeighbors(
                n_neighbors=k_select,
                metric="euclidean"
            )
        )

        nn_select.fit(
            X_normal
        )

        candidate_dist, _ = (
            nn_select
            .kneighbors(
                X_candidate
            )
        )

        candidate_kdist = (
            candidate_dist[
                :,
                -1
            ]
        )

        reference = (
            candidates
            .loc[
                candidate_kdist
                <=
                threshold
            ]
            .copy()
        )


        # reference 너무 적으면 실패 처리
        if len(reference) < 50:

            print(
                f"[SKIP] {label} / "
                f"Session {valid_session} / "
                f"Reference={len(reference)}"
            )

            return None, None


        # ====================================================
        # Anomaly scaler
        # ====================================================

        anomaly_scaler = (
            StandardScaler()
        )

        X_ref = (
            anomaly_scaler
            .fit_transform(
                reference[
                    raw_features
                ]
            )
        )

        X_train = (
            anomaly_scaler
            .transform(
                train[
                    raw_features
                ]
            )
        )

        X_valid = (
            anomaly_scaler
            .transform(
                valid[
                    raw_features
                ]
            )
        )


        # ====================================================
        # PCA
        # ====================================================

        pca = (
            PCA(
                n_components=0.95,
                random_state=42
            )
        )

        pca.fit(
            X_ref
        )


        # Train
        train_z = (
            pca.transform(
                X_train
            )
        )

        train_recon = (
            pca.inverse_transform(
                train_z
            )
        )

        train_pca = (
            np.mean(
                (
                    X_train
                    -
                    train_recon
                ) ** 2,
                axis=1
            )
        )


        # Valid
        valid_z = (
            pca.transform(
                X_valid
            )
        )

        valid_recon = (
            pca.inverse_transform(
                valid_z
            )
        )

        valid_pca = (
            np.mean(
                (
                    X_valid
                    -
                    valid_recon
                ) ** 2,
                axis=1
            )
        )


        # ====================================================
        # KNN anomaly
        # ====================================================

        k_anomaly = min(
            10,
            len(reference)
        )

        knn = (
            NearestNeighbors(
                n_neighbors=k_anomaly,
                metric="euclidean"
            )
        )

        knn.fit(
            X_ref
        )

        train_knn_dist, _ = (
            knn.kneighbors(
                X_train
            )
        )

        valid_knn_dist, _ = (
            knn.kneighbors(
                X_valid
            )
        )

        train_knn = (
            train_knn_dist
            .mean(axis=1)
        )

        valid_knn = (
            valid_knn_dist
            .mean(axis=1)
        )


        # ====================================================
        # 최종 PCA + KNN
        # ====================================================

        X_train_model = pd.DataFrame({

            "anomaly_pca":
                train_pca,

            "anomaly_knn":
                train_knn
        })

        X_valid_model = pd.DataFrame({

            "anomaly_pca":
                valid_pca,

            "anomaly_knn":
                valid_knn
        })


        y_train = (
            train["PassOrFail"]
            .astype(int)
            .reset_index(drop=True)
        )

        y_valid = (
            valid["PassOrFail"]
            .astype(int)
            .reset_index(drop=True)
        )


        # ====================================================
        # ExtraTrees
        # ====================================================

        model = (
            make_model()
        )

        model.fit(
            X_train_model,
            y_train
        )

        prob = (
            model
            .predict_proba(
                X_valid_model
            )[:, 1]
        )


        # ====================================================
        # Fold metric
        # ====================================================

        baseline = (
            y_valid.mean()
        )

        pr_auc = (
            average_precision_score(
                y_valid,
                prob
            )
        )

        roc_auc = (
            roc_auc_score(
                y_valid,
                prob
            )
        )


        fold_results.append({

            "Strategy":
                label,

            "Valid_Session":
                valid_session,

            "Valid_N":
                len(valid),

            "Defects":
                int(
                    y_valid.sum()
                ),

            "Candidates":
                len(candidates),

            "Reference_N":
                len(reference),

            "Reference_Ratio":
                len(reference)
                /
                len(candidates),

            "Baseline_PR":
                baseline,

            "PR_AUC":
                pr_auc,

            "PR_Ratio":
                pr_auc
                /
                baseline,

            "ROC_AUC":
                roc_auc
        })


        # ====================================================
        # OOF 저장
        # ====================================================

        temp = pd.DataFrame({

            "Strategy":
                label,

            "TimeStamp":
                valid[
                    "TimeStamp"
                ].values,

            "session_id":
                valid_session,

            "y_true":
                y_valid.values,

            "y_prob":
                prob
        })

        oof_rows.append(
            temp
        )


    return (
        pd.DataFrame(
            fold_results
        ),
        pd.concat(
            oof_rows,
            ignore_index=True
        )
    )


# ============================================================
# 6. 실험 설정
# ============================================================

configs = [

    # --------------------------------------------
    # Q 민감도
    # --------------------------------------------

    {
        "label":
            "Q90_All",

        "quantile":
            0.90,

        "history_days":
            None
    },

    {
        "label":
            "Q95_All",

        "quantile":
            0.95,

        "history_days":
            None
    },

    {
        "label":
            "Q99_All",

        "quantile":
            0.99,

        "history_days":
            None
    },


    # --------------------------------------------
    # 최근 Reference
    # --------------------------------------------

    {
        "label":
            "Q95_Recent3D",

        "quantile":
            0.95,

        "history_days":
            3
    },

    {
        "label":
            "Q95_Recent1D",

        "quantile":
            0.95,

        "history_days":
            1
    }
]


# ============================================================
# 7. 전체 실행
# ============================================================

all_fold = []
all_oof = []


for config in configs:

    print(
        "\n"
        + "=" * 100
    )

    print(
        "RUN:",
        config[
            "label"
        ]
    )

    print(
        "=" * 100
    )


    fold_result, oof_result = (
        evaluate_reference_strategy(
            quantile=
                config[
                    "quantile"
                ],

            history_days=
                config[
                    "history_days"
                ],

            label=
                config[
                    "label"
                ]
        )
    )


    if fold_result is None:
        continue


    all_fold.append(
        fold_result
    )

    all_oof.append(
        oof_result
    )


# ============================================================
# 8. 결과 합치기
# ============================================================

fold_df = (
    pd.concat(
        all_fold,
        ignore_index=True
    )
)

oof_df = (
    pd.concat(
        all_oof,
        ignore_index=True
    )
)


# ============================================================
# 9. Fold 결과
# ============================================================

print(
    "\n\n"
    + "=" * 100
)

print(
    "ROBUSTNESS - FOLD RESULTS"
)

print(
    "=" * 100
)


print(
    fold_df[
        [
            "Strategy",
            "Valid_Session",
            "Defects",
            "Candidates",
            "Reference_N",
            "Reference_Ratio",
            "Baseline_PR",
            "PR_AUC",
            "PR_Ratio",
            "ROC_AUC"
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 10. 전략별 OOF 성능
# ============================================================

summary_rows = []


for strategy, group in (
    oof_df
    .groupby(
        "Strategy"
    )
):

    baseline = (
        group[
            "y_true"
        ]
        .mean()
    )

    pr_auc = (
        average_precision_score(
            group[
                "y_true"
            ],
            group[
                "y_prob"
            ]
        )
    )

    roc_auc = (
        roc_auc_score(
            group[
                "y_true"
            ],
            group[
                "y_prob"
            ]
        )
    )


    fold_part = (
        fold_df[
            fold_df[
                "Strategy"
            ]
            ==
            strategy
        ]
    )


    wins = (
        fold_part[
            "PR_AUC"
        ]
        >
        fold_part[
            "Baseline_PR"
        ]
    ).sum()


    summary_rows.append({

        "Strategy":
            strategy,

        "OOF_PR_AUC":
            pr_auc,

        "Baseline_PR":
            baseline,

        "PR_Ratio":
            pr_auc
            /
            baseline,

        "PR_Median":
            fold_part[
                "PR_AUC"
            ]
            .median(),

        "PR_Min":
            fold_part[
                "PR_AUC"
            ]
            .min(),

        "PR_Max":
            fold_part[
                "PR_AUC"
            ]
            .max(),

        "Fold_Wins":
            int(
                wins
            ),

        "OOF_ROC_AUC":
            roc_auc,

        "Reference_Mean":
            fold_part[
                "Reference_N"
            ]
            .mean(),

        "Reference_Min":
            fold_part[
                "Reference_N"
            ]
            .min()
    })


summary_df = (
    pd.DataFrame(
        summary_rows
    )
    .sort_values(
        "OOF_PR_AUC",
        ascending=False
    )
)


print(
    "\n\n"
    + "=" * 100
)

print(
    "ROBUSTNESS SUMMARY"
)

print(
    "=" * 100
)


print(
    summary_df
    .to_string(
        index=False
    )
)


# ============================================================
# 11. Canonical Q95_All만 Top-K 정책 평가
#
# 중요:
# 설정 중 최고 성능 전략이 아니라
# 기존에 확정한 Q95_All 사용
# ============================================================

canonical = (
    oof_df[
        oof_df[
            "Strategy"
        ]
        ==
        "Q95_All"
    ]
    .copy()
)


print(
    "\n\n"
    + "=" * 100
)

print(
    "CANONICAL MODEL"
)

print(
    "=" * 100
)

print(
    "Strategy: Q95_All"
)

print(
    "OOF N:",
    len(canonical)
)

print(
    "Defects:",
    int(
        canonical[
            "y_true"
        ].sum()
    )
)

print(
    "PR-AUC:",
    average_precision_score(
        canonical[
            "y_true"
        ],
        canonical[
            "y_prob"
        ]
    )
)


# ============================================================
# 12. Session별 Top-K
#
# 각 미래 Session 안에서
# 위험도 상위 X%만 검사한다고 가정
# ============================================================

top_fractions = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]


topk_rows = []


for fraction in top_fractions:

    inspection_parts = []


    for session_id, group in (
        canonical
        .groupby(
            "session_id"
        )
    ):

        group = (
            group
            .sort_values(
                "y_prob",
                ascending=False
            )
            .copy()
        )


        n_inspect = max(
            1,
            int(
                np.ceil(
                    len(group)
                    *
                    fraction
                )
            )
        )


        group[
            "inspect"
        ] = 0


        group.iloc[
            :n_inspect,
            group.columns
            .get_loc(
                "inspect"
            )
        ] = 1


        inspection_parts.append(
            group
        )


    policy = (
        pd.concat(
            inspection_parts,
            ignore_index=True
        )
    )


    inspected = (
        policy[
            policy[
                "inspect"
            ]
            ==
            1
        ]
    )


    tp = int(
        inspected[
            "y_true"
        ]
        .sum()
    )


    total_defects = int(
        policy[
            "y_true"
        ]
        .sum()
    )


    inspection_n = (
        len(inspected)
    )


    recall = (
        tp
        /
        total_defects
    )


    precision = (
        tp
        /
        inspection_n
        if inspection_n > 0
        else 0
    )


    lift = (
        precision
        /
        (
            total_defects
            /
            len(policy)
        )
    )


    topk_rows.append({

        "Top_Fraction":
            fraction,

        "Inspection_N":
            inspection_n,

        "Inspection_Rate":
            inspection_n
            /
            len(policy),

        "Captured_Defects":
            tp,

        "Total_Defects":
            total_defects,

        "Recall":
            recall,

        "Precision":
            precision,

        "Precision_Lift":
            lift
    })


topk_df = (
    pd.DataFrame(
        topk_rows
    )
)


# ============================================================
# 13. Top-K 출력
# ============================================================

print(
    "\n\n"
    + "=" * 100
)

print(
    "TOP-K INSPECTION POLICY"
)

print(
    "=" * 100
)


pretty = (
    topk_df
    .copy()
)


pretty[
    "Top_Fraction"
] *= 100

pretty[
    "Inspection_Rate"
] *= 100

pretty[
    "Recall"
] *= 100

pretty[
    "Precision"
] *= 100


pretty = (
    pretty.rename(
        columns={
            "Top_Fraction":
                "Target_Top_%",

            "Inspection_Rate":
                "Actual_Inspection_%",

            "Recall":
                "Recall_%",

            "Precision":
                "Precision_%"
        }
    )
)


print(
    pretty
    .to_string(
        index=False
    )
)


# ============================================================
# 14. Session별 Top30 상세
# ============================================================

TOP30 = 0.30

session_top30_rows = []


for session_id, group in (
    canonical
    .groupby(
        "session_id"
    )
):

    group = (
        group
        .sort_values(
            "y_prob",
            ascending=False
        )
        .copy()
    )


    n_inspect = max(
        1,
        int(
            np.ceil(
                len(group)
                *
                TOP30
            )
        )
    )


    inspected = (
        group.iloc[
            :n_inspect
        ]
    )


    total_defects = int(
        group[
            "y_true"
        ]
        .sum()
    )


    captured = int(
        inspected[
            "y_true"
        ]
        .sum()
    )


    session_top30_rows.append({

        "Session":
            session_id,

        "Session_N":
            len(group),

        "Defects":
            total_defects,

        "Inspection_N":
            n_inspect,

        "Captured_Defects":
            captured,

        "Recall":
            (
                captured
                /
                total_defects
                if total_defects > 0
                else np.nan
            ),

        "Precision":
            (
                captured
                /
                n_inspect
            )
    })


session_top30 = (
    pd.DataFrame(
        session_top30_rows
    )
)


print(
    "\n\n"
    + "=" * 100
)

print(
    "SESSION TOP30 POLICY"
)

print(
    "=" * 100
)


print(
    session_top30
    .to_string(
        index=False
    )
)

# 1단계 — labeled_data.csv 전체 RG3 구조 재감사

In [ ]:
# ============================================================
# RG3 FULL LABELED DATA AUDIT
#
# 목적:
# 1. labeled_data.csv 전체 기간 RG3 W/SHLD 확인
# 2. 날짜별 LH/RH 불량 분포
# 3. 2020-11-04 Holdout 존재 여부
# 4. Cycle-level target 구조
# 5. LH/RH 센서 동일 여부
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. LOAD
# ============================================================

BASE_DIR = (
    r"../../.."
    r"\1. 사출성형기 AI 데이터셋"
    r"\1. 사출성형기 AI 데이터셋"
)

PATH = (
    BASE_DIR
    + r"\labeled_data.csv"
)


df = pd.read_csv(PATH)


print("=" * 100)
print("RAW LABELED DATA")
print("=" * 100)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. TimeStamp
# ============================================================

df["TimeStamp"] = pd.to_datetime(
    df["TimeStamp"],
    errors="coerce"
)


# ============================================================
# 3. RG3 관련 제품명 전체 확인
# ============================================================

rg3_names = (
    df.loc[
        df["PART_NAME"]
        .astype(str)
        .str.contains(
            "RG3",
            case=False,
            na=False
        ),
        "PART_NAME"
    ]
    .value_counts()
)


print("\n" + "=" * 100)
print("ALL RG3 PRODUCT NAMES")
print("=" * 100)

print(
    rg3_names.to_string()
)


# ============================================================
# 4. 우리가 볼 W/SHLD LH/RH만 정확히 선택
# ============================================================

TARGET_LH = "RG3 MOLD'G W/SHLD, LH"
TARGET_RH = "RG3 MOLD'G W/SHLD, RH"


rg3 = (
    df[
        df["PART_NAME"]
        .isin(
            [
                TARGET_LH,
                TARGET_RH
            ]
        )
    ]
    .copy()
    .sort_values(
        [
            "TimeStamp",
            "PART_NAME"
        ]
    )
    .reset_index(drop=True)
)


rg3["Side"] = np.where(
    rg3["PART_NAME"] == TARGET_LH,
    "LH",
    "RH"
)


print("\n" + "=" * 100)
print("RG3 W/SHLD RAW")
print("=" * 100)

print(
    "Rows:",
    len(rg3)
)

print(
    "Unique TimeStamp:",
    rg3["TimeStamp"].nunique()
)

print(
    "Time:",
    rg3["TimeStamp"].min(),
    "~",
    rg3["TimeStamp"].max()
)

print("\nSide:")
print(
    rg3["Side"]
    .value_counts()
)


# ============================================================
# 5. Target 원본값 확인
# ============================================================

print("\n" + "=" * 100)
print("RAW TARGET VALUES")
print("=" * 100)

print(
    rg3["PassOrFail"]
    .value_counts(
        dropna=False
    )
)


# ============================================================
# 6. Target 변환
#
# KAMP 원본에서
# Y = 양품
# N = 불량
#
# 만약 이미 0/1이면 그대로 처리
# ============================================================

raw_target = (
    rg3["PassOrFail"]
    .astype(str)
    .str.strip()
    .str.upper()
)


def convert_target(x):

    # 원본 문자
    if x == "Y":
        return 0

    if x == "N":
        return 1

    # 이미 숫자형으로 저장된 경우
    if x in [
        "0",
        "0.0"
    ]:
        return 0

    if x in [
        "1",
        "1.0"
    ]:
        return 1

    return np.nan


rg3["Target"] = (
    raw_target
    .map(
        convert_target
    )
)


print("\nConverted Target:")

print(
    rg3["Target"]
    .value_counts(
        dropna=False
    )
)


# 변환 실패 방지
if rg3["Target"].isna().any():

    print(
        "\n[WARNING] "
        "변환되지 않은 Target 존재"
    )

    print(
        rg3.loc[
            rg3["Target"].isna(),
            "PassOrFail"
        ]
        .value_counts(
            dropna=False
        )
    )


# ============================================================
# 7. 날짜
# ============================================================

rg3["date"] = (
    rg3["TimeStamp"]
    .dt.date
)


# ============================================================
# 8. 날짜 × Side × Target
# ============================================================

date_side_target = (
    rg3
    .groupby(
        [
            "date",
            "Side",
            "Target"
        ],
        dropna=False
    )
    .size()
    .unstack(
        fill_value=0
    )
)


print("\n" + "=" * 100)
print("DATE × SIDE × TARGET")
print("=" * 100)

print(
    date_side_target
    .to_string()
)


# ============================================================
# 9. 날짜별 Side 불량
# ============================================================

date_side_defects = (
    rg3
    .groupby(
        [
            "date",
            "Side"
        ]
    )
    ["Target"]
    .agg(
        Rows="size",
        Defects="sum"
    )
    .reset_index()
)


date_side_defects[
    "Defect_Rate"
] = (
    date_side_defects[
        "Defects"
    ]
    /
    date_side_defects[
        "Rows"
    ]
)


print("\n" + "=" * 100)
print("DATE × SIDE DEFECT SUMMARY")
print("=" * 100)

print(
    date_side_defects
    .to_string(
        index=False
    )
)


# ============================================================
# 10. Cycle 단위 구조
# ============================================================

cycle = (
    rg3
    .groupby(
        "TimeStamp"
    )
    .agg(

        Rows=(
            "Target",
            "size"
        ),

        LH_Target=(
            "Target",
            lambda x:
            rg3.loc[
                x.index
            ]
            .loc[
                lambda z:
                z["Side"] == "LH",
                "Target"
            ]
            .iloc[0]
            if (
                rg3.loc[
                    x.index,
                    "Side"
                ]
                ==
                "LH"
            ).any()
            else np.nan
        ),

        RH_Target=(
            "Target",
            lambda x:
            rg3.loc[
                x.index
            ]
            .loc[
                lambda z:
                z["Side"] == "RH",
                "Target"
            ]
            .iloc[0]
            if (
                rg3.loc[
                    x.index,
                    "Side"
                ]
                ==
                "RH"
            ).any()
            else np.nan
        )
    )
    .reset_index()
)


cycle["CycleDefect"] = (
    cycle[
        [
            "LH_Target",
            "RH_Target"
        ]
    ]
    .max(
        axis=1
    )
)


cycle["date"] = (
    cycle["TimeStamp"]
    .dt.date
)


print("\n" + "=" * 100)
print("CYCLE STRUCTURE")
print("=" * 100)

print(
    "Cycles:",
    len(cycle)
)

print("\nRows per Cycle:")

print(
    cycle["Rows"]
    .value_counts()
    .sort_index()
)


print("\nLH/RH Target Combination:")

print(
    cycle[
        [
            "LH_Target",
            "RH_Target"
        ]
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 11. 날짜별 Cycle defect
# ============================================================

date_cycle = (
    cycle
    .groupby(
        "date"
    )
    .agg(

        Cycles=(
            "CycleDefect",
            "size"
        ),

        Defect_Cycles=(
            "CycleDefect",
            "sum"
        ),

        LH_Defects=(
            "LH_Target",
            "sum"
        ),

        RH_Defects=(
            "RH_Target",
            "sum"
        )
    )
)


date_cycle[
    "Cycle_Defect_Rate"
] = (
    date_cycle[
        "Defect_Cycles"
    ]
    /
    date_cycle[
        "Cycles"
    ]
)


print("\n" + "=" * 100)
print("DATE CYCLE SUMMARY")
print("=" * 100)

print(
    date_cycle
    .to_string()
)


# ============================================================
# 12. 2020-11-04 상세
# ============================================================

HOLDOUT_DATE = pd.Timestamp(
    "2020-11-04"
).date()


nov4_rows = (
    rg3[
        rg3["date"]
        ==
        HOLDOUT_DATE
    ]
    .copy()
)


nov4_cycle = (
    cycle[
        cycle["date"]
        ==
        HOLDOUT_DATE
    ]
    .copy()
)


print("\n" + "=" * 100)
print("2020-11-04 HOLDOUT CHECK")
print("=" * 100)


print(
    "Rows:",
    len(nov4_rows)
)

print(
    "Cycles:",
    len(nov4_cycle)
)


print("\nSide × Target:")

print(
    pd.crosstab(
        nov4_rows[
            "Side"
        ],
        nov4_rows[
            "Target"
        ]
    )
)


print("\nCycle Target Combination:")

print(
    nov4_cycle[
        [
            "LH_Target",
            "RH_Target"
        ]
    ]
    .value_counts()
)


print(
    "\nDefect Cycles:",
    int(
        nov4_cycle[
            "CycleDefect"
        ]
        .sum()
    )
)


# ============================================================
# 13. 11/4 불량 Cycle 상세
# ============================================================

nov4_defects = (
    nov4_cycle[
        nov4_cycle[
            "CycleDefect"
        ]
        ==
        1
    ]
    .copy()
)


print("\n" + "=" * 100)
print("2020-11-04 DEFECT CYCLES")
print("=" * 100)

print(
    nov4_defects[
        [
            "TimeStamp",
            "LH_Target",
            "RH_Target",
            "CycleDefect"
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 14. 센서 컬럼
# ============================================================

sensor_cols = [
    'Injection_Time',
    'Filling_Time',
    'Plasticizing_Time',
    'Cycle_Time',
    'Clamp_Close_Time',
    'Cushion_Position',
    'Plasticizing_Position',
    'Clamp_Open_Position',
    'Max_Injection_Speed',
    'Max_Screw_RPM',
    'Average_Screw_RPM',
    'Max_Injection_Pressure',
    'Max_Switch_Over_Pressure',
    'Max_Back_Pressure',
    'Average_Back_Pressure',
    'Barrel_Temperature_1',
    'Barrel_Temperature_2',
    'Barrel_Temperature_3',
    'Barrel_Temperature_4',
    'Barrel_Temperature_5',
    'Barrel_Temperature_6',
    'Hopper_Temperature',
    'Mold_Temperature_3',
    'Mold_Temperature_4'
]


sensor_cols = [
    c
    for c in sensor_cols
    if c in rg3.columns
]


# ============================================================
# 15. 동일 TimeStamp에서 LH/RH 센서 동일 여부
#
# np.isclose 사용:
# float 저장 정밀도 차이 허용
# ============================================================

sensor_check_rows = []


for timestamp, group in (
    rg3
    .groupby(
        "TimeStamp"
    )
):

    if set(
        group["Side"]
    ) != {
        "LH",
        "RH"
    }:
        continue


    lh = (
        group[
            group[
                "Side"
            ]
            ==
            "LH"
        ]
        .iloc[0]
    )


    rh = (
        group[
            group[
                "Side"
            ]
            ==
            "RH"
        ]
        .iloc[0]
    )


    different_cols = []


    for col in sensor_cols:

        a = lh[col]
        b = rh[col]


        if (
            pd.isna(a)
            and
            pd.isna(b)
        ):
            continue


        if not np.isclose(
            a,
            b,
            rtol=1e-7,
            atol=1e-7,
            equal_nan=True
        ):

            different_cols.append(
                col
            )


    sensor_check_rows.append({

        "TimeStamp":
            timestamp,

        "date":
            timestamp.date(),

        "LH_Target":
            lh[
                "Target"
            ],

        "RH_Target":
            rh[
                "Target"
            ],

        "N_Different_Sensors":
            len(
                different_cols
            ),

        "Different_Sensors":
            ", ".join(
                different_cols
            )
    })


sensor_check = pd.DataFrame(
    sensor_check_rows
)


print("\n" + "=" * 100)
print("LH / RH SENSOR CONSISTENCY")
print("=" * 100)


print(
    "Checked Cycles:",
    len(sensor_check)
)

print(
    "Identical sensors:",
    (
        sensor_check[
            "N_Different_Sensors"
        ]
        ==
        0
    ).sum()
)

print(
    "Different sensors:",
    (
        sensor_check[
            "N_Different_Sensors"
        ]
        >
        0
    ).sum()
)


# ============================================================
# 16. 11/4 sensor consistency
# ============================================================

nov4_sensor = (
    sensor_check[
        sensor_check[
            "date"
        ]
        ==
        HOLDOUT_DATE
    ]
)


print("\n" + "=" * 100)
print("2020-11-04 LH / RH SENSOR CONSISTENCY")
print("=" * 100)


print(
    "Cycles:",
    len(nov4_sensor)
)

print(
    "Identical:",
    (
        nov4_sensor[
            "N_Different_Sensors"
        ]
        ==
        0
    ).sum()
)

print(
    "Different:",
    (
        nov4_sensor[
            "N_Different_Sensors"
        ]
        >
        0
    ).sum()
)


if (
    nov4_sensor[
        "N_Different_Sensors"
    ]
    >
    0
).any():

    print(
        "\nDifferent sample:"
    )

    print(
        nov4_sensor[
            nov4_sensor[
                "N_Different_Sensors"
            ]
            >
            0
        ]
        .head(20)
        .to_string(
            index=False
        )
    )


# ============================================================
# 17. 전체 핵심 요약
# ============================================================

print("\n\n" + "=" * 100)
print("FINAL AUDIT SUMMARY")
print("=" * 100)


print(
    f"""
RG3 W/SHLD Rows       : {len(rg3)}
RG3 W/SHLD Cycles     : {len(cycle)}

Date Range            : {rg3['TimeStamp'].min()} ~ {rg3['TimeStamp'].max()}

Total Defect Cycles   : {int(cycle['CycleDefect'].sum())}

Total LH Defects      : {int(cycle['LH_Target'].sum())}
Total RH Defects      : {int(cycle['RH_Target'].sum())}

2020-11-04 Rows       : {len(nov4_rows)}
2020-11-04 Cycles     : {len(nov4_cycle)}
2020-11-04 Def Cycles : {int(nov4_cycle['CycleDefect'].sum())}

2020-11-04 LH Defects : {int(nov4_cycle['LH_Target'].sum())}
2020-11-04 RH Defects : {int(nov4_cycle['RH_Target'].sum())}

All Sensor-identical Cycles:
{(sensor_check['N_Different_Sensors'] == 0).sum()} / {len(sensor_check)}

Nov4 Sensor-identical Cycles:
{(nov4_sensor['N_Different_Sensors'] == 0).sum()} / {len(nov4_sensor)}
"""
)

In [ ]:
# ============================================================
# 11/4 불완전 Cycle 확인
# ============================================================

nov4 = rg3[
    rg3["date"] == pd.Timestamp("2020-11-04").date()
].copy()

nov4_cycle_size = (
    nov4
    .groupby("TimeStamp")
    .size()
)

incomplete_times = (
    nov4_cycle_size[
        nov4_cycle_size != 2
    ]
    .index
)

print("=" * 100)
print("11/04 INCOMPLETE CYCLE")
print("=" * 100)

print(
    nov4[
        nov4["TimeStamp"].isin(incomplete_times)
    ][
        [
            "TimeStamp",
            "Side",
            "PassOrFail",
            "Target",
            "Reason"
        ]
    ].to_string(index=False)
)


print("\nComplete Holdout:")

nov4_complete_times = (
    nov4_cycle_size[
        nov4_cycle_size == 2
    ]
    .index
)

nov4_complete = (
    nov4[
        nov4["TimeStamp"].isin(nov4_complete_times)
    ]
)

holdout_cycle_target = (
    nov4_complete
    .groupby("TimeStamp")["Target"]
    .max()
)

print("Cycles :", len(holdout_cycle_target))
print("Defects:", int(holdout_cycle_target.sum()))
print("Normal :", int((holdout_cycle_target == 0).sum()))

In [ ]:
# ============================================================
# RG3 FINAL TEMPORAL HOLDOUT
#
# Development:
#   2020-10-21 ~ 2020-10-23
#   591 Cycles / 25 Defects
#
# Final Holdout:
#   2020-11-04
#   Complete paired cycles only
#   35 Cycles / 7 Defects
#
# FIXED PIPELINE:
#   Q95 Local Reference
#   PCA + KNN
#   ExtraTrees
#
# 중요:
#   이 결과를 본 뒤 모델/설정을 다시 변경하지 않음
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# ============================================================
# 1. 원본 데이터
# ============================================================

BASE_DIR = (
    r"../../.."
    r"\1. 사출성형기 AI 데이터셋"
    r"\1. 사출성형기 AI 데이터셋"
)

LABELED_PATH = (
    BASE_DIR
    + r"\labeled_data.csv"
)

UNLABELED_PATH = (
    BASE_DIR
    + r"\unlabeled_data.csv"
)


labeled = pd.read_csv(
    LABELED_PATH
)

unlabeled = pd.read_csv(
    UNLABELED_PATH
)


labeled["TimeStamp"] = pd.to_datetime(
    labeled["TimeStamp"]
)

unlabeled["TimeStamp"] = pd.to_datetime(
    unlabeled["TimeStamp"]
)


# ============================================================
# 2. RG3
# ============================================================

TARGET_LH = "RG3 MOLD'G W/SHLD, LH"
TARGET_RH = "RG3 MOLD'G W/SHLD, RH"


rg3 = (
    labeled[
        labeled["PART_NAME"]
        .isin(
            [
                TARGET_LH,
                TARGET_RH
            ]
        )
    ]
    .copy()
)


rg3["Side"] = np.where(
    rg3["PART_NAME"] == TARGET_LH,
    "LH",
    "RH"
)


# Y = 정상 0
# N = 불량 1
rg3["Target"] = (
    rg3["PassOrFail"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({
        "Y": 0,
        "N": 1
    })
)


# ============================================================
# 3. 센서
# ============================================================

sensor_cols = [
    'Injection_Time',
    'Filling_Time',
    'Plasticizing_Time',
    'Cycle_Time',
    'Clamp_Close_Time',
    'Cushion_Position',
    'Plasticizing_Position',
    'Clamp_Open_Position',
    'Max_Injection_Speed',
    'Max_Screw_RPM',
    'Average_Screw_RPM',
    'Max_Injection_Pressure',
    'Max_Switch_Over_Pressure',
    'Max_Back_Pressure',
    'Average_Back_Pressure',
    'Barrel_Temperature_1',
    'Barrel_Temperature_2',
    'Barrel_Temperature_3',
    'Barrel_Temperature_4',
    'Barrel_Temperature_5',
    'Barrel_Temperature_6',
    'Hopper_Temperature',
    'Mold_Temperature_3',
    'Mold_Temperature_4'
]


# ============================================================
# 4. Cycle 단위 데이터 생성
#
# LH/RH 센서는 동일하므로
# 첫 번째 행의 Sensor 사용
#
# Target은 LH/RH 중 하나라도 불량이면 1
# ============================================================

cycle_size = (
    rg3
    .groupby("TimeStamp")
    .size()
)


cycle_sensor = (
    rg3
    .groupby("TimeStamp")[
        sensor_cols
    ]
    .first()
)


cycle_target = (
    rg3
    .groupby("TimeStamp")[
        "Target"
    ]
    .max()
)


cycle_df = (
    cycle_sensor
    .join(
        cycle_target
        .rename(
            "CycleDefect"
        )
    )
    .reset_index()
)


cycle_df["Rows"] = (
    cycle_df["TimeStamp"]
    .map(
        cycle_size
    )
)


cycle_df["date"] = (
    cycle_df["TimeStamp"]
    .dt.date
)


# ============================================================
# 5. Development / Holdout
# ============================================================

DEV_END = (
    pd.Timestamp(
        "2020-10-23 23:59:59"
    )
)

HOLDOUT_DATE = (
    pd.Timestamp(
        "2020-11-04"
    )
    .date()
)


dev = (
    cycle_df[
        cycle_df["TimeStamp"]
        <=
        DEV_END
    ]
    .copy()
    .sort_values(
        "TimeStamp"
    )
    .reset_index(drop=True)
)


# Holdout은 LH/RH 둘 다 존재하는 Cycle만
holdout = (
    cycle_df[
        (
            cycle_df["date"]
            ==
            HOLDOUT_DATE
        )
        &
        (
            cycle_df["Rows"]
            ==
            2
        )
    ]
    .copy()
    .sort_values(
        "TimeStamp"
    )
    .reset_index(drop=True)
)


print("=" * 100)
print("FINAL DATA SPLIT")
print("=" * 100)

print(
    "Development:",
    len(dev),
    "| Defects:",
    int(
        dev[
            "CycleDefect"
        ].sum()
    )
)

print(
    "Holdout:",
    len(holdout),
    "| Defects:",
    int(
        holdout[
            "CycleDefect"
        ].sum()
    )
)


assert len(dev) == 591
assert int(
    dev["CycleDefect"].sum()
) == 25

assert len(holdout) == 35
assert int(
    holdout["CycleDefect"].sum()
) == 7


# ============================================================
# 6. 상수 센서 제거
# ============================================================

raw_features = [
    col
    for col in sensor_cols
    if dev[col].nunique(
        dropna=False
    ) > 1
]


print(
    "\nUsable sensors:",
    len(raw_features)
)

print(
    "Removed:",
    [
        c
        for c in sensor_cols
        if c not in raw_features
    ]
)


# ============================================================
# 7. Unlabeled
#
# 기존 pipeline과 동일하게
# RG3 RH만 사용
# ============================================================

unlab = (
    unlabeled[
        unlabeled[
            "PART_NAME"
        ]
        ==
        TARGET_RH
    ]
    .copy()
    .sort_values(
        "TimeStamp"
    )
)


# ============================================================
# 8. 모든 Labeled Timestamp 제거
#
# Holdout와 동일 Cycle이
# Unlabeled를 통해 들어오는 Leakage 방지
# ============================================================

all_labeled_times = set(
    cycle_df[
        "TimeStamp"
    ]
)


unlab_clean = (
    unlab[
        ~unlab["TimeStamp"]
        .isin(
            all_labeled_times
        )
    ]
    .copy()
)


# 동일 Timestamp 중복 제거
unlab_clean = (
    unlab_clean
    .groupby(
        "TimeStamp",
        as_index=False
    )[raw_features]
    .mean()
    .sort_values(
        "TimeStamp"
    )
    .reset_index(drop=True)
)


# ============================================================
# 9. Holdout 시작 이전 Unlabeled만
# ============================================================

holdout_start = (
    holdout[
        "TimeStamp"
    ].min()
)


unlab_past = (
    unlab_clean[
        unlab_clean[
            "TimeStamp"
        ]
        <
        holdout_start
    ]
    .copy()
)


print(
    "Past unlabeled:",
    len(unlab_past)
)

print(
    "Latest unlabeled:",
    unlab_past[
        "TimeStamp"
    ].max()
)


# ============================================================
# 10. Development 정상 Cycle
# ============================================================

dev_normal = (
    dev[
        dev[
            "CycleDefect"
        ]
        ==
        0
    ]
    .copy()
)


print(
    "Dev normal:",
    len(dev_normal)
)


# ============================================================
# 11. Q95 Local Reference Selection
#
# Q95는 개발 단계에서 이미 결정됨
# ============================================================

selector_scaler = (
    StandardScaler()
)


X_normal = (
    selector_scaler
    .fit_transform(
        dev_normal[
            raw_features
        ]
    )
)


X_unlab = (
    selector_scaler
    .transform(
        unlab_past[
            raw_features
        ]
    )
)


K_SELECT = 5


# ------------------------------------------------------------
# 정상 내부 거리 → Q95
# ------------------------------------------------------------

nn_normal = (
    NearestNeighbors(
        n_neighbors=
            K_SELECT + 1,
        metric="euclidean"
    )
)


nn_normal.fit(
    X_normal
)


normal_dist, _ = (
    nn_normal
    .kneighbors(
        X_normal
    )
)


normal_kdist = (
    normal_dist[
        :,
        K_SELECT
    ]
)


Q95 = (
    np.quantile(
        normal_kdist,
        0.95
    )
)


print(
    "\nFixed Q95:",
    Q95
)


# ------------------------------------------------------------
# 과거 Unlabeled → 정상 거리
# ------------------------------------------------------------

nn_select = (
    NearestNeighbors(
        n_neighbors=
            K_SELECT,
        metric="euclidean"
    )
)


nn_select.fit(
    X_normal
)


unlab_dist, _ = (
    nn_select
    .kneighbors(
        X_unlab
    )
)


unlab_kdist = (
    unlab_dist[
        :,
        -1
    ]
)


reference = (
    unlab_past
    .loc[
        unlab_kdist
        <=
        Q95
    ]
    .copy()
)


print(
    "Final Local Reference:",
    len(reference)
)

print(
    "Reference ratio:",
    len(reference)
    /
    len(unlab_past)
)


print(
    "Reference Time:",
    reference[
        "TimeStamp"
    ].min(),
    "~",
    reference[
        "TimeStamp"
    ].max()
)


# ============================================================
# 12. Anomaly Scaler
# ============================================================

anomaly_scaler = (
    StandardScaler()
)


X_ref = (
    anomaly_scaler
    .fit_transform(
        reference[
            raw_features
        ]
    )
)


X_dev = (
    anomaly_scaler
    .transform(
        dev[
            raw_features
        ]
    )
)


X_holdout = (
    anomaly_scaler
    .transform(
        holdout[
            raw_features
        ]
    )
)


# ============================================================
# 13. PCA
# ============================================================

pca = (
    PCA(
        n_components=0.95,
        random_state=42
    )
)


pca.fit(
    X_ref
)


# Development
z_dev = (
    pca.transform(
        X_dev
    )
)

recon_dev = (
    pca.inverse_transform(
        z_dev
    )
)

dev_pca = (
    np.mean(
        (
            X_dev
            -
            recon_dev
        ) ** 2,
        axis=1
    )
)


# Holdout
z_holdout = (
    pca.transform(
        X_holdout
    )
)

recon_holdout = (
    pca.inverse_transform(
        z_holdout
    )
)

holdout_pca = (
    np.mean(
        (
            X_holdout
            -
            recon_holdout
        ) ** 2,
        axis=1
    )
)


# ============================================================
# 14. KNN Distance
# ============================================================

K_ANOMALY = 10


knn = (
    NearestNeighbors(
        n_neighbors=
            K_ANOMALY,
        metric="euclidean"
    )
)


knn.fit(
    X_ref
)


dev_knn_dist, _ = (
    knn.kneighbors(
        X_dev
    )
)

holdout_knn_dist, _ = (
    knn.kneighbors(
        X_holdout
    )
)


dev_knn = (
    dev_knn_dist
    .mean(
        axis=1
    )
)

holdout_knn = (
    holdout_knn_dist
    .mean(
        axis=1
    )
)


# ============================================================
# 15. 최종 Feature
# ============================================================

X_dev_final = pd.DataFrame({

    "anomaly_pca":
        dev_pca,

    "anomaly_knn":
        dev_knn
})


X_holdout_final = pd.DataFrame({

    "anomaly_pca":
        holdout_pca,

    "anomaly_knn":
        holdout_knn
})


y_dev = (
    dev[
        "CycleDefect"
    ]
    .astype(int)
)


y_holdout = (
    holdout[
        "CycleDefect"
    ]
    .astype(int)
)


# ============================================================
# 16. FIXED ExtraTrees
# ============================================================

model = (
    ExtraTreesClassifier(
        n_estimators=500,
        max_depth=6,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
)


model.fit(
    X_dev_final,
    y_dev
)


holdout_prob = (
    model
    .predict_proba(
        X_holdout_final
    )[:, 1]
)


# ============================================================
# 17. Holdout Ranking Metrics
# ============================================================

baseline = (
    y_holdout.mean()
)


pr_auc = (
    average_precision_score(
        y_holdout,
        holdout_prob
    )
)


roc_auc = (
    roc_auc_score(
        y_holdout,
        holdout_prob
    )
)


print(
    "\n\n"
    + "=" * 100
)

print(
    "FINAL TEMPORAL HOLDOUT PERFORMANCE"
)

print(
    "=" * 100
)


print(
    f"N              : {len(holdout)}"
)

print(
    f"Defects        : {int(y_holdout.sum())}"
)

print(
    f"Baseline PR    : {baseline:.6f}"
)

print(
    f"PR-AUC         : {pr_auc:.6f}"
)

print(
    f"PR Ratio       : {pr_auc / baseline:.3f}"
)

print(
    f"ROC-AUC        : {roc_auc:.6f}"
)


# ============================================================
# 18. 고정 운영정책: Top 30%
#
# Holdout 결과를 보기 전에 고정한 정책
# ============================================================

TOP_FRACTION = 0.30


result = pd.DataFrame({

    "TimeStamp":
        holdout[
            "TimeStamp"
        ],

    "y_true":
        y_holdout,

    "PCA_Error":
        holdout_pca,

    "KNN_Distance":
        holdout_knn,

    "Risk_Probability":
        holdout_prob
})


result = (
    result
    .sort_values(
        "Risk_Probability",
        ascending=False
    )
    .reset_index(drop=True)
)


n_inspect = int(
    np.ceil(
        len(result)
        *
        TOP_FRACTION
    )
)


result[
    "Top30_Inspect"
] = 0


result.loc[
    :n_inspect - 1,
    "Top30_Inspect"
] = 1


inspected = (
    result[
        result[
            "Top30_Inspect"
        ]
        ==
        1
    ]
)


captured = int(
    inspected[
        "y_true"
    ]
    .sum()
)


total_defects = int(
    result[
        "y_true"
    ]
    .sum()
)


top30_recall = (
    captured
    /
    total_defects
)


top30_precision = (
    captured
    /
    len(inspected)
)


top30_lift = (
    top30_precision
    /
    baseline
)


print(
    "\n"
    + "=" * 100
)

print(
    "PRE-SPECIFIED TOP30 POLICY"
)

print(
    "=" * 100
)


print(
    f"Inspection N    : "
    f"{len(inspected)} / {len(result)}"
)

print(
    f"Inspection Rate : "
    f"{len(inspected)/len(result)*100:.2f}%"
)

print(
    f"Captured Defects: "
    f"{captured} / {total_defects}"
)

print(
    f"Recall          : "
    f"{top30_recall:.4f}"
)

print(
    f"Precision       : "
    f"{top30_precision:.4f}"
)

print(
    f"Precision Lift  : "
    f"{top30_lift:.3f}x"
)


# ============================================================
# 19. Top-K sensitivity
#
# 참고용
# 최종 정책 선택용이 아님
# ============================================================

top_rows = []


for fraction in [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]:

    n = int(
        np.ceil(
            len(result)
            *
            fraction
        )
    )

    subset = (
        result.iloc[
            :n
        ]
    )

    tp = int(
        subset[
            "y_true"
        ].sum()
    )

    recall = (
        tp
        /
        total_defects
    )

    precision = (
        tp
        /
        n
    )

    lift = (
        precision
        /
        baseline
    )


    top_rows.append({

        "Top_%":
            fraction * 100,

        "Inspection_N":
            n,

        "Captured_Defects":
            tp,

        "Recall":
            recall,

        "Precision":
            precision,

        "Precision_Lift":
            lift
    })


top_table = (
    pd.DataFrame(
        top_rows
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "HOLDOUT TOP-K SENSITIVITY"
)

print(
    "=" * 100
)


print(
    top_table
    .to_string(
        index=False
    )
)


# ============================================================
# 20. Holdout Cycle 상세
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "HOLDOUT RISK RANKING"
)

print(
    "=" * 100
)


print(
    result[
        [
            "TimeStamp",
            "y_true",
            "PCA_Error",
            "KNN_Distance",
            "Risk_Probability",
            "Top30_Inspect"
        ]
    ]
    .to_string(
        index=False
    )
)

# 2단계: 11/4 Distribution Shift 원인 감사

In [ ]:
# ============================================================
# RG3 11/04 DISTRIBUTION SHIFT AUDIT
#
# 목적
# 1. Dev vs Holdout 센서 분포 비교
# 2. 어떤 센서가 11/4 OOD를 만든 것인지 확인
# 3. 설비 / 품번 / 생산조건 변경 확인
# ============================================================

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


# ============================================================
# 1. 기본 확인
# ============================================================

print("=" * 100)
print("DATA")
print("=" * 100)

print(
    "DEV:",
    len(dev),
    "|",
    dev["TimeStamp"].min(),
    "~",
    dev["TimeStamp"].max()
)

print(
    "HOLDOUT:",
    len(holdout),
    "|",
    holdout["TimeStamp"].min(),
    "~",
    holdout["TimeStamp"].max()
)


# ============================================================
# 2. Dev 기준 StandardScaler
# ============================================================

scaler = StandardScaler()

X_dev_std = scaler.fit_transform(
    dev[raw_features]
)

X_hold_std = scaler.transform(
    holdout[raw_features]
)


# ============================================================
# 3. 센서별 분포
# ============================================================

rows = []

for i, col in enumerate(raw_features):

    dev_values = (
        dev[col]
        .astype(float)
    )

    hold_values = (
        holdout[col]
        .astype(float)
    )


    dev_mean = (
        dev_values.mean()
    )

    dev_std = (
        dev_values.std()
    )

    hold_mean = (
        hold_values.mean()
    )

    hold_std = (
        hold_values.std()
    )


    # Holdout 평균이 Dev 평균에서
    # 몇 표준편차 떨어져 있는지
    if dev_std > 0:

        mean_shift_z = (
            abs(
                hold_mean
                -
                dev_mean
            )
            /
            dev_std
        )

    else:

        mean_shift_z = np.nan


    # Holdout standardized 절대값
    hold_abs_z_mean = (
        np.abs(
            X_hold_std[:, i]
        )
        .mean()
    )

    hold_abs_z_max = (
        np.abs(
            X_hold_std[:, i]
        )
        .max()
    )


    rows.append({

        "Feature":
            col,

        "Dev_Mean":
            dev_mean,

        "Dev_Std":
            dev_std,

        "Dev_Min":
            dev_values.min(),

        "Dev_Max":
            dev_values.max(),

        "Holdout_Mean":
            hold_mean,

        "Holdout_Std":
            hold_std,

        "Holdout_Min":
            hold_values.min(),

        "Holdout_Max":
            hold_values.max(),

        "Mean_Shift_Z":
            mean_shift_z,

        "Holdout_AbsZ_Mean":
            hold_abs_z_mean,

        "Holdout_AbsZ_Max":
            hold_abs_z_max
    })


shift_df = (
    pd.DataFrame(rows)
    .sort_values(
        "Mean_Shift_Z",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "DEV vs HOLDOUT SENSOR SHIFT"
)

print(
    "=" * 100
)


print(
    shift_df
    .to_string(
        index=False
    )
)


# ============================================================
# 4. 가장 심한 10개 Sensor
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "TOP 10 SHIFTED FEATURES"
)

print(
    "=" * 100
)


print(
    shift_df[
        [
            "Feature",
            "Dev_Mean",
            "Dev_Std",
            "Holdout_Mean",
            "Holdout_Std",
            "Mean_Shift_Z",
            "Holdout_AbsZ_Max"
        ]
    ]
    .head(10)
    .to_string(
        index=False
    )
)


# ============================================================
# 5. Holdout Cycle별 standardized magnitude
# ============================================================

holdout_z = pd.DataFrame(
    X_hold_std,
    columns=raw_features
)


cycle_shift = pd.DataFrame({

    "TimeStamp":
        holdout["TimeStamp"].values,

    "y_true":
        holdout[
            "CycleDefect"
        ].values,

    "Mean_Abs_Z":
        np.abs(
            X_hold_std
        ).mean(
            axis=1
        ),

    "Max_Abs_Z":
        np.abs(
            X_hold_std
        ).max(
            axis=1
        )
})


cycle_shift[
    "Worst_Feature"
] = [

    raw_features[i]

    for i in np.argmax(
        np.abs(
            X_hold_std
        ),
        axis=1
    )
]


print(
    "\n"
    + "=" * 100
)

print(
    "HOLDOUT CYCLE OOD LEVEL"
)

print(
    "=" * 100
)


print(
    cycle_shift
    .sort_values(
        "Mean_Abs_Z",
        ascending=False
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 6. Metadata 비교
# ============================================================

metadata_cols = [
    "EQUIP_CD",
    "EQUIP_NAME",
    "PART_NO",
    "PART_FACT_SERIAL",
    "PART_FACT_PLAN_DATE"
]


metadata_cols = [
    c
    for c in metadata_cols
    if c in rg3.columns
]


dev_times = set(
    dev["TimeStamp"]
)

holdout_times = set(
    holdout["TimeStamp"]
)


dev_meta = (
    rg3[
        rg3["TimeStamp"]
        .isin(
            dev_times
        )
    ]
)


hold_meta = (
    rg3[
        rg3["TimeStamp"]
        .isin(
            holdout_times
        )
    ]
)


print(
    "\n"
    + "=" * 100
)

print(
    "METADATA COMPARISON"
)

print(
    "=" * 100
)


for col in metadata_cols:

    print(
        f"\n[{col}]"
    )

    print(
        "DEV:"
    )

    print(
        dev_meta[col]
        .value_counts(
            dropna=False
        )
        .head(20)
        .to_string()
    )

    print(
        "\nHOLDOUT:"
    )

    print(
        hold_meta[col]
        .value_counts(
            dropna=False
        )
        .head(20)
        .to_string()
    )


# ============================================================
# 7. Holdout 정상 vs 불량 센서
#
# 11/4 내부에서는 불량 신호가 실제 존재하는지
# 참고용 EDA
# ============================================================

holdout_compare = []


for col in raw_features:

    normal_vals = (
        holdout.loc[
            holdout[
                "CycleDefect"
            ]
            ==
            0,
            col
        ]
    )

    defect_vals = (
        holdout.loc[
            holdout[
                "CycleDefect"
            ]
            ==
            1,
            col
        ]
    )


    holdout_compare.append({

        "Feature":
            col,

        "Normal_Mean":
            normal_vals.mean(),

        "Defect_Mean":
            defect_vals.mean(),

        "Abs_Mean_Diff":
            abs(
                normal_vals.mean()
                -
                defect_vals.mean()
            ),

        "Normal_Std":
            normal_vals.std(),

        "Defect_Std":
            defect_vals.std()
    })


holdout_compare_df = (
    pd.DataFrame(
        holdout_compare
    )
    .sort_values(
        "Abs_Mean_Diff",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "11/04 NORMAL vs DEFECT RAW SENSOR DIFFERENCE"
)

print(
    "=" * 100
)


print(
    holdout_compare_df
    .head(15)
    .to_string(
        index=False
    )
)

In [ ]:
# ============================================================
# Average_Screw_RPM SCALE BREAK AUDIT
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. RG3 전체 날짜별 분포
# ============================================================

audit = rg3.copy()

audit["date"] = (
    audit["TimeStamp"]
    .dt.date
)


audit["RPM_Ratio"] = (
    audit["Average_Screw_RPM"]
    /
    audit["Max_Screw_RPM"]
)


date_rpm = (
    audit
    .groupby("date")
    .agg(
        N=("Average_Screw_RPM", "size"),

        AvgRPM_Mean=(
            "Average_Screw_RPM",
            "mean"
        ),

        AvgRPM_Min=(
            "Average_Screw_RPM",
            "min"
        ),

        AvgRPM_Max=(
            "Average_Screw_RPM",
            "max"
        ),

        MaxRPM_Mean=(
            "Max_Screw_RPM",
            "mean"
        ),

        Ratio_Mean=(
            "RPM_Ratio",
            "mean"
        ),

        Ratio_Min=(
            "RPM_Ratio",
            "min"
        ),

        Ratio_Max=(
            "RPM_Ratio",
            "max"
        )
    )
)


print("=" * 100)
print("LABELED DATE RPM AUDIT")
print("=" * 100)

print(
    date_rpm.to_string()
)


# ============================================================
# 2. 실제 unique 값 확인
# ============================================================

print(
    "\n"
    + "=" * 100
)

print("LABELED RPM VALUES BY DATE")

print("=" * 100)


for date, group in audit.groupby("date"):

    print(
        f"\n[{date}]"
    )

    print(
        "Average_Screw_RPM:",
        sorted(
            group[
                "Average_Screw_RPM"
            ]
            .dropna()
            .unique()
        )[:30]
    )

    print(
        "Max_Screw_RPM:",
        sorted(
            group[
                "Max_Screw_RPM"
            ]
            .dropna()
            .unique()
        )[:30]
    )


# ============================================================
# 3. Unlabeled RG3 RH 전체 감사
# ============================================================

unlab_rg3 = (
    unlabeled[
        unlabeled["PART_NAME"]
        ==
        TARGET_RH
    ]
    .copy()
)


unlab_rg3["date"] = (
    unlab_rg3[
        "TimeStamp"
    ]
    .dt.date
)


unlab_rg3["RPM_Ratio"] = (
    unlab_rg3[
        "Average_Screw_RPM"
    ]
    /
    unlab_rg3[
        "Max_Screw_RPM"
    ]
)


unlab_rpm = (
    unlab_rg3
    .groupby("date")
    .agg(
        N=(
            "Average_Screw_RPM",
            "size"
        ),

        AvgRPM_Mean=(
            "Average_Screw_RPM",
            "mean"
        ),

        AvgRPM_Min=(
            "Average_Screw_RPM",
            "min"
        ),

        AvgRPM_Max=(
            "Average_Screw_RPM",
            "max"
        ),

        MaxRPM_Mean=(
            "Max_Screw_RPM",
            "mean"
        ),

        Ratio_Mean=(
            "RPM_Ratio",
            "mean"
        )
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "UNLABELED RPM AUDIT - LAST 20 DATES"
)

print(
    "=" * 100
)


print(
    unlab_rpm
    .tail(20)
    .to_string()
)


# ============================================================
# 4. 값 크기별 개수
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "LABELED SCALE GROUP"
)

print(
    "=" * 100
)


print(
    pd.cut(
        audit[
            "Average_Screw_RPM"
        ],
        bins=[
            -np.inf,
            50,
            100,
            np.inf
        ],
        labels=[
            "<50",
            "50~100",
            ">100"
        ]
    )
    .value_counts()
)


print(
    "\n"
    + "=" * 100
)

print(
    "UNLABELED SCALE GROUP"
)

print(
    "=" * 100
)


print(
    pd.cut(
        unlab_rg3[
            "Average_Screw_RPM"
        ],
        bins=[
            -np.inf,
            50,
            100,
            np.inf
        ],
        labels=[
            "<50",
            "50~100",
            ">100"
        ]
    )
    .value_counts()
)

In [ ]:
# ============================================================
# RG3 DATA-QUALITY REMEDIATION VALIDATION
#
# 변경점:
# - Average_Screw_RPM 제거
#
# 고정:
# - Q95 Local Reference
# - PCA + KNN
# - ExtraTrees
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)


# ============================================================
# 1. Feature 재정의
# ============================================================

sensor_cols_fixed = [
    c
    for c in sensor_cols
    if c != "Average_Screw_RPM"
]


raw_features_fixed = [
    c
    for c in sensor_cols_fixed
    if dev[c].nunique(dropna=False) > 1
]


print("=" * 100)
print("FIXED FEATURES")
print("=" * 100)

print("Feature N:", len(raw_features_fixed))

print(
    "Removed:",
    [
        c
        for c in sensor_cols
        if c not in raw_features_fixed
    ]
)


# ============================================================
# 2. Development Walk-forward Session 재생성
# ============================================================

dev2 = (
    dev
    .copy()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


dev2["gap_min"] = (
    dev2["TimeStamp"]
    .diff()
    .dt.total_seconds()
    / 60
)


dev2["session_id"] = (
    (dev2["gap_min"] > 5)
    .cumsum()
)


# ============================================================
# 3. Unlabeled 정리
# ============================================================

unlab_fixed = (
    unlabeled[
        unlabeled["PART_NAME"] == TARGET_RH
    ]
    .copy()
)


all_labeled_times = set(
    cycle_df["TimeStamp"]
)


unlab_fixed = (
    unlab_fixed[
        ~unlab_fixed["TimeStamp"]
        .isin(all_labeled_times)
    ]
    .copy()
)


unlab_fixed = (
    unlab_fixed
    .groupby(
        "TimeStamp",
        as_index=False
    )[raw_features_fixed]
    .mean()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


print(
    "Clean unlabeled:",
    len(unlab_fixed)
)


# ============================================================
# 4. 공통 함수
# ============================================================

def build_reference(
    train,
    candidates,
    features
):

    train_normal = (
        train[
            train["CycleDefect"] == 0
        ]
        .copy()
    )


    selector_scaler = (
        StandardScaler()
    )


    X_normal = (
        selector_scaler
        .fit_transform(
            train_normal[
                features
            ]
        )
    )


    X_candidate = (
        selector_scaler
        .transform(
            candidates[
                features
            ]
        )
    )


    k = 5


    # 정상 내부 거리
    nn_normal = (
        NearestNeighbors(
            n_neighbors=k + 1
        )
    )

    nn_normal.fit(
        X_normal
    )


    dist_normal, _ = (
        nn_normal
        .kneighbors(
            X_normal
        )
    )


    q95 = np.quantile(
        dist_normal[:, k],
        0.95
    )


    # candidate -> normal
    nn_select = (
        NearestNeighbors(
            n_neighbors=k
        )
    )

    nn_select.fit(
        X_normal
    )


    dist_candidate, _ = (
        nn_select
        .kneighbors(
            X_candidate
        )
    )


    mask = (
        dist_candidate[:, -1]
        <=
        q95
    )


    reference = (
        candidates
        .loc[mask]
        .copy()
    )


    return (
        reference,
        q95
    )


def make_anomaly_features(
    train,
    valid,
    reference,
    features
):

    scaler = (
        StandardScaler()
    )


    X_ref = (
        scaler
        .fit_transform(
            reference[
                features
            ]
        )
    )


    X_train = (
        scaler
        .transform(
            train[
                features
            ]
        )
    )


    X_valid = (
        scaler
        .transform(
            valid[
                features
            ]
        )
    )


    # PCA
    pca = (
        PCA(
            n_components=0.95,
            random_state=42
        )
    )

    pca.fit(
        X_ref
    )


    train_recon = (
        pca.inverse_transform(
            pca.transform(
                X_train
            )
        )
    )


    valid_recon = (
        pca.inverse_transform(
            pca.transform(
                X_valid
            )
        )
    )


    train_pca = (
        np.mean(
            (
                X_train
                -
                train_recon
            ) ** 2,
            axis=1
        )
    )


    valid_pca = (
        np.mean(
            (
                X_valid
                -
                valid_recon
            ) ** 2,
            axis=1
        )
    )


    # KNN
    knn = (
        NearestNeighbors(
            n_neighbors=10
        )
    )

    knn.fit(
        X_ref
    )


    train_knn = (
        knn
        .kneighbors(
            X_train
        )[0]
        .mean(axis=1)
    )


    valid_knn = (
        knn
        .kneighbors(
            X_valid
        )[0]
        .mean(axis=1)
    )


    X_train_final = pd.DataFrame({
        "anomaly_pca":
            train_pca,

        "anomaly_knn":
            train_knn
    })


    X_valid_final = pd.DataFrame({
        "anomaly_pca":
            valid_pca,

        "anomaly_knn":
            valid_knn
    })


    return (
        X_train_final,
        X_valid_final,
        scaler,
        pca,
        knn
    )


def make_model():

    return ExtraTreesClassifier(
        n_estimators=500,
        max_depth=6,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )


# ============================================================
# 5. Development Walk-forward OOF
# ============================================================

oof_parts = []
fold_rows = []


for valid_session in range(2, 7):

    train = (
        dev2[
            dev2["session_id"]
            <
            valid_session
        ]
        .copy()
    )


    valid = (
        dev2[
            dev2["session_id"]
            ==
            valid_session
        ]
        .copy()
    )


    valid_start = (
        valid["TimeStamp"]
        .min()
    )


    candidates = (
        unlab_fixed[
            unlab_fixed["TimeStamp"]
            <
            valid_start
        ]
        .copy()
    )


    reference, q95 = (
        build_reference(
            train,
            candidates,
            raw_features_fixed
        )
    )


    (
        X_train_final,
        X_valid_final,
        _,
        _,
        _
    ) = make_anomaly_features(
        train,
        valid,
        reference,
        raw_features_fixed
    )


    y_train = (
        train["CycleDefect"]
        .astype(int)
    )


    y_valid = (
        valid["CycleDefect"]
        .astype(int)
    )


    model = make_model()

    model.fit(
        X_train_final,
        y_train
    )


    prob = (
        model
        .predict_proba(
            X_valid_final
        )[:, 1]
    )


    pr = (
        average_precision_score(
            y_valid,
            prob
        )
    )


    baseline = (
        y_valid.mean()
    )


    fold_rows.append({

        "Session":
            valid_session,

        "N":
            len(valid),

        "Defects":
            int(
                y_valid.sum()
            ),

        "Reference_N":
            len(reference),

        "Q95":
            q95,

        "Baseline_PR":
            baseline,

        "PR_AUC":
            pr,

        "PR_Ratio":
            pr / baseline
    })


    oof_parts.append(
        pd.DataFrame({

            "TimeStamp":
                valid[
                    "TimeStamp"
                ].values,

            "session_id":
                valid_session,

            "y_true":
                y_valid.values,

            "y_prob":
                prob
        })
    )


fold_fixed = pd.DataFrame(
    fold_rows
)


oof_fixed = pd.concat(
    oof_parts,
    ignore_index=True
)


dev_baseline = (
    oof_fixed[
        "y_true"
    ]
    .mean()
)


dev_pr = (
    average_precision_score(
        oof_fixed[
            "y_true"
        ],
        oof_fixed[
            "y_prob"
        ]
    )
)


dev_roc = (
    roc_auc_score(
        oof_fixed[
            "y_true"
        ],
        oof_fixed[
            "y_prob"
        ]
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "22-FEATURE DEVELOPMENT OOF"
)

print(
    "=" * 100
)


print(
    fold_fixed.to_string(
        index=False
    )
)


print(
    "\nOOF N:",
    len(oof_fixed)
)

print(
    "OOF Defects:",
    int(
        oof_fixed[
            "y_true"
        ].sum()
    )
)

print(
    "Baseline:",
    dev_baseline
)

print(
    "PR-AUC:",
    dev_pr
)

print(
    "PR Ratio:",
    dev_pr
    /
    dev_baseline
)

print(
    "ROC-AUC:",
    dev_roc
)


# ============================================================
# 6. Final Development 전체로 Reference 재구축
# ============================================================

holdout_start = (
    holdout[
        "TimeStamp"
    ]
    .min()
)


candidates_final = (
    unlab_fixed[
        unlab_fixed[
            "TimeStamp"
        ]
        <
        holdout_start
    ]
    .copy()
)


reference_final, q95_final = (
    build_reference(
        dev2,
        candidates_final,
        raw_features_fixed
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "FINAL REFERENCE"
)

print(
    "=" * 100
)


print(
    "Q95:",
    q95_final
)

print(
    "Reference N:",
    len(reference_final)
)


# ============================================================
# 7. Final model
# ============================================================

(
    X_dev_final,
    X_hold_final,
    scaler_final,
    pca_final,
    knn_final
) = make_anomaly_features(
    dev2,
    holdout,
    reference_final,
    raw_features_fixed
)


final_model = make_model()


final_model.fit(
    X_dev_final,
    dev2[
        "CycleDefect"
    ].astype(int)
)


hold_prob = (
    final_model
    .predict_proba(
        X_hold_final
    )[:, 1]
)


y_hold = (
    holdout[
        "CycleDefect"
    ]
    .astype(int)
)


hold_baseline = (
    y_hold.mean()
)


hold_pr = (
    average_precision_score(
        y_hold,
        hold_prob
    )
)


hold_roc = (
    roc_auc_score(
        y_hold,
        hold_prob
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "22-FEATURE OUT-OF-TIME VALIDATION"
)

print(
    "=" * 100
)


print(
    "N:",
    len(holdout)
)

print(
    "Defects:",
    int(
        y_hold.sum()
    )
)

print(
    "Baseline PR:",
    hold_baseline
)

print(
    "PR-AUC:",
    hold_pr
)

print(
    "PR Ratio:",
    hold_pr
    /
    hold_baseline
)

print(
    "ROC-AUC:",
    hold_roc
)


# ============================================================
# 8. Probability 분포
# ============================================================

result_fixed = pd.DataFrame({

    "TimeStamp":
        holdout[
            "TimeStamp"
        ].values,

    "y_true":
        y_hold.values,

    "PCA_Error":
        X_hold_final[
            "anomaly_pca"
        ].values,

    "KNN_Distance":
        X_hold_final[
            "anomaly_knn"
        ].values,

    "Risk_Probability":
        hold_prob
})


result_fixed = (
    result_fixed
    .sort_values(
        "Risk_Probability",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "\n"
    + "=" * 100
)

print(
    "22-FEATURE HOLDOUT RANKING"
)

print(
    "=" * 100
)


print(
    result_fixed.to_string(
        index=False
    )
)


# ============================================================
# 9. Top-K 참고
# ============================================================

rows = []


for fraction in [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]:

    n = int(
        np.ceil(
            len(
                result_fixed
            )
            *
            fraction
        )
    )


    subset = (
        result_fixed
        .iloc[
            :n
        ]
    )


    captured = int(
        subset[
            "y_true"
        ]
        .sum()
    )


    precision = (
        captured
        /
        n
    )


    recall = (
        captured
        /
        y_hold.sum()
    )


    rows.append({

        "Top_%":
            fraction * 100,

        "Inspection_N":
            n,

        "Captured_Defects":
            captured,

        "Recall":
            recall,

        "Precision":
            precision,

        "Lift":
            precision
            /
            hold_baseline
    })


topk_fixed = pd.DataFrame(
    rows
)


print(
    "\n"
    + "=" * 100
)

print(
    "22-FEATURE HOLDOUT TOP-K"
)

print(
    "=" * 100
)


print(
    topk_fixed.to_string(
        index=False
    )
)

In [ ]:
# ============================================================
# RG3 DEFECT TRANSITION / TEMPORAL PATTERN AUDIT
#
# 목적
# 1. 불량이 연속 구간(run)으로 발생하는지 확인
# 2. 불량 시작 직전 ±5 Cycle 패턴 확인
# 3. 정상 -> 불량 전환 시 어떤 센서가 변하는지 확인
# 4. 10월과 11월에서 공통 변화 Feature가 존재하는지 확인
#
# 모델링 X
# 순수 EDA / 오류원인 분석
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. 사용할 Sensor
# ============================================================

TEMPORAL_FEATURES = [
    'Injection_Time',
    'Filling_Time',
    'Plasticizing_Time',
    'Cycle_Time',
    'Clamp_Close_Time',
    'Cushion_Position',
    'Plasticizing_Position',
    'Max_Injection_Speed',
    'Max_Screw_RPM',

    # 기간간 스케일 불일치 확인 -> 제외
    # 'Average_Screw_RPM',

    'Max_Injection_Pressure',
    'Max_Switch_Over_Pressure',
    'Max_Back_Pressure',
    'Average_Back_Pressure',
    'Barrel_Temperature_1',
    'Barrel_Temperature_2',
    'Barrel_Temperature_3',
    'Barrel_Temperature_4',
    'Barrel_Temperature_5',
    'Barrel_Temperature_6',
    'Hopper_Temperature',
    'Mold_Temperature_3',
    'Mold_Temperature_4'
]


# ============================================================
# 2. Cycle-level 데이터 재구성
# ============================================================

cycle_size = (
    rg3
    .groupby("TimeStamp")
    .size()
)


cycle_sensor = (
    rg3
    .groupby("TimeStamp")[
        TEMPORAL_FEATURES
    ]
    .first()
)


cycle_target = (
    rg3
    .groupby("TimeStamp")[
        "Target"
    ]
    .max()
)


cycle = (
    cycle_sensor
    .join(
        cycle_target.rename(
            "CycleDefect"
        )
    )
    .reset_index()
)


cycle["Rows"] = (
    cycle["TimeStamp"]
    .map(cycle_size)
)


cycle = (
    cycle[
        cycle["Rows"] == 2
    ]
    .copy()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


cycle["date"] = (
    cycle["TimeStamp"]
    .dt.date
)


# ============================================================
# 3. Session 정의
#
# 5분 이상 생산 중단 시 새로운 Session
# ============================================================

cycle["gap_min"] = (
    cycle["TimeStamp"]
    .diff()
    .dt.total_seconds()
    / 60
)


cycle["session_id"] = (
    (
        (cycle["gap_min"] > 5)
        |
        cycle["gap_min"].isna()
    )
    .cumsum()
)


cycle["cycle_in_session"] = (
    cycle
    .groupby("session_id")
    .cumcount()
)


print("=" * 100)
print("CYCLE DATA")
print("=" * 100)

print(
    "Cycles:",
    len(cycle)
)

print(
    "Defects:",
    int(
        cycle["CycleDefect"].sum()
    )
)

print(
    "Sessions:",
    cycle["session_id"].nunique()
)


# ============================================================
# 4. 날짜별 기본 구조
# ============================================================

date_summary = (
    cycle
    .groupby("date")
    .agg(
        Cycles=(
            "CycleDefect",
            "size"
        ),
        Defects=(
            "CycleDefect",
            "sum"
        ),
        Sessions=(
            "session_id",
            "nunique"
        )
    )
)


print(
    "\n"
    + "=" * 100
)

print("DATE SUMMARY")

print("=" * 100)

print(
    date_summary.to_string()
)


# ============================================================
# 5. Defect Run 생성
#
# 연속된 불량 Cycle 묶기
# ============================================================

cycle["prev_target"] = (
    cycle
    .groupby("session_id")[
        "CycleDefect"
    ]
    .shift(1)
)


cycle["defect_onset"] = (
    (cycle["CycleDefect"] == 1)
    &
    (
        (cycle["prev_target"] == 0)
        |
        cycle["prev_target"].isna()
    )
)


# run id
cycle["run_start"] = (
    cycle["defect_onset"]
    .astype(int)
)


cycle["defect_run_id"] = np.nan

run_id = 0

for session_id, group in (
    cycle
    .groupby("session_id")
):

    active_run = None

    for idx in group.index:

        if cycle.loc[idx, "CycleDefect"] == 1:

            if (
                active_run is None
                or
                cycle.loc[idx, "defect_onset"]
            ):

                run_id += 1
                active_run = run_id

            cycle.loc[
                idx,
                "defect_run_id"
            ] = active_run

        else:
            active_run = None


# ============================================================
# 6. Defect Run Summary
# ============================================================

defect_rows = (
    cycle[
        cycle["CycleDefect"] == 1
    ]
    .copy()
)


run_summary = (
    defect_rows
    .groupby("defect_run_id")
    .agg(
        Date=(
            "date",
            "first"
        ),
        Session=(
            "session_id",
            "first"
        ),
        Start=(
            "TimeStamp",
            "min"
        ),
        End=(
            "TimeStamp",
            "max"
        ),
        Run_Length=(
            "CycleDefect",
            "size"
        )
    )
    .reset_index()
)


print(
    "\n"
    + "=" * 100
)

print("DEFECT RUN SUMMARY")

print("=" * 100)

print(
    run_summary
    .to_string(
        index=False
    )
)


# ============================================================
# 7. Run length 분포
# ============================================================

print(
    "\n"
    + "=" * 100
)

print("DEFECT RUN LENGTH DISTRIBUTION")

print("=" * 100)

print(
    run_summary[
        "Run_Length"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 8. Sensor 1-step 변화량
# ============================================================

for col in TEMPORAL_FEATURES:

    cycle[
        f"{col}__diff1"
    ] = (
        cycle
        .groupby("session_id")[col]
        .diff()
    )


# ============================================================
# 9. 최근 3 Cycle 평균 대비 변화
# ============================================================

for col in TEMPORAL_FEATURES:

    prev3_mean = (
        cycle
        .groupby("session_id")[col]
        .transform(
            lambda s:
            s.shift(1)
            .rolling(
                3,
                min_periods=1
            )
            .mean()
        )
    )


    cycle[
        f"{col}__dev3"
    ] = (
        cycle[col]
        -
        prev3_mean
    )


# ============================================================
# 10. 정상 전환 기준 Scale 생성
#
# 절대 센서 level이 아니라
# "변화량 자체"를 비교하기 위한 scale
#
# 개발기간 / 11월을 따로 보지 않고
# 정상->정상 transition의 변동폭으로 정규화
# ============================================================

normal_transition = (
    (cycle["CycleDefect"] == 0)
    &
    (cycle["prev_target"] == 0)
)


scale_rows = []


for col in TEMPORAL_FEATURES:

    diff_col = (
        f"{col}__diff1"
    )

    vals = (
        cycle.loc[
            normal_transition,
            diff_col
        ]
        .dropna()
    )


    # Median Absolute Deviation
    med = vals.median()

    mad = (
        np.median(
            np.abs(
                vals
                -
                med
            )
        )
    )


    # MAD가 0이면 std 대체
    if (
        pd.isna(mad)
        or
        mad == 0
    ):

        scale = vals.std()

    else:

        scale = (
            1.4826
            *
            mad
        )


    if (
        pd.isna(scale)
        or
        scale == 0
    ):

        scale = 1.0


    cycle[
        f"{col}__diff_z"
    ] = (
        cycle[
            diff_col
        ]
        /
        scale
    )


    scale_rows.append({

        "Feature":
            col,

        "Normal_Diff_Scale":
            scale
    })


scale_df = pd.DataFrame(
    scale_rows
)


# ============================================================
# 11. Defect Onset만 추출
# ============================================================

onset = (
    cycle[
        cycle["defect_onset"]
    ]
    .copy()
)


print(
    "\n"
    + "=" * 100
)

print("DEFECT ONSET EVENTS")

print("=" * 100)


print(
    onset[
        [
            "TimeStamp",
            "date",
            "session_id",
            "cycle_in_session"
        ]
    ]
    .to_string(
        index=False
    )
)


print(
    "\nOnset count:",
    len(onset)
)


# ============================================================
# 12. 각 Onset에서 변화량 TOP 5
# ============================================================

onset_feature_rows = []


for idx, row in onset.iterrows():

    scores = []

    for col in TEMPORAL_FEATURES:

        value = row[
            f"{col}__diff_z"
        ]

        if pd.isna(value):
            continue

        scores.append(
            (
                col,
                value,
                abs(value)
            )
        )


    scores = sorted(
        scores,
        key=lambda x:
            x[2],
        reverse=True
    )


    record = {

        "TimeStamp":
            row["TimeStamp"],

        "date":
            row["date"],

        "session_id":
            row["session_id"]
    }


    for rank, (
        feature,
        signed_z,
        abs_z
    ) in enumerate(
        scores[:5],
        start=1
    ):

        record[
            f"Top{rank}_Feature"
        ] = feature

        record[
            f"Top{rank}_SignedChange"
        ] = signed_z

        record[
            f"Top{rank}_AbsChange"
        ] = abs_z


    onset_feature_rows.append(
        record
    )


onset_top = pd.DataFrame(
    onset_feature_rows
)


print(
    "\n"
    + "=" * 100
)

print("DEFECT ONSET TOP CHANGES")

print("=" * 100)


print(
    onset_top
    .to_string(
        index=False
    )
)


# ============================================================
# 13. Onset Top Feature 출현 빈도
# ============================================================

top_cols = [
    f"Top{i}_Feature"
    for i in range(1, 6)
]


feature_frequency = (
    onset_top[
        top_cols
    ]
    .stack()
    .value_counts()
)


print(
    "\n"
    + "=" * 100
)

print("ALL ONSET FEATURE FREQUENCY")

print("=" * 100)


print(
    feature_frequency
    .head(20)
)


# ============================================================
# 14. 10월 vs 11월 분리
# ============================================================

onset_top[
    "Period"
] = np.where(
    pd.to_datetime(
        onset_top[
            "TimeStamp"
        ]
    )
    <
    pd.Timestamp(
        "2020-11-01"
    ),
    "OCT",
    "NOV"
)


for period in [
    "OCT",
    "NOV"
]:

    sub = (
        onset_top[
            onset_top["Period"]
            ==
            period
        ]
    )


    freq = (
        sub[
            top_cols
        ]
        .stack()
        .value_counts()
    )


    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{period} ONSET FEATURE FREQUENCY"
    )

    print("=" * 100)


    print(
        freq
        .head(20)
    )


# ============================================================
# 15. October / November 공통 Feature
# ============================================================

oct_freq = (
    onset_top[
        onset_top["Period"] == "OCT"
    ][top_cols]
    .stack()
    .value_counts()
)


nov_freq = (
    onset_top[
        onset_top["Period"] == "NOV"
    ][top_cols]
    .stack()
    .value_counts()
)


common_features = (
    pd.DataFrame({
        "OCT_Count":
            oct_freq,

        "NOV_Count":
            nov_freq
    })
    .fillna(0)
)


common_features[
    "Total"
] = (
    common_features[
        "OCT_Count"
    ]
    +
    common_features[
        "NOV_Count"
    ]
)


common_features = (
    common_features
    .sort_values(
        [
            "NOV_Count",
            "OCT_Count",
            "Total"
        ],
        ascending=False
    )
)


print(
    "\n"
    + "=" * 100
)

print("OCT / NOV COMMON ONSET FEATURES")

print("=" * 100)


print(
    common_features
    .head(20)
    .to_string()
)


# ============================================================
# 16. 정상 -> 정상 vs 정상 -> 불량 변화 비교
# ============================================================

comparison_rows = []


normal_to_normal = (
    (cycle["prev_target"] == 0)
    &
    (cycle["CycleDefect"] == 0)
)


normal_to_defect = (
    (cycle["prev_target"] == 0)
    &
    (cycle["CycleDefect"] == 1)
)


for col in TEMPORAL_FEATURES:

    diff_col = (
        f"{col}__diff_z"
    )


    nn = (
        cycle.loc[
            normal_to_normal,
            diff_col
        ]
        .dropna()
        .abs()
    )


    nd = (
        cycle.loc[
            normal_to_defect,
            diff_col
        ]
        .dropna()
        .abs()
    )


    comparison_rows.append({

        "Feature":
            col,

        "Normal_Normal_Median":
            nn.median(),

        "Normal_Defect_Median":
            nd.median(),

        "Normal_Normal_Mean":
            nn.mean(),

        "Normal_Defect_Mean":
            nd.mean(),

        "Median_Ratio":
            (
                nd.median()
                /
                nn.median()
                if nn.median() > 0
                else np.nan
            )
    })


transition_compare = (
    pd.DataFrame(
        comparison_rows
    )
    .sort_values(
        "Median_Ratio",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 100
)

print("NORMAL->NORMAL vs NORMAL->DEFECT")

print("=" * 100)


print(
    transition_compare
    .to_string(
        index=False
    )
)


# ============================================================
# 17. ±5 Cycle Window
# ============================================================

window_rows = []


for onset_idx in onset.index:

    session_id = (
        cycle.loc[
            onset_idx,
            "session_id"
        ]
    )


    session = (
        cycle[
            cycle[
                "session_id"
            ]
            ==
            session_id
        ]
        .copy()
    )


    position = (
        session
        .index
        .get_loc(
            onset_idx
        )
    )


    start = max(
        0,
        position - 5
    )

    end = min(
        len(session),
        position + 6
    )


    window = (
        session
        .iloc[
            start:end
        ]
        .copy()
    )


    window[
        "Relative_Cycle"
    ] = (
        np.arange(
            start,
            end
        )
        -
        position
    )


    for _, r in window.iterrows():

        record = {

            "Onset_Time":
                cycle.loc[
                    onset_idx,
                    "TimeStamp"
                ],

            "Onset_Date":
                cycle.loc[
                    onset_idx,
                    "date"
                ],

            "Relative_Cycle":
                r[
                    "Relative_Cycle"
                ],

            "TimeStamp":
                r[
                    "TimeStamp"
                ],

            "Target":
                r[
                    "CycleDefect"
                ]
        }


        for col in TEMPORAL_FEATURES:

            record[col] = r[col]


        window_rows.append(
            record
        )


window_df = pd.DataFrame(
    window_rows
)


# ============================================================
# 18. 상대 Cycle별 Target 비율
# ============================================================

window_target_summary = (
    window_df
    .groupby(
        "Relative_Cycle"
    )
    .agg(
        N=(
            "Target",
            "size"
        ),
        Defect_Rate=(
            "Target",
            "mean"
        )
    )
)


print(
    "\n"
    + "=" * 100
)

print("±5 CYCLE DEFECT PATTERN")

print("=" * 100)


print(
    window_target_summary
    .to_string()
)


# ============================================================
# 19. 불량 시작 전 5 Cycle 존재 여부
# ============================================================

pre_window_summary = []


for onset_idx in onset.index:

    session_id = (
        cycle.loc[
            onset_idx,
            "session_id"
        ]
    )


    session = (
        cycle[
            cycle[
                "session_id"
            ]
            ==
            session_id
        ]
    )


    pos = (
        session.index
        .get_loc(
            onset_idx
        )
    )


    pre = (
        session
        .iloc[
            max(
                0,
                pos - 5
            ):
            pos
        ]
    )


    pre_window_summary.append({

        "Onset":
            cycle.loc[
                onset_idx,
                "TimeStamp"
            ],

        "Date":
            cycle.loc[
                onset_idx,
                "date"
            ],

        "Pre_N":
            len(pre),

        "Pre_Defects":
            int(
                pre[
                    "CycleDefect"
                ]
                .sum()
            )
    })


pre_window_summary = (
    pd.DataFrame(
        pre_window_summary
    )
)


print(
    "\n"
    + "=" * 100
)

print("ONSET PREVIOUS 5 CYCLES")

print("=" * 100)


print(
    pre_window_summary
    .to_string(
        index=False
    )
)


# ============================================================
# 20. 저장
# ============================================================

OUTPUT_DIR = (
    r"../../.."
    r"\rg3_final"
    r"\temporal_audit"
)


import os

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


run_summary.to_csv(
    OUTPUT_DIR
    + r"\defect_run_summary.csv",
    index=False,
    encoding="utf-8-sig"
)


onset_top.to_csv(
    OUTPUT_DIR
    + r"\defect_onset_top_changes.csv",
    index=False,
    encoding="utf-8-sig"
)


common_features.to_csv(
    OUTPUT_DIR
    + r"\oct_nov_common_features.csv",
    encoding="utf-8-sig"
)


transition_compare.to_csv(
    OUTPUT_DIR
    + r"\transition_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)


window_df.to_csv(
    OUTPUT_DIR
    + r"\defect_onset_window_pm5.csv",
    index=False,
    encoding="utf-8-sig"
)


print(
    "\nSaved:",
    OUTPUT_DIR
)

In [ ]:
# ============================================================
# RG3 CHANGE-BASED MODEL
#
# 목적:
# 절대 센서 Level이 아닌
# "직전 생산 대비 변화"로 품질위험 학습
#
# Development : 2020-10-21 ~ 2020-10-23
# OOT          : 2020-11-04
#
# Features:
#   1) diff1
#   2) deviation from previous 3 cycles
#   3) multivariate change magnitude
#
# Model:
#   ExtraTrees (기존 설정 고정)
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)


# ============================================================
# 1. Feature
# ============================================================

CHANGE_SENSORS = [
    'Injection_Time',
    'Filling_Time',
    'Plasticizing_Time',
    'Cycle_Time',
    'Clamp_Close_Time',
    'Cushion_Position',
    'Plasticizing_Position',
    'Max_Injection_Speed',
    'Max_Screw_RPM',

    # 데이터 품질 문제로 제외
    # 'Average_Screw_RPM',

    'Max_Injection_Pressure',
    'Max_Switch_Over_Pressure',
    'Max_Back_Pressure',
    'Average_Back_Pressure',
    'Barrel_Temperature_1',
    'Barrel_Temperature_2',
    'Barrel_Temperature_3',
    'Barrel_Temperature_4',
    'Barrel_Temperature_5',
    'Barrel_Temperature_6',
    'Hopper_Temperature',
    'Mold_Temperature_3',
    'Mold_Temperature_4'
]


# ============================================================
# 2. Complete Cycle 데이터
# ============================================================

change_data = (
    cycle[
        cycle["Rows"] == 2
    ]
    .copy()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


# Session 다시 명확하게 생성
change_data["gap_min"] = (
    change_data["TimeStamp"]
    .diff()
    .dt.total_seconds()
    / 60
)


change_data["session_id"] = (
    (
        (change_data["gap_min"] > 5)
        |
        change_data["gap_min"].isna()
    )
    .cumsum()
)


change_data["cycle_in_session"] = (
    change_data
    .groupby("session_id")
    .cumcount()
)


print("=" * 100)
print("CHANGE DATA")
print("=" * 100)

print(
    "N:",
    len(change_data)
)

print(
    "Defects:",
    int(
        change_data["CycleDefect"].sum()
    )
)

print(
    "Sessions:",
    change_data["session_id"].nunique()
)


# ============================================================
# 3. Base Temporal Feature
# ============================================================

for col in CHANGE_SENSORS:

    # --------------------------------------------------------
    # 직전 Cycle 대비 변화
    # --------------------------------------------------------

    change_data[
        f"{col}__diff1"
    ] = (
        change_data
        .groupby("session_id")[col]
        .diff()
    )


    # --------------------------------------------------------
    # 직전 3개 Cycle 평균
    # 반드시 shift(1)
    # --------------------------------------------------------

    prev3 = (
        change_data
        .groupby("session_id")[col]
        .transform(
            lambda s:
            s.shift(1)
            .rolling(
                window=3,
                min_periods=1
            )
            .mean()
        )
    )


    change_data[
        f"{col}__dev3"
    ] = (
        change_data[col]
        -
        prev3
    )


# 이전 Target
change_data["prev_target"] = (
    change_data
    .groupby("session_id")[
        "CycleDefect"
    ]
    .shift(1)
)


# ============================================================
# 4. Robust scaling 함수
#
# Train의 Normal -> Normal transition만으로
# 각 변화 Feature의 중심/변동폭 학습
# ============================================================

def fit_robust_change_params(
    train
):

    params = {}


    normal_transition = (
        (train["CycleDefect"] == 0)
        &
        (
            (train["prev_target"] == 0)
            |
            train["prev_target"].isna()
        )
    )


    normal = (
        train[
            normal_transition
        ]
    )


    for col in CHANGE_SENSORS:

        for suffix in [
            "diff1",
            "dev3"
        ]:

            feature = (
                f"{col}__{suffix}"
            )


            values = (
                normal[
                    feature
                ]
                .dropna()
                .astype(float)
            )


            if len(values) == 0:

                median = 0.0
                scale = 1.0

            else:

                median = (
                    values.median()
                )


                mad = (
                    np.median(
                        np.abs(
                            values
                            -
                            median
                        )
                    )
                )


                if (
                    pd.isna(mad)
                    or
                    mad == 0
                ):

                    scale = (
                        values.std()
                    )

                else:

                    scale = (
                        1.4826
                        *
                        mad
                    )


                if (
                    pd.isna(scale)
                    or
                    scale == 0
                ):

                    scale = 1.0


            params[
                feature
            ] = {
                "median":
                    median,

                "scale":
                    scale
            }


    return params


# ============================================================
# 5. Feature transform
# ============================================================

def transform_change_features(
    df,
    params
):

    out = pd.DataFrame(
        index=df.index
    )


    diff_z_cols = []
    dev3_z_cols = []


    for col in CHANGE_SENSORS:

        # ====================================================
        # diff1
        # ====================================================

        raw_col = (
            f"{col}__diff1"
        )

        z_col = (
            f"{col}__diff1_z"
        )


        p = (
            params[
                raw_col
            ]
        )


        out[z_col] = (
            (
                df[
                    raw_col
                ]
                -
                p[
                    "median"
                ]
            )
            /
            p[
                "scale"
            ]
        )


        diff_z_cols.append(
            z_col
        )


        # ====================================================
        # dev3
        # ====================================================

        raw_col = (
            f"{col}__dev3"
        )

        z_col = (
            f"{col}__dev3_z"
        )


        p = (
            params[
                raw_col
            ]
        )


        out[z_col] = (
            (
                df[
                    raw_col
                ]
                -
                p[
                    "median"
                ]
            )
            /
            p[
                "scale"
            ]
        )


        dev3_z_cols.append(
            z_col
        )


    # Session 첫 Cycle 등 history 없는 경우
    out = (
        out
        .replace(
            [
                np.inf,
                -np.inf
            ],
            np.nan
        )
        .fillna(0)
    )


    # ========================================================
    # 6. Multivariate magnitude
    #
    # 특정 센서를 고르지 않고
    # 전체 변화 크기를 요약
    # ========================================================

    diff_abs = (
        out[
            diff_z_cols
        ]
        .abs()
    )


    dev_abs = (
        out[
            dev3_z_cols
        ]
        .abs()
    )


    # diff
    out[
        "DIFF_AbsMean"
    ] = (
        diff_abs.mean(
            axis=1
        )
    )


    out[
        "DIFF_AbsMax"
    ] = (
        diff_abs.max(
            axis=1
        )
    )


    out[
        "DIFF_Top3Mean"
    ] = (
        np.sort(
            diff_abs.values,
            axis=1
        )[
            :,
            -3:
        ]
        .mean(
            axis=1
        )
    )


    out[
        "DIFF_N_GT2"
    ] = (
        (
            diff_abs
            >
            2
        )
        .sum(
            axis=1
        )
    )


    out[
        "DIFF_N_GT3"
    ] = (
        (
            diff_abs
            >
            3
        )
        .sum(
            axis=1
        )
    )


    # dev3
    out[
        "DEV3_AbsMean"
    ] = (
        dev_abs.mean(
            axis=1
        )
    )


    out[
        "DEV3_AbsMax"
    ] = (
        dev_abs.max(
            axis=1
        )
    )


    out[
        "DEV3_Top3Mean"
    ] = (
        np.sort(
            dev_abs.values,
            axis=1
        )[
            :,
            -3:
        ]
        .mean(
            axis=1
        )
    )


    out[
        "DEV3_N_GT2"
    ] = (
        (
            dev_abs
            >
            2
        )
        .sum(
            axis=1
        )
    )


    out[
        "DEV3_N_GT3"
    ] = (
        (
            dev_abs
            >
            3
        )
        .sum(
            axis=1
        )
    )


    return out


# ============================================================
# 7. 고정 모델
# ============================================================

def make_change_model():

    return ExtraTreesClassifier(
        n_estimators=500,
        max_depth=6,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )


# ============================================================
# 8. Development / Holdout
# ============================================================

DEV_END = pd.Timestamp(
    "2020-10-23 23:59:59"
)


HOLDOUT_DATE = pd.Timestamp(
    "2020-11-04"
).date()


development = (
    change_data[
        change_data[
            "TimeStamp"
        ]
        <=
        DEV_END
    ]
    .copy()
)


holdout_change = (
    change_data[
        change_data[
            "TimeStamp"
        ]
        .dt.date
        ==
        HOLDOUT_DATE
    ]
    .copy()
)


print(
    "\nDevelopment:",
    len(development),
    "| Defects:",
    int(
        development[
            "CycleDefect"
        ].sum()
    )
)


print(
    "Holdout:",
    len(holdout_change),
    "| Defects:",
    int(
        holdout_change[
            "CycleDefect"
        ].sum()
    )
)


# ============================================================
# 9. Development Walk-forward
#
# New session ids:
# 1 = 10/21
# 2~4 = 10/22
# 5~7 = 10/23
#
# Session 2를 validation으로 쓰면
# 이전 train에는 defect가 0이므로
# Session 3부터 시작
# ============================================================

oof_parts = []
fold_rows = []


for valid_session in range(
    3,
    8
):

    train = (
        development[
            development[
                "session_id"
            ]
            <
            valid_session
        ]
        .copy()
    )


    valid = (
        development[
            development[
                "session_id"
            ]
            ==
            valid_session
        ]
        .copy()
    )


    if len(valid) == 0:
        continue


    y_train = (
        train[
            "CycleDefect"
        ]
        .astype(int)
    )


    y_valid = (
        valid[
            "CycleDefect"
        ]
        .astype(int)
    )


    params = (
        fit_robust_change_params(
            train
        )
    )


    X_train = (
        transform_change_features(
            train,
            params
        )
    )


    X_valid = (
        transform_change_features(
            valid,
            params
        )
    )


    model = (
        make_change_model()
    )


    model.fit(
        X_train,
        y_train
    )


    prob = (
        model
        .predict_proba(
            X_valid
        )[:, 1]
    )


    pr_auc = (
        average_precision_score(
            y_valid,
            prob
        )
    )


    baseline = (
        y_valid.mean()
    )


    roc = (
        roc_auc_score(
            y_valid,
            prob
        )
    )


    fold_rows.append({

        "Session":
            valid_session,

        "N":
            len(valid),

        "Defects":
            int(
                y_valid.sum()
            ),

        "Baseline_PR":
            baseline,

        "PR_AUC":
            pr_auc,

        "PR_Ratio":
            pr_auc
            /
            baseline,

        "ROC_AUC":
            roc
    })


    oof_parts.append(
        pd.DataFrame({

            "TimeStamp":
                valid[
                    "TimeStamp"
                ].values,

            "session_id":
                valid_session,

            "y_true":
                y_valid.values,

            "y_prob":
                prob
        })
    )


fold_change = (
    pd.DataFrame(
        fold_rows
    )
)


oof_change = (
    pd.concat(
        oof_parts,
        ignore_index=True
    )
)


oof_baseline = (
    oof_change[
        "y_true"
    ]
    .mean()
)


oof_pr = (
    average_precision_score(
        oof_change[
            "y_true"
        ],
        oof_change[
            "y_prob"
        ]
    )
)


oof_roc = (
    roc_auc_score(
        oof_change[
            "y_true"
        ],
        oof_change[
            "y_prob"
        ]
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "CHANGE-BASED DEVELOPMENT OOF"
)

print(
    "=" * 100
)


print(
    fold_change
    .to_string(
        index=False
    )
)


print(
    "\nOOF N:",
    len(oof_change)
)

print(
    "OOF Defects:",
    int(
        oof_change[
            "y_true"
        ].sum()
    )
)

print(
    "Baseline PR:",
    oof_baseline
)

print(
    "PR-AUC:",
    oof_pr
)

print(
    "PR Ratio:",
    oof_pr
    /
    oof_baseline
)

print(
    "ROC-AUC:",
    oof_roc
)


# ============================================================
# 10. 전체 Development 학습
# ============================================================

final_params = (
    fit_robust_change_params(
        development
    )
)


X_dev_change = (
    transform_change_features(
        development,
        final_params
    )
)


X_hold_change = (
    transform_change_features(
        holdout_change,
        final_params
    )
)


y_dev = (
    development[
        "CycleDefect"
    ]
    .astype(int)
)


y_hold = (
    holdout_change[
        "CycleDefect"
    ]
    .astype(int)
)


final_change_model = (
    make_change_model()
)


final_change_model.fit(
    X_dev_change,
    y_dev
)


hold_prob = (
    final_change_model
    .predict_proba(
        X_hold_change
    )[:, 1]
)


# ============================================================
# 11. Out-of-time
# ============================================================

hold_baseline = (
    y_hold.mean()
)


hold_pr = (
    average_precision_score(
        y_hold,
        hold_prob
    )
)


hold_roc = (
    roc_auc_score(
        y_hold,
        hold_prob
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "CHANGE-BASED OUT-OF-TIME"
)

print(
    "=" * 100
)


print(
    "N:",
    len(y_hold)
)

print(
    "Defects:",
    int(
        y_hold.sum()
    )
)

print(
    "Baseline PR:",
    hold_baseline
)

print(
    "PR-AUC:",
    hold_pr
)

print(
    "PR Ratio:",
    hold_pr
    /
    hold_baseline
)

print(
    "ROC-AUC:",
    hold_roc
)


# ============================================================
# 12. Holdout ranking
# ============================================================

ranking = pd.DataFrame({

    "TimeStamp":
        holdout_change[
            "TimeStamp"
        ].values,

    "y_true":
        y_hold.values,

    "Risk":
        hold_prob,

    "Diff_AbsMean":
        X_hold_change[
            "DIFF_AbsMean"
        ].values,

    "Diff_AbsMax":
        X_hold_change[
            "DIFF_AbsMax"
        ].values,

    "Diff_N_GT2":
        X_hold_change[
            "DIFF_N_GT2"
        ].values,

    "Dev3_AbsMean":
        X_hold_change[
            "DEV3_AbsMean"
        ].values
})


ranking = (
    ranking
    .sort_values(
        "Risk",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "\n"
    + "=" * 100
)

print(
    "CHANGE-BASED HOLDOUT RANKING"
)

print(
    "=" * 100
)


print(
    ranking.to_string(
        index=False
    )
)


# ============================================================
# 13. Top-K
# ============================================================

topk_rows = []


for fraction in [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]:

    n = int(
        np.ceil(
            len(ranking)
            *
            fraction
        )
    )


    selected = (
        ranking
        .iloc[
            :n
        ]
    )


    captured = int(
        selected[
            "y_true"
        ].sum()
    )


    recall = (
        captured
        /
        y_hold.sum()
    )


    precision = (
        captured
        /
        n
    )


    topk_rows.append({

        "Top_%":
            fraction * 100,

        "Inspection_N":
            n,

        "Captured_Defects":
            captured,

        "Recall":
            recall,

        "Precision":
            precision,

        "Lift":
            precision
            /
            hold_baseline
    })


topk_change = (
    pd.DataFrame(
        topk_rows
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "CHANGE-BASED HOLDOUT TOP-K"
)

print(
    "=" * 100
)


print(
    topk_change
    .to_string(
        index=False
    )
)


# ============================================================
# 14. 모델에서 사용된 중요 Feature 참고
# ============================================================

importance = (
    pd.DataFrame({

        "Feature":
            X_dev_change.columns,

        "Importance":
            final_change_model
            .feature_importances_
    })
    .sort_values(
        "Importance",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "CHANGE MODEL TOP FEATURES"
)

print(
    "=" * 100
)


print(
    importance
    .head(20)
    .to_string(
        index=False
    )
)

## 1. Random Seed 안정성 검증

In [ ]:
# ============================================================
# RG3 CHANGE-BASED MODEL
# RANDOM SEED STABILITY TEST
#
# 모델 구조/Feature는 변경하지 않음
# ExtraTrees random_state만 변경
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)


# ============================================================
# 1. Robust Scaling
# ============================================================

def fit_change_params(
    train,
    target_col="CycleDefect"
):

    temp = (
        train
        .copy()
        .sort_values("TimeStamp")
    )

    temp["_prev_y"] = (
        temp
        .groupby("session_id")[
            target_col
        ]
        .shift(1)
    )


    normal_transition = (
        (temp[target_col] == 0)
        &
        (
            (temp["_prev_y"] == 0)
            |
            temp["_prev_y"].isna()
        )
    )


    normal = (
        temp[
            normal_transition
        ]
    )


    params = {}


    for col in CHANGE_SENSORS:

        for suffix in [
            "diff1",
            "dev3"
        ]:

            feature = (
                f"{col}__{suffix}"
            )


            values = (
                normal[
                    feature
                ]
                .dropna()
                .astype(float)
            )


            if len(values) == 0:

                median = 0.0
                scale = 1.0

            else:

                median = (
                    values.median()
                )


                mad = np.median(
                    np.abs(
                        values
                        -
                        median
                    )
                )


                if (
                    pd.isna(mad)
                    or
                    mad == 0
                ):

                    scale = (
                        values.std()
                    )

                else:

                    scale = (
                        1.4826
                        *
                        mad
                    )


                if (
                    pd.isna(scale)
                    or
                    scale == 0
                ):

                    scale = 1.0


            params[
                feature
            ] = {

                "median":
                    median,

                "scale":
                    scale
            }


    return params


# ============================================================
# 2. Transform
# ============================================================

def transform_change(
    df,
    params
):

    out = pd.DataFrame(
        index=df.index
    )

    diff_cols = []
    dev_cols = []


    for col in CHANGE_SENSORS:

        # diff1
        raw = (
            f"{col}__diff1"
        )

        name = (
            f"{raw}_z"
        )

        out[name] = (
            (
                df[raw]
                -
                params[raw][
                    "median"
                ]
            )
            /
            params[raw][
                "scale"
            ]
        )

        diff_cols.append(
            name
        )


        # dev3
        raw = (
            f"{col}__dev3"
        )

        name = (
            f"{raw}_z"
        )

        out[name] = (
            (
                df[raw]
                -
                params[raw][
                    "median"
                ]
            )
            /
            params[raw][
                "scale"
            ]
        )

        dev_cols.append(
            name
        )


    out = (
        out
        .replace(
            [
                np.inf,
                -np.inf
            ],
            np.nan
        )
        .fillna(0)
    )


    diff_abs = (
        out[
            diff_cols
        ]
        .abs()
    )

    dev_abs = (
        out[
            dev_cols
        ]
        .abs()
    )


    # --------------------------------------------
    # Multi-sensor change magnitude
    # --------------------------------------------

    out[
        "DIFF_AbsMean"
    ] = (
        diff_abs.mean(
            axis=1
        )
    )

    out[
        "DIFF_AbsMax"
    ] = (
        diff_abs.max(
            axis=1
        )
    )

    out[
        "DIFF_Top3Mean"
    ] = (
        np.sort(
            diff_abs.values,
            axis=1
        )[:, -3:]
        .mean(axis=1)
    )

    out[
        "DIFF_N_GT2"
    ] = (
        (
            diff_abs > 2
        )
        .sum(axis=1)
    )

    out[
        "DIFF_N_GT3"
    ] = (
        (
            diff_abs > 3
        )
        .sum(axis=1)
    )


    out[
        "DEV3_AbsMean"
    ] = (
        dev_abs.mean(
            axis=1
        )
    )

    out[
        "DEV3_AbsMax"
    ] = (
        dev_abs.max(
            axis=1
        )
    )

    out[
        "DEV3_Top3Mean"
    ] = (
        np.sort(
            dev_abs.values,
            axis=1
        )[:, -3:]
        .mean(axis=1)
    )

    out[
        "DEV3_N_GT2"
    ] = (
        (
            dev_abs > 2
        )
        .sum(axis=1)
    )

    out[
        "DEV3_N_GT3"
    ] = (
        (
            dev_abs > 3
        )
        .sum(axis=1)
    )


    return out


# ============================================================
# 3. Seed 평가 함수
# ============================================================

def evaluate_seed(
    seed
):

    # ========================================================
    # Development OOF
    # ========================================================

    oof_parts = []


    for valid_session in range(
        3,
        8
    ):

        train = (
            development[
                development[
                    "session_id"
                ]
                <
                valid_session
            ]
            .copy()
        )


        valid = (
            development[
                development[
                    "session_id"
                ]
                ==
                valid_session
            ]
            .copy()
        )


        if len(valid) == 0:
            continue


        params = (
            fit_change_params(
                train
            )
        )


        X_train = (
            transform_change(
                train,
                params
            )
        )

        X_valid = (
            transform_change(
                valid,
                params
            )
        )


        y_train = (
            train[
                "CycleDefect"
            ]
            .astype(int)
        )

        y_valid = (
            valid[
                "CycleDefect"
            ]
            .astype(int)
        )


        model = (
            ExtraTreesClassifier(
                n_estimators=500,
                max_depth=6,
                min_samples_leaf=2,
                max_features="sqrt",
                class_weight="balanced",
                random_state=seed,
                n_jobs=-1
            )
        )


        model.fit(
            X_train,
            y_train
        )


        prob = (
            model
            .predict_proba(
                X_valid
            )[:, 1]
        )


        oof_parts.append(
            pd.DataFrame({

                "y_true":
                    y_valid.values,

                "y_prob":
                    prob
            })
        )


    oof = pd.concat(
        oof_parts,
        ignore_index=True
    )


    dev_pr = (
        average_precision_score(
            oof[
                "y_true"
            ],
            oof[
                "y_prob"
            ]
        )
    )


    dev_roc = (
        roc_auc_score(
            oof[
                "y_true"
            ],
            oof[
                "y_prob"
            ]
        )
    )


    # ========================================================
    # Out-of-time
    # ========================================================

    params = (
        fit_change_params(
            development
        )
    )


    X_dev = (
        transform_change(
            development,
            params
        )
    )


    X_hold = (
        transform_change(
            holdout_change,
            params
        )
    )


    y_dev = (
        development[
            "CycleDefect"
        ]
        .astype(int)
    )


    y_hold = (
        holdout_change[
            "CycleDefect"
        ]
        .astype(int)
    )


    model = (
        ExtraTreesClassifier(
            n_estimators=500,
            max_depth=6,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1
        )
    )


    model.fit(
        X_dev,
        y_dev
    )


    hold_prob = (
        model
        .predict_proba(
            X_hold
        )[:, 1]
    )


    hold_pr = (
        average_precision_score(
            y_hold,
            hold_prob
        )
    )


    hold_roc = (
        roc_auc_score(
            y_hold,
            hold_prob
        )
    )


    # ========================================================
    # Top 30%
    # ========================================================

    ranking = pd.DataFrame({

        "y_true":
            y_hold.values,

        "prob":
            hold_prob
    })


    ranking = (
        ranking
        .sort_values(
            "prob",
            ascending=False
        )
        .reset_index(drop=True)
    )


    inspect_n = int(
        np.ceil(
            len(ranking)
            *
            0.30
        )
    )


    selected = (
        ranking
        .iloc[
            :inspect_n
        ]
    )


    captured = int(
        selected[
            "y_true"
        ]
        .sum()
    )


    recall = (
        captured
        /
        y_hold.sum()
    )


    precision = (
        captured
        /
        inspect_n
    )


    return {

        "Seed":
            seed,

        "Development_PR_AUC":
            dev_pr,

        "Development_ROC_AUC":
            dev_roc,

        "OOT_PR_AUC":
            hold_pr,

        "OOT_ROC_AUC":
            hold_roc,

        "Top30_Captured":
            captured,

        "Top30_Recall":
            recall,

        "Top30_Precision":
            precision
    }


# ============================================================
# 4. 30 Seeds
# ============================================================

SEEDS = list(
    range(
        30
    )
)


seed_results = []


for seed in SEEDS:

    print(
        f"Seed {seed:02d} ..."
    )

    seed_results.append(
        evaluate_seed(
            seed
        )
    )


seed_df = pd.DataFrame(
    seed_results
)


# ============================================================
# 5. 출력
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "SEED STABILITY - ALL RESULTS"
)

print(
    "=" * 100
)


print(
    seed_df
    .to_string(
        index=False
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "SEED STABILITY SUMMARY"
)

print(
    "=" * 100
)


summary = (
    seed_df[
        [
            "Development_PR_AUC",
            "Development_ROC_AUC",
            "OOT_PR_AUC",
            "OOT_ROC_AUC",
            "Top30_Captured",
            "Top30_Recall",
            "Top30_Precision"
        ]
    ]
    .agg(
        [
            "mean",
            "std",
            "min",
            "median",
            "max"
        ]
    )
    .T
)


print(
    summary.to_string()
)

In [ ]:
# ============================================================
# RG3 SESSION-PRESERVING PERMUTATION TEST
#
# H0:
# 같은 Session 안에서
# Sensor Change와 Defect Label의 관계는 우연이다.
#
# Primary metric:
# Development Walk-forward OOF PR-AUC
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)


# ============================================================
# 1. OOF 평가 함수
# ============================================================

def evaluate_dev_target(
    dev_df,
    target_col,
    model_seed=42,
    n_estimators=300
):

    oof_parts = []


    for valid_session in range(
        3,
        8
    ):

        train = (
            dev_df[
                dev_df[
                    "session_id"
                ]
                <
                valid_session
            ]
            .copy()
        )


        valid = (
            dev_df[
                dev_df[
                    "session_id"
                ]
                ==
                valid_session
            ]
            .copy()
        )


        if len(valid) == 0:
            continue


        y_train = (
            train[
                target_col
            ]
            .astype(int)
        )


        y_valid = (
            valid[
                target_col
            ]
            .astype(int)
        )


        # 혹시 학습에 클래스 하나만 남으면 skip
        if (
            y_train.nunique()
            <
            2
        ):
            continue


        # ====================================================
        # Target에 맞춰 Normal transition scale 재계산
        # ====================================================

        params = (
            fit_change_params(
                train,
                target_col=
                    target_col
            )
        )


        X_train = (
            transform_change(
                train,
                params
            )
        )


        X_valid = (
            transform_change(
                valid,
                params
            )
        )


        model = (
            ExtraTreesClassifier(
                n_estimators=
                    n_estimators,
                max_depth=6,
                min_samples_leaf=2,
                max_features="sqrt",
                class_weight="balanced",
                random_state=
                    model_seed,
                n_jobs=-1
            )
        )


        model.fit(
            X_train,
            y_train
        )


        prob = (
            model
            .predict_proba(
                X_valid
            )[:, 1]
        )


        oof_parts.append(
            pd.DataFrame({

                "y_true":
                    y_valid.values,

                "y_prob":
                    prob
            })
        )


    oof = pd.concat(
        oof_parts,
        ignore_index=True
    )


    return (
        average_precision_score(
            oof[
                "y_true"
            ],
            oof[
                "y_prob"
            ]
        ),
        roc_auc_score(
            oof[
                "y_true"
            ],
            oof[
                "y_prob"
            ]
        )
    )


# ============================================================
# 2. 동일 조건에서 실제 Label 성능
#
# permutation과 공정하게 비교하기 위해
# n_estimators=300으로 다시 계산
# ============================================================

OBSERVED_PR, OBSERVED_ROC = (
    evaluate_dev_target(
        development,
        target_col=
            "CycleDefect",
        model_seed=42,
        n_estimators=300
    )
)


print("=" * 100)
print("OBSERVED PERFORMANCE")
print("=" * 100)

print(
    "Observed PR-AUC :",
    OBSERVED_PR
)

print(
    "Observed ROC-AUC:",
    OBSERVED_ROC
)


# ============================================================
# 3. Session 내부 Label Shuffle
# ============================================================

N_PERMUTATIONS = 100

rng = np.random.default_rng(
    42
)


perm_rows = []


for perm_id in range(
    N_PERMUTATIONS
):

    perm = (
        development
        .copy()
    )


    perm[
        "PermutedTarget"
    ] = 0


    # --------------------------------------------------------
    # 각 Session 안에서만 Target shuffle
    # → session별 불량 수 유지
    # --------------------------------------------------------

    for session_id, idx in (
        perm
        .groupby(
            "session_id"
        )
        .groups
        .items()
    ):

        original = (
            perm.loc[
                idx,
                "CycleDefect"
            ]
            .values
            .copy()
        )


        shuffled = (
            rng.permutation(
                original
            )
        )


        perm.loc[
            idx,
            "PermutedTarget"
        ] = shuffled


    try:

        perm_pr, perm_roc = (
            evaluate_dev_target(
                perm,
                target_col=
                    "PermutedTarget",
                model_seed=
                    42,
                n_estimators=
                    300
            )
        )

    except Exception as e:

        print(
            "Permutation failed:",
            perm_id,
            e
        )

        continue


    perm_rows.append({

        "Permutation":
            perm_id,

        "PR_AUC":
            perm_pr,

        "ROC_AUC":
            perm_roc
    })


    if (
        (perm_id + 1)
        %
        10
        ==
        0
    ):

        print(
            f"{perm_id + 1}"
            f"/{N_PERMUTATIONS}"
        )


perm_df = pd.DataFrame(
    perm_rows
)


# ============================================================
# 4. Empirical p-value
# ============================================================

pr_pvalue = (
    1
    +
    (
        perm_df[
            "PR_AUC"
        ]
        >=
        OBSERVED_PR
    )
    .sum()
) / (
    len(perm_df)
    +
    1
)


roc_pvalue = (
    1
    +
    (
        perm_df[
            "ROC_AUC"
        ]
        >=
        OBSERVED_ROC
    )
    .sum()
) / (
    len(perm_df)
    +
    1
)


# ============================================================
# 5. Z score
# ============================================================

pr_null_mean = (
    perm_df[
        "PR_AUC"
    ]
    .mean()
)


pr_null_std = (
    perm_df[
        "PR_AUC"
    ]
    .std()
)


pr_z = (
    (
        OBSERVED_PR
        -
        pr_null_mean
    )
    /
    pr_null_std
)


# ============================================================
# 6. 출력
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "PERMUTATION TEST RESULT"
)

print(
    "=" * 100
)


print(
    f"Permutations      : "
    f"{len(perm_df)}"
)

print(
    f"Observed PR-AUC   : "
    f"{OBSERVED_PR:.6f}"
)

print(
    f"Null PR Mean      : "
    f"{pr_null_mean:.6f}"
)

print(
    f"Null PR Std       : "
    f"{pr_null_std:.6f}"
)

print(
    f"Null PR Min       : "
    f"{perm_df['PR_AUC'].min():.6f}"
)

print(
    f"Null PR Median    : "
    f"{perm_df['PR_AUC'].median():.6f}"
)

print(
    f"Null PR Max       : "
    f"{perm_df['PR_AUC'].max():.6f}"
)

print(
    f"Observed PR Z     : "
    f"{pr_z:.3f}"
)

print(
    f"Empirical PR p    : "
    f"{pr_pvalue:.4f}"
)


print(
    "\nObserved ROC-AUC:",
    round(
        OBSERVED_ROC,
        6
    )
)

print(
    "Null ROC Mean    :",
    round(
        perm_df[
            "ROC_AUC"
        ].mean(),
        6
    )
)

print(
    "Empirical ROC p  :",
    round(
        roc_pvalue,
        4
    )
)


# ============================================================
# 7. 우연보다 높은 비율
# ============================================================

print(
    "\nPermutations >= Observed PR:",
    int(
        (
            perm_df[
                "PR_AUC"
            ]
            >=
            OBSERVED_PR
        )
        .sum()
    ),
    "/",
    len(
        perm_df
    )
)

In [ ]:
# ============================================================
# RG3 CHANGE-BASED MODEL
# STRATIFIED BOOTSTRAP UNCERTAINTY ANALYSIS
#
# 목적:
# - 11/04 OOT = 35 cycles / 7 defects
# - 작은 평가표본 때문에 성능이 얼마나 불확실한지 정량화
#
# 주의:
# - 모델 튜닝 X
# - 모델 재선택 X
# - seed=42 고정 모델
# - Holdout sampling uncertainty만 측정
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)


# ============================================================
# 1. FINAL CHANGE MODEL 재생성
# ============================================================

final_params = (
    fit_change_params(
        development
    )
)


X_dev_boot = (
    transform_change(
        development,
        final_params
    )
)


X_hold_boot = (
    transform_change(
        holdout_change,
        final_params
    )
)


y_dev_boot = (
    development[
        "CycleDefect"
    ]
    .astype(int)
    .reset_index(drop=True)
)


y_hold_boot = (
    holdout_change[
        "CycleDefect"
    ]
    .astype(int)
    .reset_index(drop=True)
)


final_boot_model = (
    ExtraTreesClassifier(
        n_estimators=500,
        max_depth=6,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
)


final_boot_model.fit(
    X_dev_boot,
    y_dev_boot
)


hold_prob_boot = (
    final_boot_model
    .predict_proba(
        X_hold_boot
    )[:, 1]
)


# ============================================================
# 2. Observed Metrics
# ============================================================

N = len(
    y_hold_boot
)

N_DEFECT = int(
    y_hold_boot.sum()
)

N_NORMAL = (
    N
    -
    N_DEFECT
)

BASELINE_PR = (
    y_hold_boot.mean()
)


OBS_PR = (
    average_precision_score(
        y_hold_boot,
        hold_prob_boot
    )
)


OBS_ROC = (
    roc_auc_score(
        y_hold_boot,
        hold_prob_boot
    )
)


TOP_FRAC = 0.30

TOP_N = int(
    np.ceil(
        N
        *
        TOP_FRAC
    )
)


observed_df = (
    pd.DataFrame({
        "y":
            y_hold_boot,

        "prob":
            hold_prob_boot
    })
    .sort_values(
        "prob",
        ascending=False
    )
)


obs_selected = (
    observed_df
    .iloc[
        :TOP_N
    ]
)


OBS_CAPTURED = int(
    obs_selected[
        "y"
    ]
    .sum()
)


OBS_RECALL = (
    OBS_CAPTURED
    /
    N_DEFECT
)


OBS_PRECISION = (
    OBS_CAPTURED
    /
    TOP_N
)


OBS_LIFT = (
    OBS_PRECISION
    /
    BASELINE_PR
)


print("=" * 100)
print("OBSERVED OOT PERFORMANCE")
print("=" * 100)

print(
    f"N               : {N}"
)

print(
    f"Defects         : {N_DEFECT}"
)

print(
    f"Normal          : {N_NORMAL}"
)

print(
    f"Baseline PR     : {BASELINE_PR:.6f}"
)

print(
    f"PR-AUC          : {OBS_PR:.6f}"
)

print(
    f"PR Lift         : {OBS_PR / BASELINE_PR:.3f}x"
)

print(
    f"ROC-AUC         : {OBS_ROC:.6f}"
)

print(
    f"Top30 N         : {TOP_N}"
)

print(
    f"Top30 Captured  : {OBS_CAPTURED}/{N_DEFECT}"
)

print(
    f"Top30 Recall    : {OBS_RECALL:.6f}"
)

print(
    f"Top30 Precision : {OBS_PRECISION:.6f}"
)

print(
    f"Top30 Lift      : {OBS_LIFT:.3f}x"
)


# ============================================================
# 3. Stratified Bootstrap
#
# 7 defect, 28 normal 숫자를 매 bootstrap마다 유지
#
# -> 평가표본 composition 변화가 아니라
#    동일 prevalence 하에서 cycle sampling uncertainty 측정
# ============================================================

N_BOOT = 5000

rng = (
    np.random.default_rng(
        20261002
    )
)


y_np = (
    y_hold_boot
    .to_numpy()
)


prob_np = (
    np.asarray(
        hold_prob_boot
    )
)


pos_idx = np.where(
    y_np == 1
)[0]


neg_idx = np.where(
    y_np == 0
)[0]


boot_rows = []


for b in range(
    N_BOOT
):

    # --------------------------------------------------------
    # 불량 7개 복원추출
    # --------------------------------------------------------

    boot_pos = (
        rng.choice(
            pos_idx,
            size=len(
                pos_idx
            ),
            replace=True
        )
    )


    # --------------------------------------------------------
    # 정상 28개 복원추출
    # --------------------------------------------------------

    boot_neg = (
        rng.choice(
            neg_idx,
            size=len(
                neg_idx
            ),
            replace=True
        )
    )


    boot_idx = np.concatenate(
        [
            boot_pos,
            boot_neg
        ]
    )


    # 순서를 랜덤하게 섞음
    boot_idx = (
        rng.permutation(
            boot_idx
        )
    )


    y_b = (
        y_np[
            boot_idx
        ]
    )


    p_b = (
        prob_np[
            boot_idx
        ]
    )


    # ========================================================
    # Ranking metrics
    # ========================================================

    pr_b = (
        average_precision_score(
            y_b,
            p_b
        )
    )


    roc_b = (
        roc_auc_score(
            y_b,
            p_b
        )
    )


    # ========================================================
    # Top30
    # ========================================================

    order = (
        np.argsort(
            -p_b,
            kind="stable"
        )
    )


    top_idx = (
        order[
            :TOP_N
        ]
    )


    captured = int(
        y_b[
            top_idx
        ]
        .sum()
    )


    recall = (
        captured
        /
        N_DEFECT
    )


    precision = (
        captured
        /
        TOP_N
    )


    lift = (
        precision
        /
        BASELINE_PR
    )


    boot_rows.append({

        "Bootstrap":
            b,

        "PR_AUC":
            pr_b,

        "PR_Ratio":
            pr_b
            /
            BASELINE_PR,

        "PR_Above_Baseline":
            int(
                pr_b
                >
                BASELINE_PR
            ),

        "PR_Difference":
            pr_b
            -
            BASELINE_PR,

        "ROC_AUC":
            roc_b,

        "Top30_Captured":
            captured,

        "Top30_Recall":
            recall,

        "Top30_Precision":
            precision,

        "Top30_Lift":
            lift
    })


boot_df = (
    pd.DataFrame(
        boot_rows
    )
)


# ============================================================
# 4. Bootstrap CI 함수
# ============================================================

def bootstrap_summary(
    series,
    observed
):

    return {

        "Observed":
            observed,

        "Boot_Mean":
            series.mean(),

        "Boot_Std":
            series.std(),

        "CI_2.5%":
            series.quantile(
                0.025
            ),

        "CI_50%":
            series.quantile(
                0.50
            ),

        "CI_97.5%":
            series.quantile(
                0.975
            )
    }


# ============================================================
# 5. Metric Summary
# ============================================================

summary_rows = []


summary_rows.append({
    "Metric":
        "PR_AUC",

    **bootstrap_summary(
        boot_df[
            "PR_AUC"
        ],
        OBS_PR
    )
})


summary_rows.append({
    "Metric":
        "PR_Ratio",

    **bootstrap_summary(
        boot_df[
            "PR_Ratio"
        ],
        OBS_PR
        /
        BASELINE_PR
    )
})


summary_rows.append({
    "Metric":
        "ROC_AUC",

    **bootstrap_summary(
        boot_df[
            "ROC_AUC"
        ],
        OBS_ROC
    )
})


summary_rows.append({
    "Metric":
        "Top30_Captured",

    **bootstrap_summary(
        boot_df[
            "Top30_Captured"
        ],
        OBS_CAPTURED
    )
})


summary_rows.append({
    "Metric":
        "Top30_Recall",

    **bootstrap_summary(
        boot_df[
            "Top30_Recall"
        ],
        OBS_RECALL
    )
})


summary_rows.append({
    "Metric":
        "Top30_Precision",

    **bootstrap_summary(
        boot_df[
            "Top30_Precision"
        ],
        OBS_PRECISION
    )
})


summary_rows.append({
    "Metric":
        "Top30_Lift",

    **bootstrap_summary(
        boot_df[
            "Top30_Lift"
        ],
        OBS_LIFT
    )
})


bootstrap_summary_df = (
    pd.DataFrame(
        summary_rows
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "BOOTSTRAP 95% INTERVAL"
)

print(
    "=" * 100
)


print(
    bootstrap_summary_df
    .to_string(
        index=False
    )
)


# ============================================================
# 6. Baseline 초과 여부
#
# 주의:
# 이것은 permutation p-value가 아님.
# bootstrap sample 중 baseline을 초과한 비율일 뿐.
# ============================================================

pr_above_rate = (
    boot_df[
        "PR_Above_Baseline"
    ]
    .mean()
)


print(
    "\n"
    + "=" * 100
)

print(
    "BOOTSTRAP ROBUSTNESS CHECK"
)

print(
    "=" * 100
)


print(
    f"Bootstrap samples : "
    f"{N_BOOT}"
)


print(
    f"PR-AUC > baseline "
    f"({BASELINE_PR:.3f}) : "
    f"{pr_above_rate*100:.2f}%"
)


print(
    f"PR difference "
    f"95% interval : "
    f"["
    f"{boot_df['PR_Difference'].quantile(0.025):.6f}, "
    f"{boot_df['PR_Difference'].quantile(0.975):.6f}"
    f"]"
)


print(
    f"ROC > 0.5 : "
    f"{(boot_df['ROC_AUC'] > 0.5).mean()*100:.2f}%"
)


print(
    f"Top30 captures >= 3 defects : "
    f"{(boot_df['Top30_Captured'] >= 3).mean()*100:.2f}%"
)


print(
    f"Top30 captures >= 4 defects : "
    f"{(boot_df['Top30_Captured'] >= 4).mean()*100:.2f}%"
)


# ============================================================
# 7. Top30 포착 개수 분포
# ============================================================

capture_distribution = (
    boot_df[
        "Top30_Captured"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "Captured_Defects"
    )
    .reset_index(
        name="Count"
    )
)


capture_distribution[
    "Rate_%"
] = (
    capture_distribution[
        "Count"
    ]
    /
    N_BOOT
    *
    100
)


print(
    "\n"
    + "=" * 100
)

print(
    "TOP30 CAPTURE DISTRIBUTION"
)

print(
    "=" * 100
)


print(
    capture_distribution
    .to_string(
        index=False
    )
)